# 03 — Model Comparison, Validation, and Interpretation

This notebook is the single model-development notebook for the portfolio project. It answers four questions clearly:

1. **Which model family performs best?**
2. **Should the men's and women's tournaments use separate models or a pooled model?**
3. **Which feature groups add genuine out-of-season value?**
4. **Are the winning probabilities well calibrated and interpretable?**

## Models compared

Every requested framework is evaluated on the same chronological outer seasons:

- seed-only logistic regression;
- Elo-plus-seed logistic regression;
- rich elastic-net logistic regression;
- XGBoost classification;
- LightGBM classification;
- PyTorch multilayer perceptron;
- TensorFlow/Keras multilayer perceptron;
- ridge, XGBoost, and LightGBM point-margin models converted to win probabilities.

## Architectures compared

- **Separate men's model:** trained only on men's tournament history and allowed to use men's-only inputs.
- **Separate women's model:** trained only on women's tournament history.
- **Pooled common-feature model:** trained on both tournaments using only features available to both and an explicit gender indicator.
- **Partial-pooling and cross-family ensembles:** blends are learned only from earlier out-of-fold predictions.

## How the notebook identifies a winner

The notebook publishes both:

- **Lowest-score winner:** the model with the lowest mean season-level Brier score.
- **Recommended winner:** the least complex model whose score is within one standard error of the lowest score.

This distinction prevents a tiny, unstable numerical advantage from automatically defeating a simpler and more reproducible model.

## What “ablation” means

A feature ablation is a controlled experiment in which feature groups are added or removed while the model, folds, and evaluation metric stay fixed. For example:

```text
Seeds only
→ add team performance
→ add ratings
→ add schedule strength
→ add efficiency
→ add history
→ add matchup interactions
```

If Brier score improves after a block is added across held-out seasons, that block contributes useful forecasting information. If performance worsens, the block is noise, redundant, or too unstable.

## Scientific boundaries

- Tournament seasons are indivisible validation groups.
- Every training season precedes its validation season.
- The model-selection comparison uses the same held-out seasons for every framework and architecture.
- Imputation, scaling, feature selection, tuning, early stopping, calibration, and blending are fitted inside training data only.
- The 2022–2025 benchmark remains unopened.
- The 2026 Stage 2 matrix remains unloaded.
- Brier score is the primary selection metric; classification metrics are secondary diagnostics at a fixed 0.50 threshold.
- All training is sequential, thread-capped, checkpointed, and restartable.

> **Canonical notebook.** This file is self-contained. It performs model comparison, diagnostics, interpretation, integrity verification, and recipe freezing without requiring post-run repair or finalization scripts. Valid checkpoints are reused when the notebook is rerun.

## Why the notebook limits dimensionality

Notebook `02` produced a broad research bank of candidate features. That is valuable for hypothesis testing, but it is not appropriate to feed every candidate into every model when the number of historical tournament games is comparatively small.

Each training fold therefore performs its own:

1. missingness and variance screening;
2. season-by-season stability scoring;
3. feature-block balancing;
4. correlation pruning;
5. model-family-specific feature cap.

The caps are intentionally conservative:

- linear models: at most 60 predictors;
- tree models: at most 120 predictors;
- neural models: at most 72 predictors;
- point-margin models: at most 100 predictors.

The resulting comparison emphasizes generalization, calibration, and stability rather than raw dimensionality.

In [ ]:
from __future__ import annotations

# Resource limits are set before numerical libraries are imported.
import os

_DEFAULT_THREAD_CAP = 4
os.environ.setdefault("PYTHONHASHSEED", "2026")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
os.environ.setdefault("KERAS_BACKEND", "tensorflow")
for _variable in (
    "OMP_NUM_THREADS",
    "MKL_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
):
    os.environ.setdefault(_variable, str(_DEFAULT_THREAD_CAP))

import ctypes
import gc
import hashlib
import json
import logging
import math
import platform
import random
import re
import shutil
import sys
import time
import traceback
import warnings
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Iterable, Sequence

# Import PyTorch before LightGBM, XGBoost, TensorFlow, and other compiled numerical libraries on Windows.
# This reduces the chance that an incompatible OpenMP/DLL runtime is loaded first.
try:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset
except Exception as exc:
    torch = None
    nn = None
    DataLoader = None
    TensorDataset = None
    TORCH_IMPORT_ERROR = repr(exc)
else:
    TORCH_IMPORT_ERROR = None

import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objects as go
import psutil
import scipy
import shap
import sklearn
import xgboost as xgb
import yaml
from plotly.subplots import make_subplots
from scipy.optimize import minimize, minimize_scalar
from scipy.special import expit, logit
from scipy.stats import ks_2samp
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    auc,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    matthews_corrcoef,
    mean_absolute_error,
    mean_squared_error,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import SplineTransformer, StandardScaler
from threadpoolctl import threadpool_limits


try:
    import tensorflow as tf
except Exception as exc:
    tf = None
    TF_IMPORT_ERROR = repr(exc)
else:
    TF_IMPORT_ERROR = None


class _ExpectedTensorFlowMessageFilter(logging.Filter):
    """Remove only known, non-actionable TensorFlow/Keras notices."""

    _SUPPRESSED_FRAGMENTS = (
        "The name tf.reset_default_graph is deprecated",
        "TensorFlow GPU support is not available on native Windows",
    )

    def filter(self, record: logging.LogRecord) -> bool:
        message = record.getMessage()
        return not any(
            fragment in message for fragment in self._SUPPRESSED_FRAGMENTS
        )


if tf is not None:
    tf.get_logger().addFilter(_ExpectedTensorFlowMessageFilter())

try:
    from march_mania.paths import get_project_paths
except Exception as exc:
    raise ImportError(
        "The local march_mania package is unavailable. From the repository root, "
        "run `python -m pip install -e . --no-deps`, restart the kernel, and rerun."
    ) from exc

warnings.filterwarnings("once", category=RuntimeWarning)
warnings.filterwarnings(
    "ignore",
    message=r"(?s).*Found Intel OpenMP.*LLVM OpenMP.*",
    category=RuntimeWarning,
)
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)

PHYSICAL_CORES = psutil.cpu_count(logical=False) or 2
LOGICAL_CORES = psutil.cpu_count(logical=True) or PHYSICAL_CORES
MAX_THREADS = max(1, min(_DEFAULT_THREAD_CAP, PHYSICAL_CORES))
PROCESS = psutil.Process(os.getpid())
RAM_GB = psutil.virtual_memory().total / 1024**3

if torch is not None:
    torch.manual_seed(SEED)
    torch.set_num_threads(MAX_THREADS)
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass

if tf is not None:
    try:
        tf.keras.utils.set_random_seed(SEED)
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass
    try:
        tf.config.threading.set_intra_op_parallelism_threads(MAX_THREADS)
        tf.config.threading.set_inter_op_parallelism_threads(1)
    except RuntimeError:
        # TensorFlow may already have initialized threading in an interactive kernel.
        pass

# One notebook, one default run. Change to "quick" only for troubleshooting.
RUN_PROFILE = "full"
assert RUN_PROFILE in {"quick", "full"}
RUN_MODE = "standard"  # Internal name retained for stable artifact naming.
NOTEBOOK_IMPLEMENTATION_VERSION = 5

MODE_SETTINGS: dict[str, dict[str, Any]] = {
    "quick": {
        "outer_folds_per_context": 1,
        "inner_fold_cap": 2,
        "linear_feature_cap": 35,
        "tree_feature_cap": 60,
        "neural_feature_cap": 48,
        "margin_feature_cap": 60,
        "precorrelation_pool_cap": 110,
        "trial_budgets": {
            "elastic_logistic": 1,
            "xgb_classifier": 1,
            "lgb_classifier": 1,
            "xgb_margin": 1,
            "lgb_margin": 1,
            "torch_mlp": 1,
            "tensorflow_mlp": 1,
        },
        "run_hist_classifier": True,
        "run_margin_models": True,
        "run_neural": True,
        "run_ablation": False,
        "run_explainability": False,
        "bootstrap_repetitions": 250,
        "importance_rows": 100,
        "permutation_repeats": 2,
    },
    "full": {
        "outer_folds_per_context": None,
        "inner_fold_cap": 3,
        "linear_feature_cap": 60,
        "tree_feature_cap": 120,
        "neural_feature_cap": 72,
        "margin_feature_cap": 100,
        "precorrelation_pool_cap": 240,
        "trial_budgets": {
            "elastic_logistic": 4,
            "xgb_classifier": 5,
            "lgb_classifier": 5,
            "xgb_margin": 4,
            "lgb_margin": 3,
            "torch_mlp": 2,
            "tensorflow_mlp": 2,
        },
        "run_hist_classifier": True,
        "run_margin_models": True,
        "run_neural": True,
        "run_ablation": True,
        "run_explainability": True,
        "bootstrap_repetitions": 1500,
        "importance_rows": 400,
        "permutation_repeats": 5,
    },
}
MODE = MODE_SETTINGS[RUN_PROFILE]
MATCHED_VALIDATION_SEASONS = [2016, 2017, 2018, 2019, 2021]

PATHS = get_project_paths()
ROOT = PATHS.root
INTERIM = PATHS.interim
PROCESSED = PATHS.processed
CONFIG_DIR = ROOT / "configs"
MODEL_REPORTS = ROOT / "reports" / "modeling" / "03_model_comparison"
FIGURE_DIR = ROOT / "reports" / "figures" / "modeling_03"
CACHE_DIR = ROOT / "data" / "model_cache" / "03_model_comparison" / RUN_PROFILE
STUDY_DIR = CACHE_DIR / "optuna"
FAILURE_DIR = CACHE_DIR / "failures"
LOG_DIR = MODEL_REPORTS / "logs"

for _directory in (
    CONFIG_DIR,
    MODEL_REPORTS,
    FIGURE_DIR,
    CACHE_DIR,
    STUDY_DIR,
    FAILURE_DIR,
    LOG_DIR,
):
    _directory.mkdir(parents=True, exist_ok=True)

LOGGER = logging.getLogger("march_mania_model_comparison")
LOGGER.setLevel(logging.INFO)
LOGGER.handlers.clear()
_file_handler = logging.FileHandler(LOG_DIR / "run.log", mode="a", encoding="utf-8")
_file_handler.setFormatter(
    logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
)
_stream_handler = logging.StreamHandler(sys.stdout)
_stream_handler.setFormatter(logging.Formatter("%(levelname)s | %(message)s"))
LOGGER.addHandler(_file_handler)
LOGGER.addHandler(_stream_handler)

EVENTS_PATH = LOG_DIR / "events.jsonl"


def log_event(event: str, **payload: Any) -> None:
    record = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "event": event,
        **payload,
    }
    with EVENTS_PATH.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, default=str) + "\n")


PLOT_TEMPLATE = "plotly_white"
MODEL_DISPLAY_NAMES = {
    "constant_probability": "Constant 0.50",
    "seed_logistic": "Seed logistic",
    "elo_seed_logistic": "Elo + seed logistic",
    "elastic_logistic": "Rich elastic-net logistic",
    "hist_classifier": "Histogram boosting",
    "xgb_classifier": "XGBoost classifier",
    "lgb_classifier": "LightGBM classifier",
    "ridge_margin": "Ridge margin",
    "xgb_margin": "XGBoost margin",
    "lgb_margin": "LightGBM margin",
    "torch_mlp": "PyTorch MLP",
    "tensorflow_mlp": "TensorFlow MLP",
    "constrained_ensemble": "Constrained ensemble",
    "partial_pooling_ensemble": "Partial-pooling ensemble",
}


def save_plotly(fig: go.Figure, name: str) -> None:
    fig.update_layout(
        template=PLOT_TEMPLATE,
        font=dict(family="Arial", size=13),
        title_x=0.02,
        margin=dict(l=60, r=35, t=80, b=60),
        hoverlabel=dict(font_size=12),
    )
    html_path = FIGURE_DIR / f"{name}.html"
    fig.write_html(html_path, include_plotlyjs="cdn", full_html=True)
    try:
        fig.write_image(FIGURE_DIR / f"{name}.png", scale=2)
    except Exception as exc:
        LOGGER.info("Static Plotly export skipped for %s: %s", name, exc)
    fig.show()


package_versions = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "xgboost": xgb.__version__,
    "lightgbm": lgb.__version__,
    "optuna": optuna.__version__,
    "shap": shap.__version__,
    "plotly": plotly.__version__,
    "pytorch": None if torch is None else torch.__version__,
    "tensorflow": None if tf is None else tf.__version__,
}

assert "ml-modeling" in str(sys.executable).lower(), (
    "Select the Python (ml-modeling) kernel before continuing."
)
assert torch is not None, (
    "PyTorch could not load its Windows binaries. Close Jupyter completely, "
    "repair the CPU-only torch installation from Miniforge Prompt, reopen Jupyter, "
    f"and rerun. Original error: {TORCH_IMPORT_ERROR}"
)
assert tf is not None, f"TensorFlow import failed: {TF_IMPORT_ERROR}"

log_event(
    "notebook_started",
    run_profile=RUN_PROFILE,
    matched_validation_seasons=MATCHED_VALIDATION_SEASONS,
    package_versions=package_versions,
    ram_gb=round(RAM_GB, 3),
    physical_cores=PHYSICAL_CORES,
    logical_cores=LOGICAL_CORES,
    max_threads=MAX_THREADS,
)

print("Project root:", ROOT)
print("Run profile:", RUN_PROFILE)
print("Notebook implementation version:", NOTEBOOK_IMPLEMENTATION_VERSION)
print("Matched validation seasons:", MATCHED_VALIDATION_SEASONS)
print("Python:", sys.executable)
print("Package versions:", json.dumps(package_versions, indent=2))
print("Physical/logical cores:", PHYSICAL_CORES, "/", LOGICAL_CORES)
print("Model threads:", MAX_THREADS)
print("System RAM GB:", round(RAM_GB, 2))
print("Current process RSS MB:", round(PROCESS.memory_info().rss / 1024**2, 2))

if RAM_GB < 8:
    warnings.warn(
        "Less than 8 GB of RAM was detected. Change RUN_PROFILE to 'quick' "
        "for troubleshooting before attempting the full run."
    )

## 1. Load and verify the completed feature store

Only development seasons are loaded. The locked 2022–2025 outcomes and the 2026 Stage 2 matrix remain outside this notebook.

In [ ]:
def sha256_file(path: Path, chunk_bytes: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_bytes), b""):
            digest.update(chunk)
    return digest.hexdigest()


def read_json(path: Path) -> Any:
    return json.loads(path.read_text(encoding="utf-8"))


def rss_mb() -> float:
    return PROCESS.memory_info().rss / 1024**2


split_path = CONFIG_DIR / "splits.yaml"
feature_path = CONFIG_DIR / "features.yaml"
readiness_01_path = ROOT / "reports" / "modeling" / "01_readiness_summary.json"
readiness_02_path = ROOT / "reports" / "feature_engineering" / "02_readiness_summary.json"
feature_checks_path = (
    ROOT
    / "reports"
    / "feature_engineering"
    / "feature_store_leakage_and_integrity_checks.csv"
)
candidate_sets_path = (
    ROOT / "reports" / "feature_engineering" / "candidate_feature_sets.json"
)
feature_registry_path = (
    ROOT / "reports" / "feature_engineering" / "feature_registry_v1.csv"
)
if not feature_registry_path.exists():
    feature_registry_path = (
        ROOT / "reports" / "feature_engineering" / "feature_registry.csv"
    )
artifact_manifest_path = (
    ROOT / "reports" / "feature_engineering" / "feature_store_artifact_manifest.csv"
)

required_control_files = [
    split_path,
    feature_path,
    readiness_01_path,
    readiness_02_path,
    feature_checks_path,
    candidate_sets_path,
    feature_registry_path,
    artifact_manifest_path,
]
missing_control = [str(path) for path in required_control_files if not path.exists()]
assert not missing_control, (
    "Required notebook 01/02 control artifacts are missing: "
    f"{missing_control}"
)

SPLITS = yaml.safe_load(split_path.read_text(encoding="utf-8"))
FEATURE_CONFIG = yaml.safe_load(feature_path.read_text(encoding="utf-8"))
READINESS_01 = read_json(readiness_01_path)
READINESS_02 = read_json(readiness_02_path)
FEATURE_CHECKS_02 = pd.read_csv(feature_checks_path)
CANDIDATE_SETS: dict[str, list[str]] = read_json(candidate_sets_path)
FEATURE_REGISTRY = pd.read_csv(feature_registry_path)
ARTIFACT_MANIFEST_02 = pd.read_csv(artifact_manifest_path)

assert READINESS_01["status"] == "complete"
assert READINESS_02["status"] == "complete"
assert READINESS_02["symmetry_failures"] == 0
assert READINESS_02["blocking_feature_check_failures"] == 0
assert FEATURE_CHECKS_02["Passed"].astype(bool).all()
assert READINESS_02["split_contract_sha256"] == SPLITS["contract_sha256"]
assert (
    READINESS_02["feature_contract_sha256"]
    == FEATURE_CONFIG["feature_contract_sha256"]
)
assert (
    FEATURE_CONFIG["source_split_contract_sha256"]
    == SPLITS["contract_sha256"]
)

HISTORICAL_STORE_PATH = (
    PROCESSED / "historical_matchup_feature_store_v1.parquet"
)
STAGE2_STORE_PATH = PROCESSED / "stage2_matchup_feature_store_v1.parquet"
OUTER_FOLDS_PATH = INTERIM / "fold_manifest_outer.parquet"
INNER_FOLDS_PATH = INTERIM / "fold_manifest_inner.parquet"
LOCKED_FOLDS_PATH = INTERIM / "fold_manifest_locked_benchmark.parquet"

required_data_files = [
    HISTORICAL_STORE_PATH,
    STAGE2_STORE_PATH,
    OUTER_FOLDS_PATH,
    INNER_FOLDS_PATH,
    LOCKED_FOLDS_PATH,
]
missing_data = [str(path) for path in required_data_files if not path.exists()]
assert not missing_data, f"Required data artifacts are missing: {missing_data}"

# The historical store is small enough to hash; Stage 2 is intentionally not
# read or re-hashed here because it is not part of model selection.
expected_historical_hash = READINESS_02["artifact_sha256"][
    "historical_matchup_feature_store_v1"
]
actual_historical_hash = sha256_file(HISTORICAL_STORE_PATH)
assert actual_historical_hash == expected_historical_hash, (
    "The historical feature store changed after notebook 02."
)

candidate_union = sorted(
    set().union(*(set(columns) for columns in CANDIDATE_SETS.values()))
)
assert len(candidate_union) == int(READINESS_02["candidate_feature_union"])

METADATA_COLUMNS = [
    "TargetKey",
    "GameKey",
    "Gender",
    "Season",
    "DayNum",
    "Team1ID",
    "Team2ID",
    "Team1Win",
    "Team1Margin",
    "DatasetRole",
    "CompactUniverseEligible",
    "RichUniverseEligible",
    "PrimaryModelRoute",
    "PooledChallengerEligible",
    "FeatureRoute",
]

# Read the Parquet schema without materializing the large matrix.
try:
    import pyarrow.parquet as pq

    historical_schema_columns = set(
        pq.ParquetFile(HISTORICAL_STORE_PATH).schema.names
    )
except Exception:
    # pandas can still read the file, but notebook 00/02 normally installed pyarrow.
    historical_schema_columns = set(
        pd.read_parquet(HISTORICAL_STORE_PATH).columns
    )

projected_columns = [
    column
    for column in METADATA_COLUMNS + candidate_union
    if column in historical_schema_columns
]
missing_metadata = {
    "TargetKey",
    "Gender",
    "Season",
    "Team1ID",
    "Team2ID",
    "Team1Win",
    "Team1Margin",
    "DatasetRole",
}.difference(projected_columns)
assert not missing_metadata, f"Historical store missing metadata: {missing_metadata}"

# The filter ensures locked labels never enter the development DataFrame.
development = pd.read_parquet(
    HISTORICAL_STORE_PATH,
    columns=projected_columns,
    filters=[("DatasetRole", "==", "development")],
)
outer_folds = pd.read_parquet(OUTER_FOLDS_PATH)
inner_folds = pd.read_parquet(INNER_FOLDS_PATH)
locked_fold_manifest = pd.read_parquet(LOCKED_FOLDS_PATH)

for column in development.select_dtypes(include=["float64"]).columns:
    development[column] = development[column].astype("float32")
for column in development.select_dtypes(include=["int64"]).columns:
    if column not in {"Season", "Team1ID", "Team2ID"}:
        minimum = development[column].min()
        maximum = development[column].max()
        if pd.notna(minimum) and pd.notna(maximum):
            if np.iinfo(np.int16).min <= minimum <= maximum <= np.iinfo(np.int16).max:
                development[column] = development[column].astype("int16")
            elif np.iinfo(np.int32).min <= minimum <= maximum <= np.iinfo(np.int32).max:
                development[column] = development[column].astype("int32")

LOCKED_SEASONS = set(map(int, SPLITS["locked_benchmark_seasons"]))
DEVELOPMENT_LAST_SEASON = int(SPLITS["development_last_season"])
TARGET_SEASON = int(SPLITS["target_season"])

assert development["DatasetRole"].eq("development").all()
assert development["Season"].max() <= DEVELOPMENT_LAST_SEASON
assert not development["Season"].isin(LOCKED_SEASONS).any()
assert development["TargetKey"].is_unique
assert development["Team1Win"].isin([0, 1]).all()
assert development["Team1ID"].lt(development["Team2ID"]).all()
numeric_development = development[candidate_union].select_dtypes(
    include=[np.number, "bool"]
)
numeric_development_array = numeric_development.to_numpy(
    dtype=np.float32,
    na_value=np.nan,
)
assert not np.isinf(numeric_development_array).any(), (
    "The development feature matrix contains positive or negative infinity. "
    "NaN is allowed for fold-fitted imputation, but infinity is not."
)
del numeric_development_array

print(
    json.dumps(
        {
            "development_rows_loaded": int(len(development)),
            "development_seasons": sorted(
                map(int, development["Season"].unique())
            ),
            "candidate_union": len(candidate_union),
            "historical_store_size_mb": round(
                HISTORICAL_STORE_PATH.stat().st_size / 1024**2, 3
            ),
            "stage2_store_size_mb_not_loaded": round(
                STAGE2_STORE_PATH.stat().st_size / 1024**2, 3
            ),
            "outer_folds": int(len(outer_folds)),
            "inner_folds": int(len(inner_folds)),
            "locked_manifest_rows_loaded_without_labels": int(
                len(locked_fold_manifest)
            ),
            "rss_mb_after_projected_load": round(rss_mb(), 2),
        },
        indent=2,
    )
)
input_fingerprint = {
    "split_contract_sha256": SPLITS["contract_sha256"],
    "feature_contract_sha256": FEATURE_CONFIG["feature_contract_sha256"],
    "historical_store_sha256": actual_historical_hash,
    "historical_store_size_mb": round(
        HISTORICAL_STORE_PATH.stat().st_size / 1024**2, 3
    ),
    "stage2_store_size_mb_not_loaded": round(
        STAGE2_STORE_PATH.stat().st_size / 1024**2, 3
    ),
    "development_rows": int(len(development)),
    "development_seasons": sorted(
        map(int, development["Season"].unique())
    ),
    "candidate_feature_union": int(len(candidate_union)),
}
(MODEL_REPORTS / "input_fingerprint.json").write_text(
    json.dumps(input_fingerprint, indent=2),
    encoding="utf-8",
)
log_event("development_data_loaded", **input_fingerprint)


## 2. Freeze the model-comparison contract

The contract records the matched validation seasons, model families, architectures, metrics, feature limits, calibration rules, and resource limits before any new results are generated.

In [ ]:
MODEL_CONFIG: dict[str, Any] = {
    "model_contract_version": 4,
    "source_split_contract_sha256": SPLITS["contract_sha256"],
    "source_feature_contract_sha256": FEATURE_CONFIG[
        "feature_contract_sha256"
    ],
    "target_season": TARGET_SEASON,
    "development_last_season": DEVELOPMENT_LAST_SEASON,
    "locked_benchmark_seasons": sorted(LOCKED_SEASONS),
    "matched_outer_validation_seasons": MATCHED_VALIDATION_SEASONS,
    "primary_selection_metric": "macro_mean_season_brier",
    "winner_reporting": {
        "lowest_score_winner": "minimum macro mean season Brier",
        "recommended_winner": "lowest complexity within one standard error of the minimum",
        "classification_threshold": 0.50,
    },
    "secondary_metrics": [
        "game_weighted_brier",
        "log_loss",
        "roc_auc",
        "average_precision",
        "pr_auc",
        "accuracy",
        "balanced_accuracy",
        "precision",
        "recall",
        "f1",
        "matthews_correlation",
        "calibration_intercept",
        "calibration_slope",
        "expected_calibration_error",
    ],
    "architectures": {
        "separate": ["men", "women"],
        "pooled": ["pooled_common"],
        "prediction_level_challengers": [
            "partial_pooling",
            "constrained_cross_family_ensemble",
        ],
    },
    "targets": {
        "classification": "Team1Win",
        "margin_regression": "Team1Margin",
    },
    "feature_selection": {
        "fit_scope": "inner_or_outer_training_rows_only",
        "missingness_max": 0.45,
        "minimum_nonmissing_rows": 40,
        "near_zero_variance_tolerance": 1e-10,
        "correlation_prune_threshold": 0.985,
        "linear_feature_cap": MODE["linear_feature_cap"],
        "tree_feature_cap": MODE["tree_feature_cap"],
        "neural_feature_cap": MODE["neural_feature_cap"],
        "margin_feature_cap": MODE["margin_feature_cap"],
        "row_adaptive_linear_divisor": 14,
        "row_adaptive_tree_divisor": 8,
        "row_adaptive_neural_divisor": 12,
        "row_adaptive_margin_divisor": 9,
        "block_balancing": True,
        "mandatory_core_signals": [
            "seed difference",
            "margin-aware Elo",
            "robust scoring margin",
            "schedule strength",
            "adjusted net efficiency",
            "Massey consensus when legally available",
            "gender context for pooled models",
        ],
    },
    "model_families": [
        "constant_probability",
        "seed_logistic",
        "elo_seed_logistic",
        "elastic_net_logistic",
        "histogram_gradient_boosting",
        "xgboost_classifier",
        "lightgbm_classifier",
        "pytorch_mlp",
        "tensorflow_mlp",
        "ridge_margin",
        "xgboost_margin",
        "lightgbm_margin",
    ],
    "optimization": {
        "sampler": "independent_tpe",
        "experimental_sampler_options": False,
        "parallel_trials": False,
        "persistent_sqlite_studies": True,
    },
    "boosting_rules": {
        "outer_validation_never_used_for_early_stopping": True,
        "outer_round_count": "median inner-fold best iteration",
        "xgboost_tree_method": "hist",
        "xgboost_device": "cpu",
        "lightgbm_deterministic": True,
        "lightgbm_force_col_wise": True,
        "maximum_threads": MAX_THREADS,
    },
    "neural_rules": {
        "outer_validation_never_used_for_epoch_selection": True,
        "outer_epoch_count": "median inner-fold best epoch",
        "frameworks": ["pytorch", "tensorflow"],
        "matched_architecture": True,
        "deterministic_seeds": True,
        "maximum_threads": MAX_THREADS,
        "tensorflow_prediction_path": "direct_eager_model_call",
        "tensorflow_session_cleanup": "tf.keras.utils.clear_session",
    },
    "calibration": {
        "fit_source": "inner_oof_predictions_only",
        "probability_methods": [
            "identity",
            "temperature",
            "platt",
            "beta",
            "spline",
            "isotonic",
        ],
        "margin_methods": [
            "raw_platt",
            "raw_spline",
            "raw_isotonic",
        ],
        "selection": "cross_fitted_by_inner_validation_season_one_se_rule",
    },
    "ablation": {
        "reference_models": ["elastic_logistic", "xgb_classifier"],
        "method": "forward feature-block ladder under identical outer folds",
        "selection_use": "diagnostic; no locked-benchmark influence",
    },
    "ensemble": {
        "weights": "nonnegative_simplex",
        "objective": "macro_season_brier_plus_l2_penalty",
        "maximum_members": 6,
        "redundancy_correlation_threshold": 0.995,
        "partial_pooling": "gender_specific_prequential_blend",
    },
    "resource_policy": {
        "checkpoint_checked_before_memory_gate": True,
        "memory_pressure_is_not_model_failure": True,
        "hard_headroom_gb_by_model": {
            "constant_probability": 0.10,
            "seed_logistic": 0.20,
            "elo_seed_logistic": 0.20,
            "elastic_logistic": 0.30,
            "ridge_margin": 0.30,
            "hist_classifier": 0.40,
            "xgb_classifier": 0.45,
            "xgb_margin": 0.45,
            "lgb_classifier": 0.45,
            "lgb_margin": 0.45,
            "torch_mlp": 0.55,
            "tensorflow_mlp": 0.60,
        },
        "absolute_emergency_floor_gb": 0.25,
        "soft_warning_floor_gb": 1.00,
        "minimum_virtual_memory_headroom_gb": 0.50,
        "windows_working_set_trim_under_pressure": True,
        "model_execution": "sequential",
        "checkpoint_resume": True,
    },
    "benchmark_boundary": {
        "locked_labels_loaded_here": False,
        "locked_evaluation_begins_in_replacement_notebook_04": True,
    },
    "run_profile": RUN_PROFILE,
}

model_contract_payload = json.dumps(
    MODEL_CONFIG, sort_keys=True, separators=(",", ":")
)
MODEL_CONFIG["model_contract_sha256"] = hashlib.sha256(
    model_contract_payload.encode("utf-8")
).hexdigest()

model_config_path = CONFIG_DIR / "modeling.yaml"
model_config_path.write_text(
    yaml.safe_dump(MODEL_CONFIG, sort_keys=False),
    encoding="utf-8",
)
(MODEL_REPORTS / "model_contract.json").write_text(
    json.dumps(MODEL_CONFIG, indent=2),
    encoding="utf-8",
)

log_event(
    "model_contract_frozen",
    model_contract_sha256=MODEL_CONFIG["model_contract_sha256"],
    run_profile=RUN_PROFILE,
    matched_validation_seasons=MATCHED_VALIDATION_SEASONS,
)

print("Model contract:", model_config_path)
print("Model contract SHA-256:", MODEL_CONFIG["model_contract_sha256"])
print("Run controls:", json.dumps(MODE, indent=2))

## 3. Logging, checkpoints, hashes, and memory protection

Every fold/model task writes an atomic checkpoint. The notebook can be rerun after interruption without repeating completed work. Human-readable logs and structured JSON events are also written for portfolio review.

In [ ]:
def canonical_json(value: Any) -> str:
    return json.dumps(
        value,
        sort_keys=True,
        separators=(",", ":"),
        default=lambda x: (
            x.item()
            if isinstance(x, np.generic)
            else str(x)
        ),
    )


def object_sha256(value: Any) -> str:
    return hashlib.sha256(canonical_json(value).encode("utf-8")).hexdigest()


def sanitize_name(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value)).strip("_")


def atomic_write_json(path: Path, value: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(value, indent=2, default=str),
        encoding="utf-8",
    )
    temporary.replace(path)


def atomic_write_csv(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary, index=False)
    temporary.replace(path)


def atomic_write_parquet(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_parquet(
        temporary,
        index=False,
        compression="zstd",
    )
    temporary.replace(path)


def parse_json_int_list(value: str | float | None) -> list[int]:
    if value is None or (
        isinstance(value, float) and np.isnan(value)
    ):
        return []
    return [int(item) for item in json.loads(value)]


def clip_probability(probability: np.ndarray | pd.Series) -> np.ndarray:
    return np.clip(
        np.asarray(probability, dtype=np.float64),
        1e-6,
        1.0 - 1e-6,
    )


class MemoryPressurePause(RuntimeError):
    """A resumable resource pause, not a model-training failure."""


MODEL_HARD_HEADROOM_GB: dict[str, float] = {
    "constant_probability": 0.10,
    "seed_logistic": 0.20,
    "elo_seed_logistic": 0.20,
    "elastic_logistic": 0.30,
    "ridge_margin": 0.30,
    "hist_classifier": 0.40,
    "xgb_classifier": 0.45,
    "xgb_margin": 0.45,
    "lgb_classifier": 0.45,
    "lgb_margin": 0.45,
    "torch_mlp": 0.55,
    "tensorflow_mlp": 0.60,
}
ABSOLUTE_EMERGENCY_FLOOR_GB = 0.25
SOFT_WARNING_FLOOR_GB = 1.00
MINIMUM_VIRTUAL_HEADROOM_GB = 0.50
LOW_MEMORY_WARNING_EMITTED = False


def memory_snapshot() -> dict[str, float]:
    physical = psutil.virtual_memory()
    swap = psutil.swap_memory()
    return {
        "total_physical_gb": physical.total / 1024**3,
        "available_physical_gb": physical.available / 1024**3,
        "available_physical_fraction": physical.available / max(physical.total, 1),
        "process_rss_mb": rss_mb(),
        "swap_total_gb": swap.total / 1024**3,
        "swap_free_gb": swap.free / 1024**3,
    }


def top_memory_processes(limit: int = 10) -> pd.DataFrame:
    records: list[dict[str, Any]] = []
    for process in psutil.process_iter(
        attrs=["pid", "name", "memory_info"]
    ):
        try:
            info = process.info
            memory_info = info.get("memory_info")
            if memory_info is None:
                continue
            records.append(
                {
                    "Process": info.get("name") or "unknown",
                    "PID": int(info["pid"]),
                    "WorkingSetGB": float(memory_info.rss / 1024**3),
                }
            )
        except (psutil.NoSuchProcess, psutil.AccessDenied, psutil.ZombieProcess):
            continue
    if not records:
        return pd.DataFrame(columns=["Process", "PID", "WorkingSetGB"])
    return (
        pd.DataFrame(records)
        .sort_values("WorkingSetGB", ascending=False)
        .head(limit)
        .reset_index(drop=True)
    )


def trim_current_process_working_set() -> bool:
    """Ask Windows to release reclaimable pages from this Python process."""
    if os.name != "nt":
        return False
    try:
        kernel32 = ctypes.windll.kernel32
        process_handle = kernel32.GetCurrentProcess()
        result = kernel32.SetProcessWorkingSetSize(
            process_handle,
            ctypes.c_size_t(-1),
            ctypes.c_size_t(-1),
        )
        return bool(result)
    except Exception:
        return False


def release_runtime_memory(*, aggressive: bool = False) -> dict[str, float]:
    """Release framework state and collect Python objects between tasks."""
    if tf is not None:
        try:
            tf.keras.utils.clear_session(free_memory=True)
        except Exception:
            pass
    gc.collect()
    if aggressive:
        trim_current_process_working_set()
        gc.collect()
    return memory_snapshot()


def model_memory_requirement_gb(model_name: str | None) -> float:
    if model_name is None:
        return ABSOLUTE_EMERGENCY_FLOOR_GB
    base_name = str(model_name).split("__", maxsplit=1)[0]
    return float(
        MODEL_HARD_HEADROOM_GB.get(base_name, 0.40)
    )


def ensure_memory_headroom(
    stage: str,
    *,
    model_name: str | None = None,
) -> dict[str, float]:
    """Apply a model-specific hard floor and a nonblocking soft warning.

    This function is called only after a checkpoint miss. Low memory never
    becomes a model failure: the notebook pauses once, keeps completed
    checkpoints, and can be resumed after memory is freed.
    """
    snapshot = release_runtime_memory(aggressive=False)
    required_gb = max(
        ABSOLUTE_EMERGENCY_FLOOR_GB,
        model_memory_requirement_gb(model_name),
    )

    if snapshot["available_physical_gb"] < SOFT_WARNING_FLOOR_GB:
        snapshot = release_runtime_memory(aggressive=True)

    available_gb = snapshot["available_physical_gb"]
    swap_free_gb = snapshot["swap_free_gb"]
    below_physical_floor = available_gb < required_gb
    below_virtual_floor = (
        snapshot["swap_total_gb"] > 0
        and swap_free_gb < MINIMUM_VIRTUAL_HEADROOM_GB
    )

    if below_physical_floor or below_virtual_floor:
        process_table = top_memory_processes(limit=10)
        process_text = (
            process_table.to_string(index=False)
            if not process_table.empty
            else "Process information unavailable."
        )
        raise MemoryPressurePause(
            f"Execution paused before {stage}; no model was marked failed. "
            f"Model={model_name or 'unknown'} requires approximately "
            f"{required_gb:.2f} GB of physical headroom. "
            f"Available physical RAM={available_gb:.2f} GB "
            f"({snapshot['available_physical_fraction']:.1%}); "
            f"free virtual-memory backing={swap_free_gb:.2f} GB. "
            "Close memory-heavy applications or old Jupyter/Python kernels, "
            "then rerun this cell. Completed checkpoints will be reused.\n\n"
            "Largest working sets at the time of the pause:\n"
            f"{process_text}"
        )

    if available_gb < SOFT_WARNING_FLOOR_GB:
        global LOW_MEMORY_WARNING_EMITTED
        warning_message = (
            f"Low system RAM before {stage}: {available_gb:.2f} GB available. "
            f"The {model_name or 'current'} task is allowed because its "
            f"model-specific hard floor is {required_gb:.2f} GB. "
            "Execution remains sequential and checkpointed."
        )
        if not LOW_MEMORY_WARNING_EMITTED:
            LOGGER.warning(warning_message)
            LOW_MEMORY_WARNING_EMITTED = True
        log_event(
            "low_memory_soft_warning",
            stage=stage,
            model=model_name,
            required_gb=required_gb,
            **snapshot,
        )

    return {
        **snapshot,
        "required_physical_headroom_gb": required_gb,
    }


RESOURCE_LOG: list[dict[str, Any]] = []


class ResourceTimer:
    def __init__(
        self,
        stage: str,
        *,
        outer_fold_id: str | None = None,
        model_name: str | None = None,
    ) -> None:
        self.stage = stage
        self.outer_fold_id = outer_fold_id
        self.model_name = model_name

    def __enter__(self) -> "ResourceTimer":
        self.start_time = time.perf_counter()
        self.start_rss = rss_mb()
        self.start_available_gb = (
            psutil.virtual_memory().available / 1024**3
        )
        return self

    def __exit__(self, exc_type, exc, tb) -> None:
        end_rss = rss_mb()
        end_available_gb = psutil.virtual_memory().available / 1024**3
        RESOURCE_LOG.append(
            {
                "Stage": self.stage,
                "OuterFoldID": self.outer_fold_id,
                "Model": self.model_name,
                "ElapsedSeconds": time.perf_counter() - self.start_time,
                "StartRSSMB": self.start_rss,
                "EndRSSMB": end_rss,
                "RSSDeltaMB": end_rss - self.start_rss,
                "AvailableRAMGBStart": self.start_available_gb,
                "AvailableRAMGBEnd": end_available_gb,
                "Succeeded": exc_type is None,
            }
        )


EXPECTED_CONTRACTS = {
    "split": SPLITS["contract_sha256"],
    "feature": FEATURE_CONFIG["feature_contract_sha256"],
    "model": MODEL_CONFIG["model_contract_sha256"],
    "historical_store": expected_historical_hash,
}


def checkpoint_is_valid(directory: Path) -> bool:
    metadata_path = directory / "metadata.json"
    prediction_path = directory / "outer_predictions.parquet"
    inner_path = directory / "inner_oof.parquet"
    if not (
        metadata_path.exists()
        and prediction_path.exists()
        and inner_path.exists()
    ):
        return False
    try:
        metadata = read_json(metadata_path)
    except Exception:
        return False
    return (
        metadata.get("status") == "complete"
        and metadata.get("contracts") == EXPECTED_CONTRACTS
    )


print("Checkpoint root:", CACHE_DIR)
print("Expected contracts:", json.dumps(EXPECTED_CONTRACTS, indent=2))


## 4. Fold-fitted feature selection

Feature selection is performed separately inside each training fold. The validation season never influences missing-value handling, scaling, feature ranking, correlation pruning, or dimensionality.

In [ ]:
FEATURE_REGISTRY = FEATURE_REGISTRY.drop_duplicates("Feature").copy()
REGISTRY_BY_FEATURE = (
    FEATURE_REGISTRY.set_index("Feature").to_dict(orient="index")
)


def underlying_feature_name(column: str) -> str:
    for prefix in ("diff__", "absdiff__", "mean__", "max__", "min__"):
        if column.startswith(prefix):
            return column[len(prefix):]
    return column


def infer_feature_block(column: str) -> str:
    if column in REGISTRY_BY_FEATURE:
        return str(REGISTRY_BY_FEATURE[column].get("Block", "unregistered"))
    base = underlying_feature_name(column)
    if base in REGISTRY_BY_FEATURE:
        return str(REGISTRY_BY_FEATURE[base].get("Block", "unregistered"))
    if column.startswith("seedprior__"):
        return "prequential_seed_priors"
    if column.startswith("matchup__seed"):
        return "selection_committee_prior"
    if column.startswith("interaction__"):
        return "matchup_interactions"
    if column.startswith("context__") or column.startswith("availability__"):
        return "availability_and_context"
    if "massey__" in column:
        return "massey_consensus"
    if "coach__" in column:
        return "prior_coach_history"
    if "prior__" in column:
        return "prior_program_history"
    if "schedule__" in column:
        return "schedule_and_accomplishment"
    if "conference__" in column:
        return "conference_context"
    if "adj__" in column:
        return "opponent_adjusted_efficiency"
    if "rating__" in column:
        return "dynamic_ratings"
    if "detailed__" in column:
        return "detailed_efficiency"
    if "compact__" in column:
        return "compact_performance"
    if "norm__" in column:
        return "within_season_normalization"
    return "other"


FEATURE_TO_BLOCK = {
    column: infer_feature_block(column)
    for column in candidate_union
}

BLOCK_BASE_QUOTAS = {
    "selection_committee_prior": 14,
    "prequential_seed_priors": 10,
    "compact_performance": 30,
    "dynamic_ratings": 24,
    "global_strength_ratings": 20,
    "schedule_and_accomplishment": 22,
    "conference_context": 10,
    "detailed_efficiency": 34,
    "opponent_adjusted_efficiency": 18,
    "prior_program_history": 10,
    "prior_coach_history": 7,
    "massey_consensus": 24,
    "matchup_interactions": 12,
    "within_season_normalization": 18,
    "availability_and_context": 10,
    "geography": 8,
    "other": 8,
}

MANDATORY_TOKEN_GROUPS = [
    ("matchup__seed_diff",),
    ("rating__elo_mov_538", "rating__elo_mov_log", "rating__elo_standard"),
    ("compact__margin_trim02", "compact__margin_mean"),
    ("schedule__opponent_strength_mean", "schedule__rpi"),
    ("adj__net_rtg_a50p0", "adj__net_rtg_a10p0"),
    ("massey__strength_mean", "massey__rank_mean"),
]


def first_matching_feature(
    candidates: Sequence[str],
    tokens: Sequence[str],
) -> str | None:
    for token in tokens:
        exact_or_containing = [
            column
            for column in candidates
            if column == token
            or column.endswith(token)
            or token in column
        ]
        directional = [
            column
            for column in exact_or_containing
            if column.startswith("diff__")
            or column.startswith("matchup__")
            or column.startswith("interaction__")
        ]
        if directional:
            return sorted(directional)[0]
        if exact_or_containing:
            return sorted(exact_or_containing)[0]
    return None


def mandatory_features(
    candidates: Sequence[str],
    *,
    pooled: bool,
) -> list[str]:
    chosen: list[str] = []
    for tokens in MANDATORY_TOKEN_GROUPS:
        match = first_matching_feature(candidates, tokens)
        if match is not None:
            chosen.append(match)
    if pooled and "context__women" in candidates:
        chosen.append("context__women")
    return list(dict.fromkeys(chosen))


def baseline_features(
    model_name: str,
    candidates: Sequence[str],
    *,
    pooled: bool,
) -> list[str]:
    candidate_set = set(candidates)
    if model_name == "constant_probability":
        return []

    if model_name == "seed_logistic":
        ordered = [
            column
            for column in candidates
            if (
                column == "matchup__seed_diff"
                or column == "matchup__seed_gap_abs"
                or column == "matchup__team1_is_seed_favorite"
                or column == "matchup__equal_seed"
                or column.startswith("seedprior__")
            )
        ]
        # Keep a transparent baseline rather than every redundant seed prior.
        preferred = [
            column
            for column in ordered
            if any(
                token in column
                for token in (
                    "matchup__seed_diff",
                    "matchup__seed_gap_abs",
                    "team1_is_seed_favorite",
                    "pc8",
                    "overall",
                )
            )
        ]
        selected = preferred[:12] if preferred else ordered[:12]
    elif model_name == "elo_seed_logistic":
        selected = baseline_features(
            "seed_logistic", candidates, pooled=pooled
        )
        elo = [
            column
            for column in candidates
            if (
                "rating__elo_mov_538" in column
                or "rating__elo_mov_log" in column
                or "rating__elo_standard" in column
            )
            and column.startswith(("diff__", "absdiff__"))
        ]
        selected += sorted(elo)[:12]
    else:
        selected = []

    if pooled and "context__women" in candidate_set:
        selected.append("context__women")
    return list(dict.fromkeys(selected))



def finite_column_medians(matrix: np.ndarray) -> np.ndarray:
    """Return finite column medians without emitting all-NaN warnings."""
    matrix = np.asarray(matrix)
    medians = np.zeros(matrix.shape[1], dtype=np.float64)
    finite_counts = np.isfinite(matrix).sum(axis=0)
    for index in np.flatnonzero(finite_counts):
        medians[index] = float(np.nanmedian(matrix[:, index]))
    return medians


def vectorized_correlations(
    frame: pd.DataFrame,
    features: Sequence[str],
    target: np.ndarray,
) -> np.ndarray:
    if not features:
        return np.array([], dtype=np.float64)
    # pandas 3 may expose a read-only NumPy view. The selector performs
    # fold-local in-place centering/imputation, so request an explicit writable copy.
    matrix = np.array(
        frame[list(features)].to_numpy(
            dtype=np.float64,
            na_value=np.nan,
        ),
        dtype=np.float64,
        copy=True,
        order="C",
    )
    medians = finite_column_medians(matrix)
    missing = ~np.isfinite(matrix)
    if missing.any():
        matrix[missing] = np.take(medians, np.where(missing)[1])
    matrix -= matrix.mean(axis=0, keepdims=True)
    centered_target = target.astype(np.float64) - np.mean(target)
    numerator = matrix.T @ centered_target
    denominator = np.sqrt(
        np.sum(matrix * matrix, axis=0)
        * np.sum(centered_target * centered_target)
    )
    correlation = np.divide(
        numerator,
        denominator,
        out=np.zeros_like(numerator, dtype=np.float64),
        where=denominator > 0,
    )
    return correlation


@dataclass
class SelectionResult:
    selected_features: list[str]
    audit: pd.DataFrame
    effective_cap: int
    precorrelation_pool: list[str]
    mandatory_features: list[str]
    selector_hash: str


def effective_feature_cap(
    *,
    family: str,
    training_rows: int,
) -> int:
    if family == "linear":
        fixed = int(MODE["linear_feature_cap"])
        adaptive = max(12, training_rows // int(
            MODEL_CONFIG["feature_selection"]["row_adaptive_linear_divisor"]
        ))
    elif family == "neural":
        fixed = int(MODE["neural_feature_cap"])
        adaptive = max(16, training_rows // int(
            MODEL_CONFIG["feature_selection"]["row_adaptive_neural_divisor"]
        ))
    elif family == "margin":
        fixed = int(MODE["margin_feature_cap"])
        adaptive = max(20, training_rows // int(
            MODEL_CONFIG["feature_selection"]["row_adaptive_margin_divisor"]
        ))
    else:
        fixed = int(MODE["tree_feature_cap"])
        adaptive = max(24, training_rows // int(
            MODEL_CONFIG["feature_selection"]["row_adaptive_tree_divisor"]
        ))
    return max(8, min(fixed, adaptive))


def fit_block_aware_selector(
    training: pd.DataFrame,
    candidates: Sequence[str],
    *,
    target_column: str,
    family: str,
    pooled: bool,
) -> SelectionResult:
    assert target_column in {"Team1Win", "Team1Margin"}
    candidates = [
        column
        for column in candidates
        if column in training.columns
        and pd.api.types.is_numeric_dtype(training[column])
    ]
    assert candidates, "No candidate features are available."

    target = pd.to_numeric(
        training[target_column], errors="coerce"
    ).to_numpy(dtype=np.float64)
    assert np.isfinite(target).all()

    missingness = training[candidates].isna().mean()
    nonmissing = training[candidates].notna().sum()
    unique_counts = training[candidates].nunique(dropna=True)

    eligible = [
        column
        for column in candidates
        if missingness[column]
        <= float(MODEL_CONFIG["feature_selection"]["missingness_max"])
        and nonmissing[column]
        >= min(
            int(MODEL_CONFIG["feature_selection"]["minimum_nonmissing_rows"]),
            len(training),
        )
        and unique_counts[column] > 1
    ]

    mandatory = [
        column
        for column in mandatory_features(candidates, pooled=pooled)
        if column in candidates
        and unique_counts.get(column, 0) > 1
    ]
    eligible = list(dict.fromkeys(mandatory + eligible))
    assert eligible, "Every feature was removed by eligibility filters."

    global_corr = vectorized_correlations(training, eligible, target)
    per_group_corr: dict[str, list[float]] = {
        feature: [] for feature in eligible
    }

    group_columns = ["Season", "Gender"] if pooled else ["Season"]
    grouper = group_columns[0] if len(group_columns) == 1 else group_columns
    for _, group in training.groupby(grouper, observed=True, sort=True):
        if len(group) < 20:
            continue
        group_target = pd.to_numeric(
            group[target_column], errors="coerce"
        ).to_numpy(dtype=np.float64)
        if np.nanstd(group_target) <= 0:
            continue
        correlations = vectorized_correlations(
            group, eligible, group_target
        )
        for feature, value in zip(eligible, correlations, strict=True):
            if np.isfinite(value):
                per_group_corr[feature].append(float(value))

    records: list[dict[str, Any]] = []
    for index, feature in enumerate(eligible):
        correlations = np.asarray(
            per_group_corr[feature], dtype=np.float64
        )
        nonzero = correlations[np.abs(correlations) > 1e-12]
        sign_consistency = (
            abs(np.mean(np.sign(nonzero)))
            if nonzero.size
            else 0.0
        )
        group_coverage = (
            len(correlations)
            / max(
                1,
                training[group_columns]
                .drop_duplicates()
                .shape[0],
            )
        )
        median_abs = (
            float(np.median(np.abs(correlations)))
            if correlations.size
            else 0.0
        )
        stability_iqr = (
            float(
                np.subtract(
                    *np.percentile(correlations, [75, 25])
                )
            )
            if correlations.size >= 2
            else 0.0
        )
        score = (
            0.48 * median_abs
            + 0.22 * abs(float(global_corr[index]))
            + 0.16 * sign_consistency
            + 0.10 * math.sqrt(max(group_coverage, 0.0))
            - 0.08 * float(missingness[feature])
            - 0.04 * min(stability_iqr, 1.0)
        )
        records.append(
            {
                "Feature": feature,
                "Block": FEATURE_TO_BLOCK.get(feature, "other"),
                "MissingRate": float(missingness[feature]),
                "NonMissingRows": int(nonmissing[feature]),
                "UniqueValues": int(unique_counts[feature]),
                "GlobalCorrelation": float(global_corr[index]),
                "MedianAbsoluteSeasonCorrelation": median_abs,
                "SeasonSignConsistency": float(sign_consistency),
                "SeasonCoverage": float(group_coverage),
                "SeasonCorrelationIQR": stability_iqr,
                "StabilityScore": float(score),
                "Mandatory": feature in mandatory,
            }
        )

    audit = pd.DataFrame(records).sort_values(
        ["Mandatory", "StabilityScore", "Feature"],
        ascending=[False, False, True],
    ).reset_index(drop=True)

    cap = effective_feature_cap(
        family=family,
        training_rows=len(training),
    )
    pool_cap = min(
        int(MODE["precorrelation_pool_cap"]),
        max(cap * 2, cap + 20),
    )

    # Scale block quotas to the requested family cap.
    quota_scale = cap / max(1, sum(BLOCK_BASE_QUOTAS.values()))
    block_choices: list[str] = []
    for block, group in audit.groupby("Block", observed=True, sort=False):
        base_quota = BLOCK_BASE_QUOTAS.get(block, 8)
        quota = max(
            2,
            min(
                base_quota,
                int(math.ceil(base_quota * max(0.65, quota_scale * 5))),
            ),
        )
        block_choices.extend(group.head(quota)["Feature"].tolist())

    ranked = audit["Feature"].tolist()
    pool = list(dict.fromkeys(mandatory + block_choices + ranked))
    pool = pool[:pool_cap]

    # Correlation pruning is bounded to the pre-screened pool.
    # Explicit writable copy is required because correlation pruning performs
    # in-place median imputation and pandas 3 can return a read-only array.
    pool_matrix = np.array(
        training[pool].to_numpy(
            dtype=np.float32,
            na_value=np.nan,
        ),
        dtype=np.float32,
        copy=True,
        order="C",
    )
    medians = finite_column_medians(pool_matrix).astype(np.float32)
    missing = ~np.isfinite(pool_matrix)
    if missing.any():
        pool_matrix[missing] = np.take(
            medians, np.where(missing)[1]
        )
    standard_deviation = pool_matrix.std(axis=0)
    safe_std = np.where(standard_deviation > 1e-12, standard_deviation, 1.0)
    normalized = (
        pool_matrix - pool_matrix.mean(axis=0, keepdims=True)
    ) / safe_std
    correlation = np.abs(
        np.clip(
            (normalized.T @ normalized)
            / max(1, normalized.shape[0] - 1),
            -1,
            1,
        )
    )

    selected: list[str] = []
    selected_indices: list[int] = []
    threshold = float(
        MODEL_CONFIG["feature_selection"]["correlation_prune_threshold"]
    )
    for index, feature in enumerate(pool):
        if feature in mandatory:
            selected.append(feature)
            selected_indices.append(index)
            continue
        if selected_indices and np.any(
            correlation[index, selected_indices] >= threshold
        ):
            continue
        selected.append(feature)
        selected_indices.append(index)
        if len(selected) >= cap:
            break

    # Mandatory features survive even if mutually correlated; trim only
    # nonmandatory tail if needed.
    if len(selected) > cap:
        mandatory_set = set(mandatory)
        retained = [f for f in selected if f in mandatory_set]
        retained += [
            f for f in selected if f not in mandatory_set
        ][: max(0, cap - len(retained))]
        selected = retained

    selected_set = set(selected)
    pool_set = set(pool)
    audit["InPrecorrelationPool"] = audit["Feature"].isin(pool_set)
    audit["Selected"] = audit["Feature"].isin(selected_set)
    audit["SelectionRank"] = audit["Feature"].map(
        {feature: index + 1 for index, feature in enumerate(selected)}
    )
    audit["ExclusionReason"] = np.select(
        [
            audit["Selected"],
            ~audit["InPrecorrelationPool"],
        ],
        [
            "selected",
            "below_bounded_prescreen",
        ],
        default="correlation_pruned_or_cap",
    )

    selector_payload = {
        "target": target_column,
        "family": family,
        "pooled": pooled,
        "training_seasons": sorted(
            map(int, training["Season"].unique())
        ),
        "training_rows": len(training),
        "candidate_hash": object_sha256(sorted(candidates)),
        "selected": selected,
        "contracts": EXPECTED_CONTRACTS,
    }
    selector_hash = object_sha256(selector_payload)
    return SelectionResult(
        selected_features=selected,
        audit=audit,
        effective_cap=cap,
        precorrelation_pool=pool,
        mandatory_features=mandatory,
        selector_hash=selector_hash,
    )


SELECTION_MEMO: dict[str, SelectionResult] = {}


def get_selection(
    training: pd.DataFrame,
    candidates: Sequence[str],
    *,
    target_column: str,
    family: str,
    pooled: bool,
    selection_key: str,
) -> SelectionResult:
    payload = {
        "selection_key": selection_key,
        "target_column": target_column,
        "family": family,
        "pooled": pooled,
        "training_seasons": sorted(
            map(int, training["Season"].unique())
        ),
        "training_rows": len(training),
        "candidate_hash": object_sha256(sorted(candidates)),
        "mode_caps": {
            "linear": MODE["linear_feature_cap"],
            "tree": MODE["tree_feature_cap"],
            "neural": MODE["neural_feature_cap"],
            "margin": MODE["margin_feature_cap"],
            "pool": MODE["precorrelation_pool_cap"],
        },
        "contracts": EXPECTED_CONTRACTS,
    }
    key = object_sha256(payload)
    if key in SELECTION_MEMO:
        return SELECTION_MEMO[key]

    directory = CACHE_DIR / "selectors" / key
    metadata_path = directory / "metadata.json"
    audit_path = directory / "audit.csv"
    if metadata_path.exists() and audit_path.exists():
        metadata = read_json(metadata_path)
        if metadata.get("payload") == payload:
            result = SelectionResult(
                selected_features=list(metadata["selected_features"]),
                audit=pd.read_csv(audit_path),
                effective_cap=int(metadata["effective_cap"]),
                precorrelation_pool=list(metadata["precorrelation_pool"]),
                mandatory_features=list(metadata["mandatory_features"]),
                selector_hash=str(metadata["selector_hash"]),
            )
            SELECTION_MEMO[key] = result
            return result

    result = fit_block_aware_selector(
        training,
        candidates,
        target_column=target_column,
        family=family,
        pooled=pooled,
    )
    directory.mkdir(parents=True, exist_ok=True)
    atomic_write_csv(audit_path, result.audit)
    atomic_write_json(
        metadata_path,
        {
            "payload": payload,
            "selected_features": result.selected_features,
            "effective_cap": result.effective_cap,
            "precorrelation_pool": result.precorrelation_pool,
            "mandatory_features": result.mandatory_features,
            "selector_hash": result.selector_hash,
        },
    )
    SELECTION_MEMO[key] = result
    return result


block_inventory = (
    pd.Series(FEATURE_TO_BLOCK, name="Block")
    .rename_axis("Feature")
    .reset_index()
    .groupby("Block", observed=True)
    .size()
    .sort_values(ascending=False)
    .to_frame("CandidateFeatures")
)
block_inventory.to_csv(
    MODEL_REPORTS / "candidate_feature_blocks.csv"
)
block_inventory


### Feature-selector preflight

The preflight checks the men's, women's, and pooled candidate banks across linear, tree, neural, and margin-model limits before training begins.

In [ ]:
def candidate_set_name(
    *,
    gender: str,
    universe: str,
    seed_aware: bool = True,
) -> str:
    route = "seed_aware" if seed_aware else "seed_free"
    if gender == "Pooled":
        assert universe == "rich"
        return f"pooled_rich_{route}"
    prefix = "men" if gender == "M" else "women"
    return f"{prefix}_{universe}_{route}"


def context_mask(
    frame: pd.DataFrame,
    *,
    seasons: Sequence[int],
    gender: str,
    universe: str,
) -> pd.Series:
    mask = frame["Season"].isin(list(map(int, seasons)))
    if gender != "Pooled":
        mask &= frame["Gender"].eq(gender)
    eligibility_column = (
        "CompactUniverseEligible"
        if universe == "compact"
        else "RichUniverseEligible"
    )
    if eligibility_column in frame.columns:
        mask &= frame[eligibility_column].astype(bool)
    return mask


selector_preflight_records: list[dict[str, Any]] = []
for gender in ("M", "W", "Pooled"):
    universe = "rich"
    key = candidate_set_name(
        gender=gender,
        universe=universe,
        seed_aware=True,
    )
    candidates = [
        column
        for column in CANDIDATE_SETS[key]
        if column in development.columns
    ]
    seasons = sorted(
        map(
            int,
            development.loc[
                context_mask(
                    development,
                    seasons=development["Season"].unique(),
                    gender=gender,
                    universe=universe,
                ),
                "Season",
            ].unique(),
        )
    )
    training = development.loc[
        context_mask(
            development,
            seasons=seasons,
            gender=gender,
            universe=universe,
        )
    ].copy()

    for family, target_column in (
        ("linear", "Team1Win"),
        ("tree", "Team1Win"),
        ("neural", "Team1Win"),
        ("margin", "Team1Margin"),
    ):
        result = fit_block_aware_selector(
            training,
            candidates,
            target_column=target_column,
            family=family,
            pooled=gender == "Pooled",
        )
        assert len(result.selected_features) <= result.effective_cap
        selector_preflight_records.append(
            {
                "GenderContext": gender,
                "ModelFamily": family,
                "TrainingRows": len(training),
                "CandidateFeatures": len(candidates),
                "PrecorrelationPool": len(result.precorrelation_pool),
                "AdaptiveCap": result.effective_cap,
                "SelectedFeatures": len(result.selected_features),
                "SelectedBlocks": result.audit.loc[
                    result.audit["Selected"], "Block"
                ].nunique(),
                "MandatoryRetained": len(
                    set(result.mandatory_features)
                    .intersection(result.selected_features)
                ),
            }
        )
    del training, result
    gc.collect()

selector_preflight = pd.DataFrame(selector_preflight_records)
selector_preflight.to_csv(
    MODEL_REPORTS / "selector_preflight.csv",
    index=False,
)

assert selector_preflight["SelectedFeatures"].gt(0).all()
assert (
    selector_preflight["SelectedFeatures"]
    <= selector_preflight["AdaptiveCap"]
).all()
log_event(
    "selector_preflight_complete",
    records=int(len(selector_preflight)),
    maximum_selected_features=int(
        selector_preflight["SelectedFeatures"].max()
    ),
)
selector_preflight

## 5. Evaluation metrics

Brier score is the primary probability metric. The notebook also reports log loss, ROC AUC, average precision, trapezoidal precision–recall AUC, accuracy, balanced accuracy, precision, recall, F1, Matthews correlation coefficient, confusion-matrix counts, calibration intercept and slope, and calibration error.

Precision, recall, F1, and confusion matrices use a fixed probability threshold of 0.50. They are diagnostic and do not determine the competition-oriented winner.

In [ ]:
CLASSIFICATION_THRESHOLD = 0.50


def binary_metric_bundle(
    y: np.ndarray,
    p: np.ndarray,
    *,
    threshold: float = CLASSIFICATION_THRESHOLD,
) -> dict[str, float | int]:
    y = np.asarray(y, dtype=int)
    p = clip_probability(p)
    predicted = (p >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        y, predicted, labels=[0, 1]
    ).ravel()

    if np.unique(y).size == 2:
        roc_auc = float(roc_auc_score(y, p))
        average_precision = float(average_precision_score(y, p))
        precision_curve, recall_curve, _ = precision_recall_curve(y, p)
        pr_auc = float(auc(recall_curve, precision_curve))
    else:
        roc_auc = np.nan
        average_precision = np.nan
        pr_auc = np.nan

    return {
        "Brier": float(np.mean((p - y) ** 2)),
        "LogLoss": float(log_loss(y, p, labels=[0, 1])),
        "ROCAUC": roc_auc,
        "AveragePrecision": average_precision,
        "AUCPR": pr_auc,
        "Accuracy": float(accuracy_score(y, predicted)),
        "BalancedAccuracy": float(balanced_accuracy_score(y, predicted)),
        "Precision": float(
            precision_score(y, predicted, zero_division=0)
        ),
        "Recall": float(recall_score(y, predicted, zero_division=0)),
        "F1": float(f1_score(y, predicted, zero_division=0)),
        "MCC": float(matthews_corrcoef(y, predicted)),
        "Threshold": float(threshold),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
        "PositiveRate": float(np.mean(y)),
        "PredictedPositiveRate": float(np.mean(predicted)),
    }


def season_metric_values(
    frame: pd.DataFrame,
    probability_column: str,
    *,
    outcome_column: str = "Team1Win",
) -> pd.DataFrame:
    rows = []
    for season, group in frame.groupby("Season", observed=True, sort=True):
        y = group[outcome_column].to_numpy(dtype=int)
        p = clip_probability(group[probability_column])
        rows.append(
            {
                "Season": int(season),
                "Rows": int(len(group)),
                **binary_metric_bundle(y, p),
            }
        )
    return pd.DataFrame(rows)


def calibration_intercept_slope(
    y: np.ndarray,
    p: np.ndarray,
) -> tuple[float, float]:
    """Estimate calibration intercept and slope.

    A constant forecast has no identifiable calibration slope. For that
    diagnostic baseline, record slope=0 and an intercept equal to the
    logit of the observed event rate. This keeps the result explicit,
    finite, and auditable without implying discriminative calibration.
    """
    y = np.asarray(y, dtype=int)
    p = clip_probability(p)
    if y.size == 0 or np.unique(y).size < 2:
        return np.nan, np.nan

    if np.unique(p).size == 1:
        observed_rate = float(
            np.clip(
                np.mean(y),
                1e-6,
                1.0 - 1e-6,
            )
        )
        return float(logit(observed_rate)), 0.0

    x = logit(p).reshape(-1, 1)
    try:
        model = LogisticRegression(
            C=1e6,
            solver="lbfgs",
            max_iter=5000,
        )
        model.fit(x, y)
        return (
            float(model.intercept_[0]),
            float(model.coef_[0, 0]),
        )
    except Exception:
        return np.nan, np.nan


def calibration_error(
    y: np.ndarray,
    p: np.ndarray,
    *,
    bins: int = 10,
) -> tuple[float, float]:
    """Return expected and maximum calibration error.

    Constant forecasts form one valid calibration group. They should not
    disappear because quantile binning has no distinct cut points.
    """
    y = np.asarray(y, dtype=float)
    p = clip_probability(p)
    if p.size == 0:
        return np.nan, np.nan

    unique_probability = np.unique(p)
    if unique_probability.size == 1:
        gap = abs(
            float(np.mean(y))
            - float(unique_probability[0])
        )
        return float(gap), float(gap)

    bin_count = min(
        int(bins),
        max(2, int(unique_probability.size)),
    )
    try:
        labels = pd.qcut(
            pd.Series(p),
            q=bin_count,
            duplicates="drop",
        )
    except Exception:
        labels = pd.cut(
            pd.Series(p),
            bins=bin_count,
            duplicates="drop",
        )

    table = pd.DataFrame(
        {"y": y, "p": p, "bin": labels}
    )
    grouped = table.groupby(
        "bin",
        observed=True,
    ).agg(
        rows=("y", "size"),
        observed=("y", "mean"),
        predicted=("p", "mean"),
    )

    if grouped.empty:
        gap = abs(
            float(np.mean(y))
            - float(np.mean(p))
        )
        return float(gap), float(gap)

    gap = np.abs(
        grouped["observed"]
        - grouped["predicted"]
    )
    return (
        float(
            np.average(
                gap,
                weights=grouped["rows"],
            )
        ),
        float(gap.max()),
    )


def brier_decomposition(
    y: np.ndarray,
    p: np.ndarray,
    *,
    bins: int = 10,
) -> dict[str, float]:
    """Return a grouped Murphy decomposition of the Brier score."""
    y = np.asarray(y, dtype=float)
    p = clip_probability(p)
    if p.size == 0:
        return {
            "Reliability": np.nan,
            "Resolution": np.nan,
            "Uncertainty": np.nan,
        }

    climatology = float(np.mean(y))
    uncertainty = float(
        climatology * (1.0 - climatology)
    )
    unique_probability = np.unique(p)

    if unique_probability.size == 1:
        return {
            "Reliability": float(
                (
                    float(unique_probability[0])
                    - climatology
                )
                ** 2
            ),
            "Resolution": 0.0,
            "Uncertainty": uncertainty,
        }

    bin_count = min(
        int(bins),
        max(2, int(unique_probability.size)),
    )
    try:
        labels = pd.qcut(
            pd.Series(p),
            q=bin_count,
            duplicates="drop",
        )
    except Exception:
        labels = pd.cut(
            pd.Series(p),
            bins=bin_count,
            duplicates="drop",
        )

    frame = pd.DataFrame(
        {"y": y, "p": p, "bin": labels}
    )
    grouped = frame.groupby(
        "bin",
        observed=True,
    ).agg(
        rows=("y", "size"),
        observed=("y", "mean"),
        predicted=("p", "mean"),
    )

    if grouped.empty:
        return {
            "Reliability": float(
                (
                    float(np.mean(p))
                    - climatology
                )
                ** 2
            ),
            "Resolution": 0.0,
            "Uncertainty": uncertainty,
        }

    weights = (
        grouped["rows"]
        / grouped["rows"].sum()
    )
    reliability = float(
        np.sum(
            weights
            * (
                grouped["predicted"]
                - grouped["observed"]
            )
            ** 2
        )
    )
    resolution = float(
        np.sum(
            weights
            * (
                grouped["observed"]
                - climatology
            )
            ** 2
        )
    )
    return {
        "Reliability": reliability,
        "Resolution": resolution,
        "Uncertainty": uncertainty,
    }


def evaluate_probability_frame(
    frame: pd.DataFrame,
    probability_column: str = "Prediction",
) -> tuple[dict[str, Any], pd.DataFrame]:
    y = frame["Team1Win"].to_numpy(dtype=int)
    p = clip_probability(frame[probability_column])
    by_season = season_metric_values(frame, probability_column)
    intercept, slope = calibration_intercept_slope(y, p)
    ece, mce = calibration_error(y, p)
    decomposition = brier_decomposition(y, p)
    overall = binary_metric_bundle(y, p)

    metrics = {
        "Rows": int(len(frame)),
        "Seasons": int(frame["Season"].nunique()),
        "MacroSeasonBrier": float(by_season["Brier"].mean()),
        "GameWeightedBrier": float(overall["Brier"]),
        "SeasonBrierStd": (
            float(by_season["Brier"].std(ddof=1))
            if len(by_season) > 1
            else 0.0
        ),
        "WorstSeasonBrier": float(by_season["Brier"].max()),
        **{
            key: value
            for key, value in overall.items()
            if key != "Brier"
        },
        "CalibrationIntercept": intercept,
        "CalibrationSlope": slope,
        "ECE": ece,
        "MCE": mce,
        **decomposition,
    }
    return metrics, by_season


def robust_probability_objective(frame: pd.DataFrame) -> float:
    metrics, _ = evaluate_probability_frame(frame)
    return float(
        metrics["MacroSeasonBrier"]
        + 0.10 * metrics["SeasonBrierStd"]
        + 0.05
        * max(
            0.0,
            metrics["WorstSeasonBrier"]
            - metrics["MacroSeasonBrier"],
        )
    )


def robust_margin_objective(
    frame: pd.DataFrame,
    prediction_column: str = "RawPrediction",
) -> float:
    season_rmse = []
    for _, group in frame.groupby("Season", observed=True, sort=True):
        error = (
            group[prediction_column].to_numpy(dtype=float)
            - group["Team1Margin"].to_numpy(dtype=float)
        )
        season_rmse.append(float(np.sqrt(np.mean(error**2))))
    if not season_rmse:
        return float("inf")
    values = np.asarray(season_rmse, dtype=float)
    return float(values.mean() + 0.10 * values.std(ddof=0))


def season_cluster_bootstrap_difference(
    frame: pd.DataFrame,
    *,
    probability_a: str,
    probability_b: str,
    repetitions: int,
    seed: int = SEED,
) -> dict[str, float]:
    seasons = np.array(
        sorted(map(int, frame["Season"].unique())),
        dtype=int,
    )
    if seasons.size < 2:
        return {
            "MeanBrierDifferenceAminusB": np.nan,
            "CILower": np.nan,
            "CIUpper": np.nan,
            "ProbabilityALowerBrier": np.nan,
        }
    rng = np.random.default_rng(seed)
    observed = []
    for season in seasons:
        group = frame.loc[frame["Season"].eq(season)]
        y = group["Team1Win"].to_numpy(dtype=float)
        loss_a = (clip_probability(group[probability_a]) - y) ** 2
        loss_b = (clip_probability(group[probability_b]) - y) ** 2
        observed.append(float(np.mean(loss_a - loss_b)))
    observed = np.asarray(observed, dtype=float)
    draws = np.empty(repetitions, dtype=float)
    for index in range(repetitions):
        sample = rng.choice(observed, size=len(observed), replace=True)
        draws[index] = sample.mean()
    return {
        "MeanBrierDifferenceAminusB": float(observed.mean()),
        "CILower": float(np.quantile(draws, 0.025)),
        "CIUpper": float(np.quantile(draws, 0.975)),
        "ProbabilityALowerBrier": float(np.mean(draws < 0)),
    }


print("Metric utilities ready.")

## 6. Cross-fitted probability calibration

Identity, Platt, temperature, beta, spline, and isotonic calibration are compared using inner-season out-of-fold predictions. The outer validation season is never used to choose a calibrator.

In [ ]:
class ConstantCalibrator:
    def __init__(self, probability: float) -> None:
        self.probability = float(probability)

    def predict(self, raw: np.ndarray) -> np.ndarray:
        return np.full(
            len(np.asarray(raw)),
            self.probability,
            dtype=np.float64,
        )


class IdentityCalibrator:
    def predict(self, raw: np.ndarray) -> np.ndarray:
        return clip_probability(raw)


class TemperatureCalibrator:
    def __init__(self, temperature: float) -> None:
        self.temperature = float(temperature)

    def predict(self, raw: np.ndarray) -> np.ndarray:
        logits = logit(clip_probability(raw))
        return clip_probability(expit(logits / self.temperature))


class SklearnCalibrator:
    def __init__(
        self,
        estimator: Any,
        transform: Callable[[np.ndarray], np.ndarray],
    ) -> None:
        self.estimator = estimator
        self.transform = transform

    def predict(self, raw: np.ndarray) -> np.ndarray:
        transformed = self.transform(np.asarray(raw, dtype=float))
        if hasattr(self.estimator, "predict_proba"):
            probability = self.estimator.predict_proba(
                transformed
            )[:, 1]
        else:
            probability = self.estimator.predict(transformed)
        return clip_probability(probability)


class IsotonicCalibrator:
    def __init__(self, estimator: IsotonicRegression) -> None:
        self.estimator = estimator

    def predict(self, raw: np.ndarray) -> np.ndarray:
        return clip_probability(
            self.estimator.predict(np.asarray(raw, dtype=float))
        )


def transform_probability_logit(raw: np.ndarray) -> np.ndarray:
    return logit(clip_probability(raw)).reshape(-1, 1)


def transform_beta(raw: np.ndarray) -> np.ndarray:
    p = clip_probability(raw)
    return np.column_stack([np.log(p), -np.log1p(-p)])


def transform_raw(raw: np.ndarray) -> np.ndarray:
    return np.asarray(raw, dtype=float).reshape(-1, 1)


def fit_calibrator(
    method: str,
    raw: np.ndarray,
    y: np.ndarray,
    *,
    raw_kind: str,
) -> Any:
    raw = np.asarray(raw, dtype=float)
    y = np.asarray(y, dtype=int)
    if len(raw) == 0:
        raise ValueError("Cannot fit a calibrator without rows.")
    if np.unique(y).size < 2:
        return ConstantCalibrator(float(np.mean(y)))

    if method == "identity":
        assert raw_kind == "probability"
        return IdentityCalibrator()

    if method == "temperature":
        assert raw_kind == "probability"
        logits = logit(clip_probability(raw))

        def objective(log_temperature: float) -> float:
            temperature = math.exp(log_temperature)
            probability = expit(logits / temperature)
            return float(np.mean((probability - y) ** 2))

        result = minimize_scalar(
            objective,
            bounds=(math.log(0.20), math.log(5.0)),
            method="bounded",
        )
        return TemperatureCalibrator(math.exp(float(result.x)))

    if method == "platt":
        estimator = LogisticRegression(
            C=1e3,
            solver="lbfgs",
            max_iter=5000,
        ).fit(transform_probability_logit(raw), y)
        return SklearnCalibrator(
            estimator, transform_probability_logit
        )

    if method == "beta":
        estimator = LogisticRegression(
            C=1e3,
            solver="lbfgs",
            max_iter=5000,
        ).fit(transform_beta(raw), y)
        return SklearnCalibrator(estimator, transform_beta)

    if method == "spline":
        estimator = Pipeline(
            [
                (
                    "spline",
                    SplineTransformer(
                        n_knots=4,
                        degree=2,
                        include_bias=False,
                        extrapolation="linear",
                    ),
                ),
                (
                    "logistic",
                    LogisticRegression(
                        C=10.0,
                        solver="lbfgs",
                        max_iter=5000,
                    ),
                ),
            ]
        ).fit(transform_probability_logit(raw), y)
        return SklearnCalibrator(
            estimator, transform_probability_logit
        )

    if method == "isotonic":
        if len(raw) < 100 or np.unique(raw).size < 10:
            raise ValueError(
                "Insufficient rows or unique scores for isotonic."
            )
        estimator = IsotonicRegression(
            y_min=0.0,
            y_max=1.0,
            increasing=True,
            out_of_bounds="clip",
        ).fit(raw, y)
        return IsotonicCalibrator(estimator)

    if method == "raw_platt":
        estimator = LogisticRegression(
            C=1e3,
            solver="lbfgs",
            max_iter=5000,
        ).fit(transform_raw(raw), y)
        return SklearnCalibrator(estimator, transform_raw)

    if method == "raw_spline":
        estimator = Pipeline(
            [
                (
                    "spline",
                    SplineTransformer(
                        n_knots=4,
                        degree=2,
                        include_bias=False,
                        extrapolation="linear",
                    ),
                ),
                (
                    "logistic",
                    LogisticRegression(
                        C=10.0,
                        solver="lbfgs",
                        max_iter=5000,
                    ),
                ),
            ]
        ).fit(transform_raw(raw), y)
        return SklearnCalibrator(estimator, transform_raw)

    if method == "raw_isotonic":
        if len(raw) < 100 or np.unique(raw).size < 10:
            raise ValueError(
                "Insufficient rows or unique margins for isotonic."
            )
        estimator = IsotonicRegression(
            y_min=0.0,
            y_max=1.0,
            increasing=True,
            out_of_bounds="clip",
        ).fit(raw, y)
        return IsotonicCalibrator(estimator)

    raise KeyError(f"Unknown calibration method: {method}")


CALIBRATOR_COMPLEXITY = {
    "identity": 0,
    "temperature": 1,
    "platt": 2,
    "raw_platt": 2,
    "beta": 3,
    "spline": 4,
    "raw_spline": 4,
    "isotonic": 5,
    "raw_isotonic": 5,
}


def fallback_probability(
    raw: np.ndarray,
    *,
    raw_kind: str,
) -> np.ndarray:
    raw = np.asarray(raw, dtype=float)
    if raw_kind == "probability":
        return clip_probability(raw)
    # Ten points is a conservative fixed scale for a temporary
    # cross-fitting fallback; it is never the final selected mapping.
    return clip_probability(expit(raw / 10.0))


@dataclass
class CalibrationSelection:
    method: str
    fitted_calibrator: Any
    audit: pd.DataFrame
    cross_fitted_predictions: np.ndarray


def select_cross_fitted_calibrator(
    inner_oof: pd.DataFrame,
    *,
    raw_kind: str,
) -> CalibrationSelection:
    methods = (
        MODEL_CONFIG["calibration"]["probability_methods"]
        if raw_kind == "probability"
        else MODEL_CONFIG["calibration"]["margin_methods"]
    )
    seasons = sorted(map(int, inner_oof["Season"].unique()))
    y_all = inner_oof["Team1Win"].to_numpy(dtype=int)
    raw_all = inner_oof["RawPrediction"].to_numpy(dtype=float)

    method_predictions: dict[str, np.ndarray] = {}
    records: list[dict[str, Any]] = []

    for method in methods:
        cross_fitted = np.full(len(inner_oof), np.nan, dtype=float)
        method_failed = False
        failure_message = ""
        for season in seasons:
            validation_mask = (
                inner_oof["Season"].to_numpy(dtype=int) == season
            )
            training_mask = ~validation_mask
            raw_train = raw_all[training_mask]
            y_train = y_all[training_mask]
            raw_validation = raw_all[validation_mask]
            if (
                training_mask.sum() < 40
                or np.unique(y_train).size < 2
            ):
                cross_fitted[validation_mask] = fallback_probability(
                    raw_validation,
                    raw_kind=raw_kind,
                )
                continue
            try:
                calibrator = fit_calibrator(
                    method,
                    raw_train,
                    y_train,
                    raw_kind=raw_kind,
                )
                cross_fitted[validation_mask] = calibrator.predict(
                    raw_validation
                )
            except Exception as exc:
                method_failed = True
                failure_message = repr(exc)
                cross_fitted[validation_mask] = fallback_probability(
                    raw_validation,
                    raw_kind=raw_kind,
                )

        scored = inner_oof[
            ["TargetKey", "Gender", "Season", "Team1Win"]
        ].copy()
        scored["Prediction"] = cross_fitted
        metrics, season_scores = evaluate_probability_frame(scored)
        records.append(
            {
                "Method": method,
                "RawKind": raw_kind,
                "MacroSeasonBrier": metrics["MacroSeasonBrier"],
                "GameWeightedBrier": metrics["GameWeightedBrier"],
                "SeasonBrierStd": metrics["SeasonBrierStd"],
                "WorstSeasonBrier": metrics["WorstSeasonBrier"],
                "ComplexityRank": CALIBRATOR_COMPLEXITY[method],
                "HadFallback": method_failed,
                "FailureMessage": failure_message,
            }
        )
        method_predictions[method] = cross_fitted

    audit = pd.DataFrame(records).sort_values(
        ["MacroSeasonBrier", "ComplexityRank"]
    ).reset_index(drop=True)
    best_method = str(audit.iloc[0]["Method"])
    best_predictions = method_predictions[best_method]

    best_season_scores = season_metric_values(
        pd.DataFrame(
            {
                "Season": inner_oof["Season"],
                "Team1Win": inner_oof["Team1Win"],
                "Prediction": best_predictions,
            }
        ),
        "Prediction",
    )
    standard_error = (
        float(
            best_season_scores["Brier"].std(ddof=1)
            / math.sqrt(len(best_season_scores))
        )
        if len(best_season_scores) > 1
        else 0.0
    )
    threshold = float(audit.iloc[0]["MacroSeasonBrier"]) + standard_error
    eligible = audit.loc[
        audit["MacroSeasonBrier"] <= threshold + 1e-12
    ].sort_values(["ComplexityRank", "MacroSeasonBrier"])
    selected_method = str(eligible.iloc[0]["Method"])

    audit["BestMeanMethod"] = audit["Method"].eq(best_method)
    audit["OneSEThreshold"] = threshold
    audit["WithinOneSE"] = (
        audit["MacroSeasonBrier"] <= threshold + 1e-12
    )
    audit["Selected"] = audit["Method"].eq(selected_method)

    fitted = fit_calibrator(
        selected_method,
        raw_all,
        y_all,
        raw_kind=raw_kind,
    )
    return CalibrationSelection(
        method=selected_method,
        fitted_calibrator=fitted,
        audit=audit,
        cross_fitted_predictions=method_predictions[selected_method],
    )


print("Calibration laboratory ready.")


## 7. Model engines

All requested frameworks use the same feature governance and outer seasons. XGBoost and LightGBM use histogram training and inner-fold early stopping. PyTorch and TensorFlow use matched compact multilayer-perceptron designs, fold-fitted standardization, deterministic seeds, and inner-fold early stopping.

In [ ]:
@dataclass(frozen=True)
class ModelSpec:
    name: str
    universe: str
    task: str
    raw_kind: str
    selector_family: str
    tune: bool
    complexity_rank: int
    description: str


MODEL_SPECS: dict[str, ModelSpec] = {
    "constant_probability": ModelSpec(
        "constant_probability",
        "compact",
        "classification",
        "probability",
        "baseline",
        False,
        0,
        "Constant 0.50 absolute baseline.",
    ),
    "seed_logistic": ModelSpec(
        "seed_logistic",
        "compact",
        "classification",
        "probability",
        "baseline",
        False,
        1,
        "Seed and prequential seed-prior logistic baseline.",
    ),
    "elo_seed_logistic": ModelSpec(
        "elo_seed_logistic",
        "compact",
        "classification",
        "probability",
        "baseline",
        False,
        2,
        "Corrected current-season Elo plus seed logistic baseline.",
    ),
    "elastic_logistic": ModelSpec(
        "elastic_logistic",
        "any",
        "classification",
        "probability",
        "linear",
        True,
        3,
        "Block-selected elastic-net logistic regression.",
    ),
    "hist_classifier": ModelSpec(
        "hist_classifier",
        "rich",
        "classification",
        "probability",
        "tree",
        False,
        4,
        "CPU-safe histogram gradient-boosting challenger.",
    ),
    "xgb_classifier": ModelSpec(
        "xgb_classifier",
        "rich",
        "classification",
        "probability",
        "tree",
        True,
        5,
        "Shallow regularized XGBoost probability model.",
    ),
    "lgb_classifier": ModelSpec(
        "lgb_classifier",
        "rich",
        "classification",
        "probability",
        "tree",
        True,
        6,
        "Shallow deterministic LightGBM probability model.",
    ),
    "ridge_margin": ModelSpec(
        "ridge_margin",
        "rich",
        "margin",
        "margin",
        "margin",
        False,
        7,
        "Regularized linear point-margin model.",
    ),
    "xgb_margin": ModelSpec(
        "xgb_margin",
        "rich",
        "margin",
        "margin",
        "margin",
        True,
        8,
        "Shallow XGBoost point-margin model.",
    ),
    "lgb_margin": ModelSpec(
        "lgb_margin",
        "rich",
        "margin",
        "margin",
        "margin",
        True,
        9,
        "Shallow LightGBM point-margin model.",
    ),
    "torch_mlp": ModelSpec(
        "torch_mlp",
        "rich",
        "classification",
        "probability",
        "neural",
        True,
        10,
        "Compact regularized CPU PyTorch multilayer perceptron.",
    ),
    "tensorflow_mlp": ModelSpec(
        "tensorflow_mlp",
        "rich",
        "classification",
        "probability",
        "neural",
        True,
        11,
        "Matched compact regularized CPU TensorFlow/Keras multilayer perceptron.",
    ),
}


def model_plan_for_outer(outer: pd.Series) -> list[str]:
    # The matched comparison uses the rich universe for every architecture.
    plan = [
        "constant_probability",
        "seed_logistic",
        "elo_seed_logistic",
        "elastic_logistic",
    ]
    if MODE["run_hist_classifier"]:
        plan.append("hist_classifier")
    plan.extend(["xgb_classifier", "lgb_classifier"])
    if MODE["run_neural"]:
        plan.extend(["torch_mlp", "tensorflow_mlp"])
    if MODE["run_margin_models"]:
        plan.extend(["ridge_margin", "xgb_margin", "lgb_margin"])
    return plan


def default_parameters(spec: ModelSpec) -> dict[str, Any]:
    defaults = {
        "constant_probability": {},
        "seed_logistic": {"C": 0.50},
        "elo_seed_logistic": {"C": 0.50},
        "elastic_logistic": {"C": 0.10, "l1_ratio": 0.25},
        "hist_classifier": {
            "learning_rate": 0.04,
            "max_leaf_nodes": 15,
            "max_depth": 4,
            "min_samples_leaf": 25,
            "l2_regularization": 5.0,
            "max_iter": 300,
        },
        "xgb_classifier": {
            "eta": 0.04,
            "max_depth": 3,
            "min_child_weight": 8.0,
            "subsample": 0.85,
            "colsample_bytree": 0.70,
            "reg_alpha": 0.10,
            "reg_lambda": 12.0,
            "gamma": 0.05,
            "max_bin": 128,
        },
        "lgb_classifier": {
            "learning_rate": 0.03,
            "num_leaves": 15,
            "max_depth": 4,
            "min_data_in_leaf": 35,
            "feature_fraction": 0.70,
            "bagging_fraction": 0.85,
            "bagging_freq": 1,
            "lambda_l1": 0.10,
            "lambda_l2": 12.0,
            "min_gain_to_split": 0.02,
            "max_bin": 127,
        },
        "ridge_margin": {"alpha": 100.0},
        "xgb_margin": {
            "eta": 0.035,
            "max_depth": 3,
            "min_child_weight": 8.0,
            "subsample": 0.85,
            "colsample_bytree": 0.70,
            "reg_alpha": 0.05,
            "reg_lambda": 15.0,
            "gamma": 0.0,
            "max_bin": 128,
        },
        "lgb_margin": {
            "learning_rate": 0.03,
            "num_leaves": 15,
            "max_depth": 4,
            "min_data_in_leaf": 35,
            "feature_fraction": 0.70,
            "bagging_fraction": 0.85,
            "bagging_freq": 1,
            "lambda_l1": 0.05,
            "lambda_l2": 15.0,
            "min_gain_to_split": 0.0,
            "max_bin": 127,
        },
        "torch_mlp": {
            "hidden_width": 64,
            "dropout": 0.20,
            "learning_rate": 8e-4,
            "weight_decay": 1e-3,
            "batch_size": 64,
        },
        "tensorflow_mlp": {
            "hidden_width": 64,
            "dropout": 0.20,
            "learning_rate": 8e-4,
            "weight_decay": 1e-3,
            "batch_size": 64,
        },
    }
    return dict(defaults[spec.name])


def sample_parameters(
    trial: optuna.Trial,
    spec: ModelSpec,
) -> dict[str, Any]:
    if spec.name == "elastic_logistic":
        return {
            "C": trial.suggest_float("C", 0.01, 2.0, log=True),
            "l1_ratio": trial.suggest_float("l1_ratio", 0.0, 0.85),
        }
    if spec.name in {"xgb_classifier", "xgb_margin"}:
        return {
            "eta": trial.suggest_float("eta", 0.015, 0.08, log=True),
            "max_depth": trial.suggest_int("max_depth", 2, 4),
            "min_child_weight": trial.suggest_float(
                "min_child_weight", 2.0, 20.0, log=True
            ),
            "subsample": trial.suggest_float("subsample", 0.70, 1.0),
            "colsample_bytree": trial.suggest_float(
                "colsample_bytree", 0.45, 0.90
            ),
            "reg_alpha": trial.suggest_float(
                "reg_alpha", 1e-4, 2.0, log=True
            ),
            "reg_lambda": trial.suggest_float(
                "reg_lambda", 3.0, 50.0, log=True
            ),
            "gamma": trial.suggest_float("gamma", 0.0, 0.50),
            "max_bin": trial.suggest_categorical(
                "max_bin", [64, 128, 256]
            ),
        }
    if spec.name in {"lgb_classifier", "lgb_margin"}:
        return {
            "learning_rate": trial.suggest_float(
                "learning_rate", 0.015, 0.07, log=True
            ),
            "num_leaves": trial.suggest_categorical(
                "num_leaves", [7, 11, 15, 23]
            ),
            "max_depth": trial.suggest_int("max_depth", 3, 6),
            "min_data_in_leaf": trial.suggest_int(
                "min_data_in_leaf", 20, 90
            ),
            "feature_fraction": trial.suggest_float(
                "feature_fraction", 0.50, 0.90
            ),
            "bagging_fraction": trial.suggest_float(
                "bagging_fraction", 0.70, 1.0
            ),
            "bagging_freq": 1,
            "lambda_l1": trial.suggest_float(
                "lambda_l1", 1e-4, 2.0, log=True
            ),
            "lambda_l2": trial.suggest_float(
                "lambda_l2", 3.0, 50.0, log=True
            ),
            "min_gain_to_split": trial.suggest_float(
                "min_gain_to_split", 0.0, 0.20
            ),
            "max_bin": trial.suggest_categorical(
                "max_bin", [63, 127, 255]
            ),
        }
    if spec.name in {"torch_mlp", "tensorflow_mlp"}:
        return {
            "hidden_width": trial.suggest_categorical(
                "hidden_width", [32, 64, 96]
            ),
            "dropout": trial.suggest_float("dropout", 0.05, 0.35),
            "learning_rate": trial.suggest_float(
                "learning_rate", 2e-4, 3e-3, log=True
            ),
            "weight_decay": trial.suggest_float(
                "weight_decay", 1e-5, 1e-2, log=True
            ),
            "batch_size": trial.suggest_categorical(
                "batch_size", [32, 64, 128]
            ),
        }
    return default_parameters(spec)


def prepare_numeric_matrix(
    frame: pd.DataFrame,
    features: Sequence[str],
) -> np.ndarray:
    # Always materialize a writable C-contiguous float32 array. pandas 3 uses
    # copy-on-write semantics and may otherwise return a read-only view.
    return np.array(
        frame[list(features)].to_numpy(
            dtype=np.float32,
            na_value=np.nan,
        ),
        dtype=np.float32,
        copy=True,
        order="C",
    )



DIRECTIONAL_INTERACTION_TOKENS = (
    "_edge",
    "squared_signed",
)


def mirror_feature_frame(
    frame: pd.DataFrame,
    features: Sequence[str],
    *,
    flip_targets: bool,
) -> pd.DataFrame:
    """Create the algebraic Team2-vs-Team1 representation.

    Notebook 02 already audited which feature families are directional
    versus symmetric. This transformation preserves that convention
    without materializing a second permanent feature store.
    """
    mirrored = frame.copy()
    feature_set = set(features)

    # Swap explicit team seed columns before other transformations.
    if {
        "matchup__team1_seed",
        "matchup__team2_seed",
    }.issubset(feature_set):
        mirrored[
            ["matchup__team1_seed", "matchup__team2_seed"]
        ] = frame[
            ["matchup__team2_seed", "matchup__team1_seed"]
        ].to_numpy()

    equal_seed = (
        frame["matchup__equal_seed"].fillna(0).astype(bool)
        if "matchup__equal_seed" in frame.columns
        else pd.Series(False, index=frame.index)
    )

    for feature in features:
        if feature not in frame.columns:
            continue
        if feature in {
            "matchup__team1_seed",
            "matchup__team2_seed",
        }:
            continue
        if feature.startswith("diff__"):
            mirrored[feature] = -pd.to_numeric(
                frame[feature], errors="coerce"
            )
        elif feature == "matchup__seed_diff":
            mirrored[feature] = -pd.to_numeric(
                frame[feature], errors="coerce"
            )
        elif (
            feature.startswith("seedprior__")
            and "p_team1" in feature
        ):
            mirrored[feature] = 1.0 - pd.to_numeric(
                frame[feature], errors="coerce"
            )
        elif feature == "matchup__team1_is_seed_favorite":
            original = pd.to_numeric(
                frame[feature], errors="coerce"
            )
            mirrored[feature] = np.where(
                equal_seed,
                original,
                1.0 - original,
            )
        elif feature.startswith("interaction__") and any(
            token in feature
            for token in DIRECTIONAL_INTERACTION_TOKENS
        ):
            mirrored[feature] = -pd.to_numeric(
                frame[feature], errors="coerce"
            )
        # absdiff__, mean__, availability, context, gaps, volatility,
        # and other explicitly symmetric fields remain unchanged.

    if flip_targets:
        if "Team1Win" in mirrored.columns:
            mirrored["Team1Win"] = (
                1 - mirrored["Team1Win"].astype(int)
            )
        if "Team1Margin" in mirrored.columns:
            mirrored["Team1Margin"] = -pd.to_numeric(
                mirrored["Team1Margin"], errors="coerce"
            )
        if {
            "Team1ID",
            "Team2ID",
        }.issubset(mirrored.columns):
            mirrored[["Team1ID", "Team2ID"]] = frame[
                ["Team2ID", "Team1ID"]
            ].to_numpy()
        if "TargetKey" in mirrored.columns:
            mirrored["TargetKey"] = (
                mirrored["TargetKey"].astype(str) + "__mirror"
            )
    return mirrored


def augment_training_symmetrically(
    training: pd.DataFrame,
    features: Sequence[str],
) -> pd.DataFrame:
    mirrored = mirror_feature_frame(
        training,
        features,
        flip_targets=True,
    )
    augmented = pd.concat(
        [training, mirrored],
        ignore_index=True,
    )
    return augmented


@dataclass
class FittedPredictor:
    model_name: str
    model: Any
    preprocessor: Any
    features: list[str]
    best_iteration: int | None
    raw_kind: str
    enforce_symmetry: bool = True

    def _predict_once(self, frame: pd.DataFrame) -> np.ndarray:
        if self.model_name == "constant_probability":
            return self.model.predict(np.empty(len(frame)))

        matrix = prepare_numeric_matrix(frame, self.features)
        if self.model_name in {
            "seed_logistic",
            "elo_seed_logistic",
            "elastic_logistic",
            "ridge_margin",
        }:
            transformed = self.preprocessor.transform(matrix)
            if self.raw_kind == "probability":
                return clip_probability(
                    self.model.predict_proba(transformed)[:, 1]
                )
            return np.asarray(
                self.model.predict(transformed), dtype=float
            )
        if self.model_name == "hist_classifier":
            return clip_probability(
                self.model.predict_proba(matrix)[:, 1]
            )
        if self.model_name.startswith("xgb_"):
            data = xgb.DMatrix(
                matrix,
                feature_names=self.features,
            )
            iteration = (
                self.best_iteration
                if self.best_iteration is not None
                else int(self.model.num_boosted_rounds())
            )
            return np.asarray(
                self.model.predict(
                    data,
                    iteration_range=(0, max(1, iteration)),
                ),
                dtype=float,
            )
        if self.model_name.startswith("lgb_"):
            return np.asarray(
                self.model.predict(
                    matrix,
                    num_iteration=self.best_iteration,
                ),
                dtype=float,
            )
        if self.model_name == "torch_mlp":
            transformed = self.preprocessor.transform(matrix).astype(
                np.float32
            )
            tensor = torch.from_numpy(transformed)
            self.model.eval()
            with torch.no_grad():
                logits = self.model(tensor).squeeze(1).cpu().numpy()
            return clip_probability(expit(logits))
        if self.model_name == "tensorflow_mlp":
            transformed = self.preprocessor.transform(matrix).astype(
                np.float32
            )
            tensor = tf.convert_to_tensor(
                transformed,
                dtype=tf.float32,
            )
            probability = tf.reshape(
                self.model(tensor, training=False),
                (-1,),
            ).numpy()
            return clip_probability(probability)
        raise KeyError(self.model_name)

    def predict_raw(self, frame: pd.DataFrame) -> np.ndarray:
        original = self._predict_once(frame)
        if (
            not self.enforce_symmetry
            or self.model_name == "constant_probability"
        ):
            return (
                clip_probability(original)
                if self.raw_kind == "probability"
                else np.asarray(original, dtype=float)
            )
        mirrored_frame = mirror_feature_frame(
            frame,
            self.features,
            flip_targets=False,
        )
        mirrored = self._predict_once(mirrored_frame)
        if self.raw_kind == "probability":
            return clip_probability(
                0.5 * (original + (1.0 - mirrored))
            )
        return 0.5 * (
            np.asarray(original, dtype=float)
            - np.asarray(mirrored, dtype=float)
        )


def xgb_brier_metric(
    prediction: np.ndarray,
    data: xgb.DMatrix,
) -> tuple[str, float]:
    y = data.get_label()
    return "brier", float(np.mean((prediction - y) ** 2))


def lgb_brier_metric(
    prediction: np.ndarray,
    data: lgb.Dataset,
) -> tuple[str, float, bool]:
    y = data.get_label()
    return "brier", float(np.mean((prediction - y) ** 2)), False


def fit_predictor(
    spec: ModelSpec,
    params: dict[str, Any],
    training: pd.DataFrame,
    validation: pd.DataFrame | None,
    features: Sequence[str],
    *,
    fixed_iterations: int | None = None,
    random_seed: int = SEED,
) -> tuple[FittedPredictor, np.ndarray | None]:
    features = list(features)
    if spec.name != "constant_probability":
        training = augment_training_symmetrically(
            training,
            features,
        )
    y_win = training["Team1Win"].to_numpy(dtype=int)
    y_margin = training["Team1Margin"].to_numpy(dtype=np.float32)
    x_train = prepare_numeric_matrix(training, features) if features else None
    x_validation = (
        prepare_numeric_matrix(validation, features)
        if validation is not None and features
        else None
    )

    if spec.name == "constant_probability":
        probability = 0.5
        fitted = FittedPredictor(
            model_name=spec.name,
            model=ConstantCalibrator(probability),
            preprocessor=None,
            features=[],
            best_iteration=None,
            raw_kind="probability",
        )
        raw = (
            np.full(len(validation), probability)
            if validation is not None
            else None
        )
        return fitted, raw

    if spec.name in {
        "seed_logistic",
        "elo_seed_logistic",
        "elastic_logistic",
    }:
        imputer = SimpleImputer(strategy="median")
        scaler = StandardScaler()
        transformed = scaler.fit_transform(
            imputer.fit_transform(x_train)
        )
        if spec.name == "elastic_logistic":
            model = LogisticRegression(
                C=float(params["C"]),
                l1_ratio=float(params["l1_ratio"]),
                solver="saga",
                max_iter=8000,
                tol=1e-4,
                random_state=random_seed,
            )
        else:
            model = LogisticRegression(
                C=float(params["C"]),
                l1_ratio=0.0,
                solver="lbfgs",
                max_iter=5000,
                random_state=random_seed,
            )
        model.fit(transformed, y_win)
        preprocessor = Pipeline(
            [("imputer", imputer), ("scaler", scaler)]
        )
        fitted = FittedPredictor(
            spec.name,
            model,
            preprocessor,
            features,
            None,
            "probability",
        )
        raw = (
            fitted.predict_raw(validation)
            if validation is not None
            else None
        )
        return fitted, raw

    if spec.name == "ridge_margin":
        imputer = SimpleImputer(strategy="median")
        scaler = StandardScaler()
        transformed = scaler.fit_transform(
            imputer.fit_transform(x_train)
        )
        model = Ridge(alpha=float(params["alpha"]))
        model.fit(transformed, y_margin)
        preprocessor = Pipeline(
            [("imputer", imputer), ("scaler", scaler)]
        )
        fitted = FittedPredictor(
            spec.name,
            model,
            preprocessor,
            features,
            None,
            "margin",
        )
        raw = (
            fitted.predict_raw(validation)
            if validation is not None
            else None
        )
        return fitted, raw

    if spec.name == "hist_classifier":
        model = HistGradientBoostingClassifier(
            **params,
            early_stopping=False,
            random_state=random_seed,
        )
        model.fit(x_train, y_win)
        fitted = FittedPredictor(
            spec.name,
            model,
            None,
            features,
            int(params["max_iter"]),
            "probability",
        )
        raw = (
            fitted.predict_raw(validation)
            if validation is not None
            else None
        )
        return fitted, raw

    if spec.name.startswith("xgb_"):
        is_classification = spec.task == "classification"
        objective = (
            "binary:logistic"
            if is_classification
            else "reg:squarederror"
        )
        xgb_params = {
            "objective": objective,
            "tree_method": "hist",
            "device": "cpu",
            "nthread": MAX_THREADS,
            "seed": random_seed,
            "verbosity": 0,
            **params,
        }
        if is_classification:
            xgb_params["disable_default_eval_metric"] = 1
        else:
            xgb_params["eval_metric"] = "rmse"

        train_matrix = xgb.QuantileDMatrix(
            x_train,
            label=y_win if is_classification else y_margin,
            feature_names=features,
            max_bin=int(params["max_bin"]),
        )
        evaluations = []
        valid_matrix = None
        callbacks = []
        train_kwargs: dict[str, Any] = {}
        num_rounds = int(fixed_iterations or 1200)

        if validation is not None and fixed_iterations is None:
            valid_label = (
                validation["Team1Win"].to_numpy(dtype=int)
                if is_classification
                else validation["Team1Margin"].to_numpy(
                    dtype=np.float32
                )
            )
            valid_matrix = xgb.QuantileDMatrix(
                x_validation,
                label=valid_label,
                feature_names=features,
                ref=train_matrix,
                max_bin=int(params["max_bin"]),
            )
            evaluations = [(valid_matrix, "valid")]
            train_kwargs["early_stopping_rounds"] = 60
        if is_classification:
            train_kwargs["custom_metric"] = xgb_brier_metric

        booster = xgb.train(
            xgb_params,
            train_matrix,
            num_boost_round=num_rounds,
            evals=evaluations,
            verbose_eval=False,
            **train_kwargs,
        )
        if (
            validation is not None
            and fixed_iterations is None
            and getattr(booster, "best_iteration", None) is not None
        ):
            best_iteration = int(booster.best_iteration) + 1
        else:
            best_iteration = int(num_rounds)
        booster.set_attr(
            max_bin_for_prediction=str(int(params["max_bin"]))
        )
        fitted = FittedPredictor(
            spec.name,
            booster,
            None,
            features,
            best_iteration,
            spec.raw_kind,
        )
        raw = (
            fitted.predict_raw(validation)
            if validation is not None
            else None
        )
        return fitted, raw

    if spec.name.startswith("lgb_"):
        is_classification = spec.task == "classification"
        lgb_params = {
            "objective": "binary" if is_classification else "regression",
            "metric": "None" if is_classification else "rmse",
            "verbosity": -1,
            "num_threads": MAX_THREADS,
            "seed": random_seed,
            "feature_fraction_seed": random_seed,
            "bagging_seed": random_seed,
            "data_random_seed": random_seed,
            "deterministic": True,
            "force_col_wise": True,
            **params,
        }
        train_set = lgb.Dataset(
            x_train,
            label=y_win if is_classification else y_margin,
            feature_name=features,
            free_raw_data=False,
        )
        callbacks = [lgb.log_evaluation(period=0)]
        valid_sets = None
        valid_names = None
        num_rounds = int(fixed_iterations or 1500)
        feval = lgb_brier_metric if is_classification else None

        if validation is not None and fixed_iterations is None:
            valid_label = (
                validation["Team1Win"].to_numpy(dtype=int)
                if is_classification
                else validation["Team1Margin"].to_numpy(
                    dtype=np.float32
                )
            )
            valid_set = lgb.Dataset(
                x_validation,
                label=valid_label,
                feature_name=features,
                reference=train_set,
                free_raw_data=False,
            )
            valid_sets = [valid_set]
            valid_names = ["valid"]
            callbacks.append(
                lgb.early_stopping(
                    stopping_rounds=60,
                    first_metric_only=True,
                    verbose=False,
                )
            )

        booster = lgb.train(
            lgb_params,
            train_set,
            num_boost_round=num_rounds,
            valid_sets=valid_sets,
            valid_names=valid_names,
            feval=feval,
            callbacks=callbacks,
        )
        best_iteration = int(
            booster.best_iteration
            if booster.best_iteration
            else num_rounds
        )
        fitted = FittedPredictor(
            spec.name,
            booster,
            None,
            features,
            best_iteration,
            spec.raw_kind,
        )
        raw = (
            fitted.predict_raw(validation)
            if validation is not None
            else None
        )
        return fitted, raw

    if spec.name == "torch_mlp":
        if torch is None:
            raise ImportError("PyTorch is unavailable.")
        torch.manual_seed(random_seed)
        torch.set_num_threads(MAX_THREADS)
        imputer = SimpleImputer(strategy="median")
        scaler = StandardScaler()
        x_train_scaled = scaler.fit_transform(
            imputer.fit_transform(x_train)
        ).astype(np.float32)
        preprocessor = Pipeline(
            [("imputer", imputer), ("scaler", scaler)]
        )
        x_valid_scaled = (
            preprocessor.transform(x_validation).astype(np.float32)
            if validation is not None
            else None
        )

        hidden = int(params["hidden_width"])
        model = nn.Sequential(
            nn.Linear(x_train_scaled.shape[1], hidden),
            nn.ReLU(),
            nn.LayerNorm(hidden),
            nn.Dropout(float(params["dropout"])),
            nn.Linear(hidden, max(16, hidden // 2)),
            nn.ReLU(),
            nn.Dropout(float(params["dropout"])),
            nn.Linear(max(16, hidden // 2), 1),
        )
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=float(params["learning_rate"]),
            weight_decay=float(params["weight_decay"]),
        )
        loss_function = nn.BCEWithLogitsLoss()
        dataset = TensorDataset(
            torch.from_numpy(x_train_scaled),
            torch.from_numpy(y_win.astype(np.float32)).reshape(-1, 1),
        )
        generator = torch.Generator().manual_seed(random_seed)
        loader = DataLoader(
            dataset,
            batch_size=int(params["batch_size"]),
            shuffle=True,
            generator=generator,
            num_workers=0,
        )
        maximum_epochs = int(fixed_iterations or 220)
        best_state = None
        best_loss = float("inf")
        best_epoch = maximum_epochs
        stale = 0

        for epoch in range(1, maximum_epochs + 1):
            model.train()
            for batch_x, batch_y in loader:
                optimizer.zero_grad(set_to_none=True)
                logits = model(batch_x)
                loss = loss_function(logits, batch_y)
                loss.backward()
                optimizer.step()

            if (
                validation is not None
                and fixed_iterations is None
                and x_valid_scaled is not None
            ):
                model.eval()
                with torch.no_grad():
                    valid_logits = model(
                        torch.from_numpy(x_valid_scaled)
                    ).squeeze(1)
                    valid_probability = torch.sigmoid(
                        valid_logits
                    ).cpu().numpy()
                valid_y = validation["Team1Win"].to_numpy(dtype=float)
                valid_brier = float(
                    np.mean((valid_probability - valid_y) ** 2)
                )
                if valid_brier < best_loss - 1e-6:
                    best_loss = valid_brier
                    best_epoch = epoch
                    best_state = {
                        key: value.detach().cpu().clone()
                        for key, value in model.state_dict().items()
                    }
                    stale = 0
                else:
                    stale += 1
                if stale >= 25:
                    break

        if best_state is not None:
            model.load_state_dict(best_state)
        fitted = FittedPredictor(
            spec.name,
            model,
            preprocessor,
            features,
            int(best_epoch),
            "probability",
        )
        raw = (
            fitted.predict_raw(validation)
            if validation is not None
            else None
        )
        return fitted, raw


    if spec.name == "tensorflow_mlp":
        if tf is None:
            raise ImportError("TensorFlow is unavailable.")
        tf.keras.utils.clear_session(free_memory=True)
        tf.keras.utils.set_random_seed(random_seed)

        imputer = SimpleImputer(strategy="median")
        scaler = StandardScaler()
        x_train_scaled = scaler.fit_transform(
            imputer.fit_transform(x_train)
        ).astype(np.float32)
        preprocessor = Pipeline(
            [("imputer", imputer), ("scaler", scaler)]
        )
        x_valid_scaled = (
            preprocessor.transform(x_validation).astype(np.float32)
            if validation is not None
            else None
        )

        hidden = int(params["hidden_width"])
        second_hidden = max(16, hidden // 2)
        regularizer = tf.keras.regularizers.l2(
            float(params["weight_decay"])
        )
        inputs = tf.keras.Input(
            shape=(x_train_scaled.shape[1],),
            dtype=tf.float32,
        )
        x = tf.keras.layers.Dense(
            hidden,
            activation="relu",
            kernel_regularizer=regularizer,
            kernel_initializer=tf.keras.initializers.GlorotUniform(
                seed=random_seed
            ),
        )(inputs)
        x = tf.keras.layers.LayerNormalization()(x)
        x = tf.keras.layers.Dropout(
            float(params["dropout"]),
            seed=random_seed + 1,
        )(x)
        x = tf.keras.layers.Dense(
            second_hidden,
            activation="relu",
            kernel_regularizer=regularizer,
            kernel_initializer=tf.keras.initializers.GlorotUniform(
                seed=random_seed + 2
            ),
        )(x)
        x = tf.keras.layers.Dropout(
            float(params["dropout"]),
            seed=random_seed + 3,
        )(x)
        outputs = tf.keras.layers.Dense(
            1,
            activation="sigmoid",
            kernel_initializer=tf.keras.initializers.GlorotUniform(
                seed=random_seed + 4
            ),
        )(x)
        model = tf.keras.Model(inputs=inputs, outputs=outputs)
        model.compile(
            optimizer=tf.keras.optimizers.Adam(
                learning_rate=float(params["learning_rate"])
            ),
            loss=tf.keras.losses.BinaryCrossentropy(),
            metrics=[
                tf.keras.metrics.MeanSquaredError(name="brier"),
            ],
        )

        maximum_epochs = int(fixed_iterations or 220)
        callbacks: list[Any] = [
            tf.keras.callbacks.TerminateOnNaN(),
        ]
        validation_data = None
        if (
            validation is not None
            and fixed_iterations is None
            and x_valid_scaled is not None
        ):
            validation_data = (
                x_valid_scaled,
                validation["Team1Win"].to_numpy(dtype=np.float32).reshape(-1, 1),
            )
            callbacks.append(
                tf.keras.callbacks.EarlyStopping(
                    monitor="val_brier",
                    mode="min",
                    patience=25,
                    min_delta=1e-6,
                    restore_best_weights=True,
                    verbose=0,
                )
            )

        history = model.fit(
            x_train_scaled,
            y_win.astype(np.float32).reshape(-1, 1),
            validation_data=validation_data,
            epochs=maximum_epochs,
            batch_size=int(params["batch_size"]),
            shuffle=True,
            verbose=0,
            callbacks=callbacks,
        )
        if validation_data is not None and "val_brier" in history.history:
            best_epoch = int(
                np.argmin(
                    np.asarray(history.history["val_brier"], dtype=float)
                )
                + 1
            )
        else:
            best_epoch = int(len(history.history.get("loss", [])))
            best_epoch = max(1, best_epoch)

        fitted = FittedPredictor(
            spec.name,
            model,
            preprocessor,
            features,
            best_epoch,
            "probability",
        )
        raw = (
            fitted.predict_raw(validation)
            if validation is not None
            else None
        )
        if raw is not None and not np.isfinite(raw).all():
            raise FloatingPointError(
                "TensorFlow produced non-finite probabilities."
            )
        return fitted, raw

    raise KeyError(f"Unsupported model: {spec.name}")


print("Model engines ready:", sorted(MODEL_SPECS))


## 8. Nested tuning

Hyperparameters and training length are selected from earlier inner validation seasons. Search budgets are deliberately bounded because the historical target set is small; a larger search would increase the risk of tuning to noise.

In [ ]:
def rows_for_context(
    frame: pd.DataFrame,
    *,
    seasons: Sequence[int],
    gender: str,
    universe: str,
) -> pd.DataFrame:
    mask = context_mask(
        frame,
        seasons=seasons,
        gender=gender,
        universe=universe,
    )
    result = frame.loc[mask].copy()
    assert not result.empty
    assert set(map(int, result["Season"].unique())).issubset(
        set(map(int, seasons))
    )
    if gender != "Pooled":
        assert result["Gender"].eq(gender).all()
    return result


def candidates_for_context(
    *,
    gender: str,
    universe: str,
    seed_aware: bool,
) -> list[str]:
    key = candidate_set_name(
        gender=gender,
        universe=universe,
        seed_aware=seed_aware,
    )
    assert key in CANDIDATE_SETS, f"Missing candidate set: {key}"
    candidates = [
        column
        for column in CANDIDATE_SETS[key]
        if column in development.columns
        and pd.api.types.is_numeric_dtype(development[column])
    ]
    assert candidates, f"No available columns for {key}"
    return candidates


def model_feature_list(
    spec: ModelSpec,
    training: pd.DataFrame,
    candidates: Sequence[str],
    *,
    pooled: bool,
    selection_key: str,
) -> tuple[list[str], SelectionResult | None]:
    if spec.selector_family == "baseline":
        selected = baseline_features(
            spec.name,
            candidates,
            pooled=pooled,
        )
        if spec.name != "constant_probability":
            assert selected, f"No baseline features for {spec.name}"
        return selected, None

    target_column = (
        "Team1Win"
        if spec.task == "classification"
        else "Team1Margin"
    )
    result = get_selection(
        training,
        candidates,
        target_column=target_column,
        family=spec.selector_family,
        pooled=pooled,
        selection_key=selection_key,
    )
    assert len(result.selected_features) <= result.effective_cap
    return result.selected_features, result


def make_prediction_frame(
    validation: pd.DataFrame,
    *,
    outer_fold_id: str,
    inner_fold_id: str | None,
    architecture: str,
    universe: str,
    model_name: str,
    raw_prediction: np.ndarray,
    feature_count: int,
    best_iteration: int | None,
) -> pd.DataFrame:
    columns = [
        "TargetKey",
        "Gender",
        "Season",
        "Team1ID",
        "Team2ID",
        "Team1Win",
        "Team1Margin",
    ]
    output = validation[columns].copy()
    output["OuterFoldID"] = outer_fold_id
    output["InnerFoldID"] = inner_fold_id
    output["Architecture"] = architecture
    output["Universe"] = universe
    output["Model"] = model_name
    output["RawPrediction"] = np.asarray(
        raw_prediction, dtype=np.float64
    )
    output["FeatureCount"] = int(feature_count)
    output["BestIteration"] = (
        np.nan if best_iteration is None else int(best_iteration)
    )
    return output


def inner_rows_for_outer(outer_fold_id: str) -> pd.DataFrame:
    rows = inner_folds.loc[
        inner_folds["OuterFoldID"].eq(outer_fold_id)
    ].sort_values("InnerValidationSeason")
    cap = int(MODE["inner_fold_cap"])
    if len(rows) > cap:
        rows = rows.tail(cap)
    assert not rows.empty, (
        f"No inner folds available for {outer_fold_id}"
    )
    return rows.reset_index(drop=True)


def run_inner_oof(
    *,
    outer: pd.Series,
    spec: ModelSpec,
    params: dict[str, Any],
    candidates: Sequence[str],
    collect_selector_audits: bool,
    trial: optuna.Trial | None = None,
) -> tuple[pd.DataFrame, list[int], list[pd.DataFrame]]:
    predictions: list[pd.DataFrame] = []
    best_iterations: list[int] = []
    selector_audits: list[pd.DataFrame] = []
    inner_rows = inner_rows_for_outer(str(outer["OuterFoldID"]))
    gender = str(outer["Gender"])
    universe = str(outer["Universe"])
    pooled = gender == "Pooled"

    for step, (_, inner) in enumerate(
        inner_rows.iterrows(), start=1
    ):
        training_seasons = parse_json_int_list(
            inner["InnerTrainingSeasonsJSON"]
        )
        validation_season = int(inner["InnerValidationSeason"])
        assert max(training_seasons) < validation_season
        assert validation_season < int(outer["ValidationSeason"])

        training = rows_for_context(
            development,
            seasons=training_seasons,
            gender=gender,
            universe=universe,
        )
        validation = rows_for_context(
            development,
            seasons=[validation_season],
            gender=gender,
            universe=universe,
        )
        assert training["Season"].max() < validation_season
        assert validation["Season"].eq(validation_season).all()

        selection_key = (
            f"{inner['InnerFoldID']}__{spec.selector_family}"
            f"__{spec.task}"
        )
        features, selection = model_feature_list(
            spec,
            training,
            candidates,
            pooled=pooled,
            selection_key=selection_key,
        )
        if selection is not None and collect_selector_audits:
            audit = selection.audit.loc[
                selection.audit["Selected"]
            ].copy()
            audit["OuterFoldID"] = outer["OuterFoldID"]
            audit["InnerFoldID"] = inner["InnerFoldID"]
            audit["Model"] = spec.name
            audit["TargetColumn"] = (
                "Team1Win"
                if spec.task == "classification"
                else "Team1Margin"
            )
            selector_audits.append(audit)

        with threadpool_limits(limits=MAX_THREADS):
            fitted, raw = fit_predictor(
                spec,
                params,
                training,
                validation,
                features,
                fixed_iterations=None,
                random_seed=SEED + validation_season,
            )
        assert raw is not None
        predictions.append(
            make_prediction_frame(
                validation,
                outer_fold_id=str(outer["OuterFoldID"]),
                inner_fold_id=str(inner["InnerFoldID"]),
                architecture=str(outer["Architecture"]),
                universe=universe,
                model_name=spec.name,
                raw_prediction=raw,
                feature_count=len(features),
                best_iteration=fitted.best_iteration,
            )
        )
        if fitted.best_iteration is not None:
            best_iterations.append(int(fitted.best_iteration))

        if trial is not None:
            partial = pd.concat(predictions, ignore_index=True)
            score = (
                robust_probability_objective(
                    partial.assign(
                        Prediction=clip_probability(
                            partial["RawPrediction"]
                        )
                    )
                )
                if spec.task == "classification"
                else robust_margin_objective(partial)
            )
            trial.report(float(score), step=step)
            if trial.should_prune():
                raise optuna.TrialPruned()

        del training, validation, fitted, raw
        if spec.name == "tensorflow_mlp" and tf is not None:
            tf.keras.utils.clear_session(free_memory=True)
        gc.collect()

    inner_oof = pd.concat(predictions, ignore_index=True)
    assert inner_oof["TargetKey"].is_unique
    return inner_oof, best_iterations, selector_audits


def tune_model(
    *,
    outer: pd.Series,
    spec: ModelSpec,
    candidates: Sequence[str],
) -> tuple[dict[str, Any], list[int], dict[str, Any]]:
    if not spec.tune:
        return default_parameters(spec), [], {
            "SearchStrategy": "fixed_preregistered",
            "TrialsRequested": 0,
            "TrialsCompleted": 0,
        }

    requested = int(MODE["trial_budgets"].get(spec.name, 0))
    if requested <= 0:
        return default_parameters(spec), [], {
            "SearchStrategy": "fixed_due_to_zero_budget",
            "TrialsRequested": 0,
            "TrialsCompleted": 0,
        }

    study_name = sanitize_name(
        f"{MODEL_CONFIG['model_contract_sha256'][:10]}"
        f"__{outer['OuterFoldID']}__{spec.name}"
        f"__{RUN_MODE}"
    )
    database_path = STUDY_DIR / f"{study_name}.sqlite3"
    storage = f"sqlite:///{database_path.as_posix()}"
    sampler = optuna.samplers.TPESampler(
        seed=SEED,
        n_startup_trials=min(3, requested),
    )
    pruner = optuna.pruners.MedianPruner(
        n_startup_trials=min(2, requested),
        n_warmup_steps=1,
        interval_steps=1,
    )
    study = optuna.create_study(
        direction="minimize",
        study_name=study_name,
        storage=storage,
        load_if_exists=True,
        sampler=sampler,
        pruner=pruner,
    )

    def objective(trial: optuna.Trial) -> float:
        params = sample_parameters(trial, spec)
        try:
            inner_oof, iterations, _ = run_inner_oof(
                outer=outer,
                spec=spec,
                params=params,
                candidates=candidates,
                collect_selector_audits=False,
                trial=trial,
            )
            score = (
                robust_probability_objective(
                    inner_oof.assign(
                        Prediction=clip_probability(
                            inner_oof["RawPrediction"]
                        )
                    )
                )
                if spec.task == "classification"
                else robust_margin_objective(inner_oof)
            )
            trial.set_user_attr(
                "best_iterations",
                list(map(int, iterations)),
            )
            trial.set_user_attr(
                "inner_validation_seasons",
                sorted(
                    map(int, inner_oof["Season"].unique())
                ),
            )
            return float(score)
        except optuna.TrialPruned:
            raise
        except Exception as exc:
            trial.set_user_attr("failure", repr(exc))
            return float(1e6)
        finally:
            gc.collect()

    completed_or_running = len(study.trials)
    additional = max(0, requested - completed_or_running)
    if additional:
        study.optimize(
            objective,
            n_trials=additional,
            n_jobs=1,
            gc_after_trial=True,
            show_progress_bar=False,
        )

    valid_trials = [
        trial
        for trial in study.trials
        if trial.state == optuna.trial.TrialState.COMPLETE
        and np.isfinite(trial.value)
        and trial.value < 1e5
    ]
    if not valid_trials:
        warnings.warn(
            f"No valid Optuna trial for {spec.name} "
            f"{outer['OuterFoldID']}; using defaults."
        )
        return default_parameters(spec), [], {
            "SearchStrategy": "default_after_failed_search",
            "TrialsRequested": requested,
            "TrialsCompleted": 0,
        }

    best = min(valid_trials, key=lambda trial: float(trial.value))
    best_iterations = list(
        map(int, best.user_attrs.get("best_iterations", []))
    )
    metadata = {
        "SearchStrategy": "optuna_tpe_nested_inner_folds",
        "StudyName": study_name,
        "DatabasePath": str(database_path),
        "TrialsRequested": requested,
        "TrialsCompleted": len(valid_trials),
        "BestTrialNumber": int(best.number),
        "BestObjective": float(best.value),
    }
    return dict(best.params), best_iterations, metadata


print("Nested tuning utilities ready.")


## 9. Outer-fold execution and resume support

After all inner decisions are fixed, each model is retrained on the complete outer-training window and predicts one untouched tournament season.

In [ ]:
MODEL_FAILURES: list[dict[str, Any]] = []
MODEL_RUN_METADATA: list[dict[str, Any]] = []
SELECTED_FEATURE_RECORDS: list[pd.DataFrame] = []
CALIBRATION_RECORDS: list[pd.DataFrame] = []


def outer_model_directory(
    outer_fold_id: str,
    model_name: str,
) -> Path:
    return (
        CACHE_DIR
        / "outer_models"
        / sanitize_name(outer_fold_id)
        / sanitize_name(model_name)
    )


def load_outer_checkpoint(
    directory: Path,
) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, Any]]:
    metadata = read_json(directory / "metadata.json")
    outer_predictions = pd.read_parquet(
        directory / "outer_predictions.parquet"
    )
    inner_oof = pd.read_parquet(directory / "inner_oof.parquet")
    return outer_predictions, inner_oof, metadata


def run_outer_model(
    outer: pd.Series,
    model_name: str,
    *,
    seed_aware: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, Any]]:
    spec = MODEL_SPECS[model_name]
    outer_fold_id = str(outer["OuterFoldID"])
    directory = outer_model_directory(
        outer_fold_id,
        model_name
        + ("" if seed_aware else "__seed_free"),
    )
    if checkpoint_is_valid(directory):
        outer_predictions, inner_oof, metadata = load_outer_checkpoint(
            directory
        )
        metadata["LoadedFromCheckpoint"] = True
        return outer_predictions, inner_oof, metadata

    memory_before_training = ensure_memory_headroom(
        f"{outer_fold_id} :: {model_name}",
        model_name=spec.name,
    )

    gender = str(outer["Gender"])
    universe = str(outer["Universe"])
    pooled = gender == "Pooled"
    training_seasons = parse_json_int_list(
        outer["TrainingSeasonsJSON"]
    )
    validation_season = int(outer["ValidationSeason"])
    assert max(training_seasons) < validation_season
    assert validation_season <= DEVELOPMENT_LAST_SEASON

    outer_training = rows_for_context(
        development,
        seasons=training_seasons,
        gender=gender,
        universe=universe,
    )
    outer_validation = rows_for_context(
        development,
        seasons=[validation_season],
        gender=gender,
        universe=universe,
    )
    assert outer_training["Season"].max() < validation_season
    assert outer_validation["Season"].eq(validation_season).all()

    candidates = candidates_for_context(
        gender=gender,
        universe=universe,
        seed_aware=seed_aware,
    )
    if spec.universe == "rich":
        assert universe == "rich"

    start_rss = rss_mb()
    start_time = time.perf_counter()

    params, search_iterations, search_metadata = tune_model(
        outer=outer,
        spec=spec,
        candidates=candidates,
    )

    # Re-run the chosen parameterization to obtain one clean inner OOF
    # vector and selection audit. This is the input to calibration.
    inner_oof, inner_iterations, selector_audits = run_inner_oof(
        outer=outer,
        spec=spec,
        params=params,
        candidates=candidates,
        collect_selector_audits=True,
        trial=None,
    )

    if spec.name == "constant_probability":
        calibration = CalibrationSelection(
            method="identity",
            fitted_calibrator=IdentityCalibrator(),
            audit=pd.DataFrame(
                [
                    {
                        "Method": "identity",
                        "RawKind": "probability",
                        "MacroSeasonBrier": float(
                            np.mean(
                                (
                                    inner_oof["RawPrediction"].to_numpy(
                                        dtype=float
                                    )
                                    - inner_oof["Team1Win"].to_numpy(
                                        dtype=float
                                    )
                                )
                                ** 2
                            )
                        ),
                        "ComplexityRank": 0,
                        "Selected": True,
                    }
                ]
            ),
            cross_fitted_predictions=clip_probability(
                inner_oof["RawPrediction"]
            ),
        )
    else:
        calibration = select_cross_fitted_calibrator(
            inner_oof,
            raw_kind=spec.raw_kind,
        )

    inner_oof["Prediction"] = (
        calibration.cross_fitted_predictions
    )
    inner_oof["CalibrationMethod"] = calibration.method

    selection_key = (
        f"{outer_fold_id}__outer__{spec.selector_family}"
        f"__{spec.task}"
    )
    outer_features, outer_selection = model_feature_list(
        spec,
        outer_training,
        candidates,
        pooled=pooled,
        selection_key=selection_key,
    )
    if outer_selection is not None:
        selected_audit = outer_selection.audit.loc[
            outer_selection.audit["Selected"]
        ].copy()
        selected_audit["OuterFoldID"] = outer_fold_id
        selected_audit["InnerFoldID"] = None
        selected_audit["Model"] = spec.name
        selected_audit["TargetColumn"] = (
            "Team1Win"
            if spec.task == "classification"
            else "Team1Margin"
        )
        selector_audits.append(selected_audit)

    usable_iterations = inner_iterations or search_iterations
    fixed_iterations = (
        max(10, int(round(float(np.median(usable_iterations)))))
        if usable_iterations
        else None
    )

    with threadpool_limits(limits=MAX_THREADS):
        fitted, raw_outer = fit_predictor(
            spec,
            params,
            outer_training,
            outer_validation,
            outer_features,
            fixed_iterations=fixed_iterations,
            random_seed=SEED + validation_season,
        )
    assert raw_outer is not None
    calibrated_outer = calibration.fitted_calibrator.predict(
        raw_outer
    )
    outer_predictions = make_prediction_frame(
        outer_validation,
        outer_fold_id=outer_fold_id,
        inner_fold_id=None,
        architecture=str(outer["Architecture"]),
        universe=universe,
        model_name=(
            spec.name
            if seed_aware
            else f"{spec.name}__seed_free"
        ),
        raw_prediction=raw_outer,
        feature_count=len(outer_features),
        best_iteration=fitted.best_iteration,
    )
    outer_predictions["Prediction"] = calibrated_outer
    outer_predictions["CalibrationMethod"] = calibration.method

    outer_metrics, _ = evaluate_probability_frame(
        outer_predictions
    )
    elapsed = time.perf_counter() - start_time

    calibration_audit = calibration.audit.copy()
    calibration_audit["OuterFoldID"] = outer_fold_id
    calibration_audit["Model"] = spec.name
    calibration_audit["ValidationSeason"] = validation_season
    calibration_audit["SeedAware"] = bool(seed_aware)

    selected_feature_frame = (
        pd.concat(selector_audits, ignore_index=True)
        if selector_audits
        else pd.DataFrame(
            {
                "Feature": outer_features,
                "Block": [
                    FEATURE_TO_BLOCK.get(feature, "other")
                    for feature in outer_features
                ],
                "MissingRate": np.nan,
                "NonMissingRows": len(outer_training),
                "UniqueValues": [
                    int(outer_training[feature].nunique(dropna=True))
                    for feature in outer_features
                ],
                "GlobalCorrelation": np.nan,
                "MedianAbsoluteSeasonCorrelation": np.nan,
                "SeasonSignConsistency": np.nan,
                "SeasonCoverage": np.nan,
                "SeasonCorrelationIQR": np.nan,
                "StabilityScore": np.nan,
                "Mandatory": True,
                "InPrecorrelationPool": True,
                "Selected": True,
                "SelectionRank": np.arange(1, len(outer_features) + 1),
                "ExclusionReason": "preregistered_baseline",
                "OuterFoldID": outer_fold_id,
                "InnerFoldID": None,
                "Model": spec.name,
            }
        )
    )
    selected_feature_frame["SeedAware"] = bool(seed_aware)

    coefficient_frame = pd.DataFrame()
    if hasattr(fitted.model, "coef_"):
        coefficients = np.asarray(fitted.model.coef_, dtype=float).reshape(-1)
        if len(coefficients) == len(outer_features):
            coefficient_frame = pd.DataFrame(
                {
                    "OuterFoldID": outer_fold_id,
                    "ValidationSeason": validation_season,
                    "Gender": gender,
                    "Architecture": str(outer["Architecture"]),
                    "Universe": universe,
                    "Model": spec.name,
                    "Feature": outer_features,
                    "Block": [
                        FEATURE_TO_BLOCK.get(feature, "other")
                        for feature in outer_features
                    ],
                    "StandardizedCoefficient": coefficients,
                }
            )

    metadata = {
        "status": "complete",
        "contracts": EXPECTED_CONTRACTS,
        "run_mode": RUN_MODE,
        "outer_fold_id": outer_fold_id,
        "architecture": str(outer["Architecture"]),
        "universe": universe,
        "gender": gender,
        "validation_season": validation_season,
        "model": spec.name,
        "seed_aware": bool(seed_aware),
        "task": spec.task,
        "raw_kind": spec.raw_kind,
        "parameters": params,
        "fixed_iterations_from_inner": fixed_iterations,
        "inner_best_iterations": list(
            map(int, usable_iterations)
        ),
        "outer_selected_features": outer_features,
        "outer_feature_count": len(outer_features),
        "calibration_method": calibration.method,
        "outer_metrics": outer_metrics,
        "search": search_metadata,
        "elapsed_seconds": elapsed,
        "rss_start_mb": start_rss,
        "rss_end_mb": rss_mb(),
        "loaded_from_checkpoint": False,
        "memory_before_training": memory_before_training,
    }

    directory.mkdir(parents=True, exist_ok=True)
    atomic_write_parquet(
        directory / "inner_oof.parquet",
        inner_oof,
    )
    atomic_write_parquet(
        directory / "outer_predictions.parquet",
        outer_predictions,
    )
    atomic_write_csv(
        directory / "selected_features.csv",
        selected_feature_frame,
    )
    atomic_write_csv(
        directory / "calibration_audit.csv",
        calibration_audit,
    )
    if not coefficient_frame.empty:
        atomic_write_csv(
            directory / "linear_coefficients.csv",
            coefficient_frame,
        )
    atomic_write_json(directory / "metadata.json", metadata)

    SELECTED_FEATURE_RECORDS.append(selected_feature_frame)
    CALIBRATION_RECORDS.append(calibration_audit)
    MODEL_RUN_METADATA.append(metadata)

    del (
        outer_training,
        outer_validation,
        fitted,
        raw_outer,
        calibration,
        inner_oof,
    )
    if spec.name == "tensorflow_mlp" and tf is not None:
        tf.keras.utils.clear_session(free_memory=True)
    gc.collect()
    return (
        pd.read_parquet(directory / "outer_predictions.parquet"),
        pd.read_parquet(directory / "inner_oof.parquet"),
        metadata,
    )


def failure_record(
    *,
    outer: pd.Series,
    model_name: str,
    exc: Exception,
) -> None:
    record = {
        "OuterFoldID": str(outer["OuterFoldID"]),
        "ValidationSeason": int(outer["ValidationSeason"]),
        "Architecture": str(outer["Architecture"]),
        "Universe": str(outer["Universe"]),
        "Gender": str(outer["Gender"]),
        "Model": model_name,
        "ErrorType": type(exc).__name__,
        "Error": str(exc),
        "Traceback": traceback.format_exc(),
    }
    MODEL_FAILURES.append(record)
    atomic_write_json(
        FAILURE_DIR
        / f"{sanitize_name(record['OuterFoldID'])}"
        f"__{sanitize_name(model_name)}.json",
        record,
    )


print("Outer-fold runner ready.")


### Framework execution preflight

Before the full comparison starts, the notebook runs one real development fold through logistic regression, XGBoost, LightGBM, PyTorch, and TensorFlow. This catches framework, dtype, early-stopping, and serialization problems before the longer experiment grid begins. The completed checkpoints are reused by the full run.

In [ ]:
framework_preflight_models = [
    "elastic_logistic",
    "xgb_classifier",
    "lgb_classifier",
    "torch_mlp",
    "tensorflow_mlp",
]
preflight_context = outer_folds.loc[
    outer_folds["Universe"].eq("rich")
    & outer_folds["Architecture"].eq("separate_gender")
    & outer_folds["Gender"].eq("M")
    & outer_folds["ValidationSeason"].eq(
        max(MATCHED_VALIDATION_SEASONS)
    )
]
assert len(preflight_context) == 1, (
    "Expected exactly one men's 2021 rich outer fold."
)
preflight_outer = preflight_context.iloc[0]

framework_preflight_records = []
for model_name in framework_preflight_models:
    started = time.perf_counter()
    outer_prediction, inner_prediction, metadata = run_outer_model(
        preflight_outer,
        model_name,
        seed_aware=True,
    )
    assert not outer_prediction.empty
    assert not inner_prediction.empty
    assert np.isfinite(
        outer_prediction["Prediction"].to_numpy(dtype=float)
    ).all()
    assert outer_prediction["Prediction"].between(0, 1).all()
    framework_preflight_records.append(
        {
            "Model": model_name,
            "ModelDisplay": MODEL_DISPLAY_NAMES.get(
                model_name, model_name
            ),
            "OuterFoldID": str(preflight_outer["OuterFoldID"]),
            "ValidationSeason": int(
                preflight_outer["ValidationSeason"]
            ),
            "OuterRows": int(len(outer_prediction)),
            "InnerOOFRows": int(len(inner_prediction)),
            "SelectedFeatures": int(
                metadata["outer_feature_count"]
            ),
            "CalibrationMethod": str(
                metadata["calibration_method"]
            ),
            "ElapsedSeconds": float(
                time.perf_counter() - started
            ),
            "Passed": True,
        }
    )
    if model_name == "tensorflow_mlp" and tf is not None:
        tf.keras.utils.clear_session(free_memory=True)
    gc.collect()

framework_preflight = pd.DataFrame(framework_preflight_records)
framework_preflight.to_csv(
    MODEL_REPORTS / "framework_execution_preflight.csv",
    index=False,
)
assert set(framework_preflight["Model"]) == set(
    framework_preflight_models
)
assert framework_preflight["Passed"].all()
log_event(
    "framework_execution_preflight_complete",
    models=framework_preflight_models,
    outer_fold_id=str(preflight_outer["OuterFoldID"]),
)

# Remove the final loop references and release framework state before the
# 180-task comparison begins.
for _name in ("outer_prediction", "inner_prediction", "metadata"):
    globals().pop(_name, None)
post_preflight_memory = release_runtime_memory(aggressive=True)
print(
    "Post-preflight memory:",
    json.dumps(post_preflight_memory, indent=2),
)
if post_preflight_memory["available_physical_gb"] < 1.0:
    print("Largest current working sets:")
    display(top_memory_processes(limit=10))
framework_preflight


## 10. Matched experiment grid

Every requested framework is compared on the same rich-feature outer seasons: 2016, 2017, 2018, 2019, and 2021. The year 2020 has no NCAA tournament target.

In [ ]:
def build_outer_execution_plan() -> pd.DataFrame:
    plan = outer_folds.loc[
        outer_folds["Universe"].eq("rich")
        & outer_folds["ValidationSeason"].isin(
            MATCHED_VALIDATION_SEASONS
        )
        & (
            (
                outer_folds["Architecture"].eq("separate_gender")
                & outer_folds["Gender"].isin(["M", "W"])
            )
            | (
                outer_folds["Architecture"].eq("pooled_common")
                & outer_folds["Gender"].eq("Pooled")
            )
        )
    ].copy()

    expected_contexts = {
        ("separate_gender", "M"),
        ("separate_gender", "W"),
        ("pooled_common", "Pooled"),
    }
    observed_contexts = set(
        map(
            tuple,
            plan[["Architecture", "Gender"]]
            .drop_duplicates()
            .itertuples(index=False, name=None),
        )
    )
    assert observed_contexts == expected_contexts, (
        f"Matched comparison contexts are incomplete: {observed_contexts}"
    )

    for architecture, gender in sorted(expected_contexts):
        seasons = set(
            map(
                int,
                plan.loc[
                    plan["Architecture"].eq(architecture)
                    & plan["Gender"].eq(gender),
                    "ValidationSeason",
                ],
            )
        )
        assert seasons == set(MATCHED_VALIDATION_SEASONS), (
            f"{architecture}/{gender} has seasons {sorted(seasons)}; "
            f"expected {MATCHED_VALIDATION_SEASONS}"
        )

    if MODE["outer_folds_per_context"] is not None:
        count = int(MODE["outer_folds_per_context"])
        plan = (
            plan.sort_values("ValidationSeason")
            .groupby(
                ["Architecture", "Gender"],
                observed=True,
                group_keys=False,
            )
            .tail(count)
        )

    plan["ModelsJSON"] = plan.apply(
        lambda row: json.dumps(model_plan_for_outer(row)),
        axis=1,
    )
    return plan.sort_values(
        ["Architecture", "Gender", "ValidationSeason"]
    ).reset_index(drop=True)


outer_plan = build_outer_execution_plan()
outer_plan.to_csv(
    MODEL_REPORTS / "outer_execution_plan.csv",
    index=False,
)

plan_summary = (
    outer_plan.groupby(
        ["Architecture", "Gender"],
        observed=True,
    )
    .agg(
        Folds=("OuterFoldID", "nunique"),
        FirstSeason=("ValidationSeason", "min"),
        LastSeason=("ValidationSeason", "max"),
    )
    .reset_index()
)
planned_model_tasks = int(
    sum(len(json.loads(value)) for value in outer_plan["ModelsJSON"])
)

fig = go.Figure()
for (architecture, gender), group in outer_plan.groupby(
    ["Architecture", "Gender"], observed=True
):
    context_label = (
        architecture.replace("_", " ").title() + " / " + str(gender)
    )
    x_values: list[Any] = []
    y_values: list[Any] = []
    hover_values: list[Any] = []
    for row in group.sort_values("ValidationSeason").itertuples():
        x_values.extend(
            [int(row.TrainingSeasonMin), int(row.ValidationSeason), None]
        )
        y_values.extend([context_label, context_label, None])
        hover_values.extend(
            [
                f"{row.OuterFoldID}: train {row.TrainingSeasonMin}–"
                f"{row.TrainingSeasonMax}",
                f"validate {row.ValidationSeason}",
                None,
            ]
        )
    fig.add_trace(
        go.Scatter(
            x=x_values,
            y=y_values,
            mode="lines+markers",
            name=context_label,
            text=hover_values,
            hovertemplate="%{text}<extra></extra>",
        )
    )
fig.update_layout(
    title="Chronological outer-fold design",
    xaxis_title="Season",
    yaxis_title="Training context",
    showlegend=False,
)
save_plotly(fig, "outer_fold_design")

log_event(
    "outer_plan_created",
    folds=int(len(outer_plan)),
    contexts=int(len(plan_summary)),
    planned_model_tasks=planned_model_tasks,
)
display(plan_summary)
print("Planned fold/model tasks:", planned_model_tasks)

In [ ]:
OUTER_PREDICTIONS: list[pd.DataFrame] = []
INNER_OOF_BY_OUTER_MODEL: dict[
    tuple[str, str], pd.DataFrame
] = {}
RUN_PROGRESS: list[dict[str, Any]] = []

total_tasks = int(
    sum(len(json.loads(value)) for value in outer_plan["ModelsJSON"])
)
completed_task_number = 0

for _, outer in outer_plan.iterrows():
    model_names = json.loads(outer["ModelsJSON"])
    for model_name in model_names:
        completed_task_number += 1
        label = (
            f"[{completed_task_number}/{total_tasks}] "
            f"{outer['OuterFoldID']} :: {model_name}"
        )
        print(label)
        log_event("model_task_started", label=label)
        try:
            with ResourceTimer(
                "outer_model",
                outer_fold_id=str(outer["OuterFoldID"]),
                model_name=model_name,
            ):
                outer_prediction, inner_oof, metadata = run_outer_model(
                    outer,
                    model_name,
                    seed_aware=True,
                )
            OUTER_PREDICTIONS.append(outer_prediction)
            INNER_OOF_BY_OUTER_MODEL[
                (str(outer["OuterFoldID"]), model_name)
            ] = inner_oof
            RUN_PROGRESS.append(
                {
                    "OuterFoldID": outer["OuterFoldID"],
                    "Model": model_name,
                    "Status": "complete",
                    "LoadedFromCheckpoint": bool(
                        metadata.get(
                            "LoadedFromCheckpoint",
                            metadata.get("loaded_from_checkpoint", False),
                        )
                    ),
                    "MacroSeasonBrier": metadata[
                        "outer_metrics"
                    ]["MacroSeasonBrier"],
                    "FeatureCount": metadata[
                        "outer_feature_count"
                    ],
                    "CalibrationMethod": metadata[
                        "calibration_method"
                    ],
                    "Architecture": str(outer["Architecture"]),
                    "TrainingContext": str(outer["Gender"]),
                    "ValidationSeason": int(outer["ValidationSeason"]),
                    "ElapsedSeconds": float(
                        metadata.get("elapsed_seconds", np.nan)
                    ),
                    "EndRSSMB": float(rss_mb()),
                }
            )
            log_event(
                "model_task_completed",
                outer_fold_id=str(outer["OuterFoldID"]),
                model=model_name,
                architecture=str(outer["Architecture"]),
                training_context=str(outer["Gender"]),
                validation_season=int(outer["ValidationSeason"]),
                feature_count=int(metadata["outer_feature_count"]),
                calibration_method=str(metadata["calibration_method"]),
                macro_season_brier=float(
                    metadata["outer_metrics"]["MacroSeasonBrier"]
                ),
            )
        except MemoryPressurePause as exc:
            pause_message = str(exc)
            print("MEMORY PRESSURE PAUSE:\n", pause_message)
            log_event(
                "model_execution_paused_for_memory",
                outer_fold_id=str(outer["OuterFoldID"]),
                model=model_name,
                message=pause_message,
            )
            pd.DataFrame(RUN_PROGRESS).to_csv(
                MODEL_REPORTS / f"run_progress_{RUN_MODE}.csv",
                index=False,
            )
            raise
        except Exception as exc:
            failure_record(
                outer=outer,
                model_name=model_name,
                exc=exc,
            )
            RUN_PROGRESS.append(
                {
                    "OuterFoldID": outer["OuterFoldID"],
                    "Model": model_name,
                    "Status": "failed",
                    "LoadedFromCheckpoint": False,
                    "MacroSeasonBrier": np.nan,
                    "FeatureCount": np.nan,
                    "CalibrationMethod": None,
                    "Error": repr(exc),
                }
            )
            print("FAILED:", repr(exc))
            log_event(
                "model_task_failed",
                outer_fold_id=str(outer["OuterFoldID"]),
                model=model_name,
                error=repr(exc),
            )
        finally:
            release_runtime_memory(
                aggressive=(
                    psutil.virtual_memory().available / 1024**3
                    < SOFT_WARNING_FLOOR_GB
                )
            )

run_progress = pd.DataFrame(RUN_PROGRESS)
run_progress.to_csv(
    MODEL_REPORTS / f"run_progress_{RUN_MODE}.csv",
    index=False,
)
failure_report_path = MODEL_REPORTS / f"model_failures_{RUN_MODE}.csv"
if MODEL_FAILURES:
    pd.DataFrame(MODEL_FAILURES).to_csv(
        failure_report_path,
        index=False,
    )
elif failure_report_path.exists():
    # Remove a stale report from an earlier failed diagnostic run.
    failure_report_path.unlink()

print(
    run_progress.groupby(["Status", "Model"], observed=True)
    .size()
    .to_frame("Tasks")
    .reset_index()
)
print("Current RSS MB:", round(rss_mb(), 2))


## 11. Constrained cross-family ensembles

Ensemble weights are nonnegative, sum to one, and are learned from inner out-of-fold predictions. Highly redundant prediction streams are removed before optimization.

In [ ]:
def prediction_correlation_prune(
    wide: pd.DataFrame,
    model_columns: Sequence[str],
    *,
    maximum_members: int,
    threshold: float,
) -> tuple[list[str], pd.DataFrame]:
    ranking_records = []
    for model in model_columns:
        frame = wide[
            ["TargetKey", "Season", "Team1Win", model]
        ].rename(columns={model: "Prediction"})
        metrics, _ = evaluate_probability_frame(frame)
        ranking_records.append(
            {
                "Model": model,
                "MacroSeasonBrier": metrics["MacroSeasonBrier"],
            }
        )
    ranking = pd.DataFrame(ranking_records).sort_values(
        "MacroSeasonBrier"
    )
    correlation = wide[list(model_columns)].corr().abs()

    kept: list[str] = []
    for model in ranking["Model"]:
        if not kept:
            kept.append(model)
            continue
        if all(
            float(correlation.loc[model, existing]) < threshold
            for existing in kept
        ):
            kept.append(model)
        if len(kept) >= maximum_members:
            break

    if len(kept) == 1 and len(ranking) > 1:
        # Preserve at least two members when available, even if highly
        # correlated; regularization can assign zero weight.
        kept.append(
            next(
                model
                for model in ranking["Model"]
                if model not in kept
            )
        )
    ranking["KeptForWeightOptimization"] = ranking["Model"].isin(kept)
    return kept, ranking


def macro_season_brier_from_arrays(
    y: np.ndarray,
    p: np.ndarray,
    seasons: np.ndarray,
) -> float:
    values = []
    for season in np.unique(seasons):
        mask = seasons == season
        values.append(float(np.mean((p[mask] - y[mask]) ** 2)))
    return float(np.mean(values))


def optimize_simplex_weights(
    prediction_matrix: np.ndarray,
    y: np.ndarray,
    seasons: np.ndarray,
    *,
    l2_penalty: float = 0.002,
) -> np.ndarray:
    prediction_matrix = np.asarray(
        prediction_matrix, dtype=np.float64
    )
    y = np.asarray(y, dtype=np.float64)
    seasons = np.asarray(seasons, dtype=int)
    n_models = prediction_matrix.shape[1]
    if n_models == 1:
        return np.ones(1, dtype=np.float64)

    initial = np.full(n_models, 1.0 / n_models)

    def objective(weights: np.ndarray) -> float:
        probability = clip_probability(
            prediction_matrix @ weights
        )
        return float(
            macro_season_brier_from_arrays(
                y, probability, seasons
            )
            + l2_penalty * np.sum(weights**2)
        )

    result = minimize(
        objective,
        initial,
        method="SLSQP",
        bounds=[(0.0, 1.0)] * n_models,
        constraints=[
            {
                "type": "eq",
                "fun": lambda weights: float(np.sum(weights) - 1.0),
            }
        ],
        options={"maxiter": 2000, "ftol": 1e-12},
    )
    if not result.success:
        warnings.warn(
            f"Ensemble optimizer did not converge: {result.message}. "
            "Using equal weights."
        )
        return initial
    weights = np.clip(result.x, 0.0, 1.0)
    return weights / weights.sum()


ENSEMBLE_WEIGHT_RECORDS: list[pd.DataFrame] = []
ENSEMBLE_PREDICTIONS: list[pd.DataFrame] = []
ENSEMBLE_INNER_OOF: dict[str, pd.DataFrame] = {}


def build_outer_ensemble(
    outer: pd.Series,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    outer_fold_id = str(outer["OuterFoldID"])
    directory = outer_model_directory(
        outer_fold_id, "constrained_ensemble"
    )
    if checkpoint_is_valid(directory):
        outer_prediction, inner_oof, metadata = load_outer_checkpoint(
            directory
        )
        weights = pd.read_csv(directory / "ensemble_weights.csv")
        return outer_prediction, inner_oof, weights

    available_models = [
        model_name
        for model_name in json.loads(outer["ModelsJSON"])
        if (
            outer_fold_id,
            model_name,
        ) in INNER_OOF_BY_OUTER_MODEL
        and model_name != "constant_probability"
    ]
    if len(available_models) < 2:
        raise RuntimeError(
            f"Fewer than two models available for ensemble {outer_fold_id}"
        )

    inner_base = None
    for model_name in available_models:
        frame = INNER_OOF_BY_OUTER_MODEL[
            (outer_fold_id, model_name)
        ][
            [
                "TargetKey",
                "Gender",
                "Season",
                "Team1Win",
                "Team1Margin",
                "Prediction",
            ]
        ].rename(columns={"Prediction": model_name})
        inner_base = (
            frame
            if inner_base is None
            else inner_base.merge(
                frame[
                    ["TargetKey", model_name]
                ],
                on="TargetKey",
                how="inner",
                validate="one_to_one",
            )
        )
    assert inner_base is not None

    outer_model_frames = [
        frame
        for frame in OUTER_PREDICTIONS
        if (
            frame["OuterFoldID"].iloc[0] == outer_fold_id
            and frame["Model"].iloc[0] in available_models
        )
    ]
    outer_base = None
    for frame in outer_model_frames:
        model_name = str(frame["Model"].iloc[0])
        part = frame[
            [
                "TargetKey",
                "Gender",
                "Season",
                "Team1ID",
                "Team2ID",
                "Team1Win",
                "Team1Margin",
                "Prediction",
            ]
        ].rename(columns={"Prediction": model_name})
        outer_base = (
            part
            if outer_base is None
            else outer_base.merge(
                part[["TargetKey", model_name]],
                on="TargetKey",
                how="inner",
                validate="one_to_one",
            )
        )
    assert outer_base is not None

    common_models = [
        model
        for model in available_models
        if model in inner_base.columns and model in outer_base.columns
    ]
    kept, ranking = prediction_correlation_prune(
        inner_base,
        common_models,
        maximum_members=int(
            MODEL_CONFIG["ensemble"]["maximum_members"]
        ),
        threshold=float(
            MODEL_CONFIG["ensemble"][
                "redundancy_correlation_threshold"
            ]
        ),
    )
    matrix_inner = inner_base[kept].to_numpy(dtype=np.float64)
    y_inner = inner_base["Team1Win"].to_numpy(dtype=float)
    season_inner = inner_base["Season"].to_numpy(dtype=int)
    weights = optimize_simplex_weights(
        matrix_inner,
        y_inner,
        season_inner,
    )
    raw_inner = clip_probability(matrix_inner @ weights)

    ensemble_inner = inner_base[
        ["TargetKey", "Gender", "Season", "Team1Win", "Team1Margin"]
    ].copy()
    ensemble_inner["OuterFoldID"] = outer_fold_id
    ensemble_inner["InnerFoldID"] = "ensemble_from_inner_oof"
    ensemble_inner["Architecture"] = str(outer["Architecture"])
    ensemble_inner["Universe"] = str(outer["Universe"])
    ensemble_inner["Model"] = "constrained_ensemble"
    ensemble_inner["RawPrediction"] = raw_inner
    ensemble_inner["FeatureCount"] = np.nan
    ensemble_inner["BestIteration"] = np.nan

    calibration = select_cross_fitted_calibrator(
        ensemble_inner,
        raw_kind="probability",
    )
    ensemble_inner["Prediction"] = (
        calibration.cross_fitted_predictions
    )
    ensemble_inner["CalibrationMethod"] = calibration.method

    raw_outer = clip_probability(
        outer_base[kept].to_numpy(dtype=np.float64) @ weights
    )
    outer_probability = calibration.fitted_calibrator.predict(
        raw_outer
    )
    ensemble_outer = outer_base[
        [
            "TargetKey",
            "Gender",
            "Season",
            "Team1ID",
            "Team2ID",
            "Team1Win",
            "Team1Margin",
        ]
    ].copy()
    ensemble_outer["OuterFoldID"] = outer_fold_id
    ensemble_outer["InnerFoldID"] = None
    ensemble_outer["Architecture"] = str(outer["Architecture"])
    ensemble_outer["Universe"] = str(outer["Universe"])
    ensemble_outer["Model"] = "constrained_ensemble"
    ensemble_outer["RawPrediction"] = raw_outer
    ensemble_outer["Prediction"] = outer_probability
    ensemble_outer["CalibrationMethod"] = calibration.method
    ensemble_outer["FeatureCount"] = np.nan
    ensemble_outer["BestIteration"] = np.nan

    weight_frame = pd.DataFrame(
        {
            "OuterFoldID": outer_fold_id,
            "Model": kept,
            "Weight": weights,
            "ValidationSeason": int(outer["ValidationSeason"]),
            "Architecture": str(outer["Architecture"]),
            "Universe": str(outer["Universe"]),
            "Gender": str(outer["Gender"]),
        }
    ).merge(ranking, on="Model", how="left")
    outer_metrics, _ = evaluate_probability_frame(ensemble_outer)

    metadata = {
        "status": "complete",
        "contracts": EXPECTED_CONTRACTS,
        "run_mode": RUN_MODE,
        "outer_fold_id": outer_fold_id,
        "model": "constrained_ensemble",
        "architecture": str(outer["Architecture"]),
        "universe": str(outer["Universe"]),
        "gender": str(outer["Gender"]),
        "validation_season": int(outer["ValidationSeason"]),
        "members": kept,
        "weights": dict(zip(kept, map(float, weights), strict=True)),
        "calibration_method": calibration.method,
        "outer_metrics": outer_metrics,
        "outer_feature_count": None,
        "loaded_from_checkpoint": False,
    }
    directory.mkdir(parents=True, exist_ok=True)
    atomic_write_parquet(
        directory / "inner_oof.parquet", ensemble_inner
    )
    atomic_write_parquet(
        directory / "outer_predictions.parquet", ensemble_outer
    )
    atomic_write_csv(
        directory / "ensemble_weights.csv", weight_frame
    )
    atomic_write_csv(
        directory / "calibration_audit.csv",
        calibration.audit.assign(
            OuterFoldID=outer_fold_id,
            Model="constrained_ensemble",
        ),
    )
    atomic_write_json(directory / "metadata.json", metadata)
    return ensemble_outer, ensemble_inner, weight_frame


for _, outer in outer_plan.iterrows():
    try:
        ensemble_outer, ensemble_inner, weight_frame = (
            build_outer_ensemble(outer)
        )
        ENSEMBLE_PREDICTIONS.append(ensemble_outer)
        ENSEMBLE_INNER_OOF[
            str(outer["OuterFoldID"])
        ] = ensemble_inner
        ENSEMBLE_WEIGHT_RECORDS.append(weight_frame)
        print(
            "Ensemble complete:",
            outer["OuterFoldID"],
            "members=",
            len(weight_frame),
        )
    except Exception as exc:
        failure_record(
            outer=outer,
            model_name="constrained_ensemble",
            exc=exc,
        )
        print(
            "ENSEMBLE FAILED:",
            outer["OuterFoldID"],
            repr(exc),
        )
    finally:
        gc.collect()

if ENSEMBLE_WEIGHT_RECORDS:
    ensemble_weights = pd.concat(
        ENSEMBLE_WEIGHT_RECORDS, ignore_index=True
    )
    ensemble_weights.to_csv(
        MODEL_REPORTS / f"ensemble_weights_{RUN_MODE}.csv",
        index=False,
    )
else:
    ensemble_weights = pd.DataFrame()

print("Completed ensembles:", len(ENSEMBLE_PREDICTIONS))


## 12. Partial pooling

For each tournament, the notebook compares its separate model with the pooled common-feature model. Blend weights use only predictions from earlier validation seasons.

In [ ]:
def optimize_two_model_weight(
    frame: pd.DataFrame,
    *,
    separate_column: str,
    pooled_column: str,
) -> float:
    y = frame["Team1Win"].to_numpy(dtype=float)
    separate = clip_probability(frame[separate_column])
    pooled = clip_probability(frame[pooled_column])
    seasons = frame["Season"].to_numpy(dtype=int)

    def objective(weight: float) -> float:
        prediction = weight * separate + (1.0 - weight) * pooled
        return macro_season_brier_from_arrays(
            y,
            prediction,
            seasons,
        ) + 0.001 * (weight - 0.5) ** 2

    result = minimize_scalar(
        objective,
        bounds=(0.0, 1.0),
        method="bounded",
    )
    return float(np.clip(result.x, 0.0, 1.0))


def ensemble_outer_table() -> pd.DataFrame:
    if not ENSEMBLE_PREDICTIONS:
        return pd.DataFrame()
    return pd.concat(
        ENSEMBLE_PREDICTIONS, ignore_index=True
    )


ensemble_outer_all = ensemble_outer_table()
PARTIAL_POOLING_PREDICTIONS: list[pd.DataFrame] = []
PARTIAL_POOLING_WEIGHTS: list[dict[str, Any]] = []

if not ensemble_outer_all.empty:
    rich_ensemble = ensemble_outer_all.loc[
        ensemble_outer_all["Universe"].eq("rich")
    ].copy()

    aligned_history_by_gender: dict[str, pd.DataFrame] = {}
    for gender in ("M", "W"):
        separate = rich_ensemble.loc[
            rich_ensemble["Architecture"].eq("separate_gender")
            & rich_ensemble["Gender"].eq(gender)
        ][
            [
                "TargetKey",
                "Gender",
                "Season",
                "Team1ID",
                "Team2ID",
                "Team1Win",
                "Team1Margin",
                "Prediction",
            ]
        ].rename(columns={"Prediction": "SeparatePrediction"})

        pooled = rich_ensemble.loc[
            rich_ensemble["Architecture"].eq("pooled_common")
            & rich_ensemble["Gender"].eq(gender)
        ][["TargetKey", "Prediction"]].rename(
            columns={"Prediction": "PooledPrediction"}
        )

        aligned = separate.merge(
            pooled,
            on="TargetKey",
            how="inner",
            validate="one_to_one",
        ).sort_values(["Season", "TargetKey"])
        aligned_history_by_gender[gender] = aligned

        for validation_season in sorted(
            map(int, aligned["Season"].unique())
        ):
            prior = aligned.loc[
                aligned["Season"].lt(validation_season)
            ]
            if (
                prior["Season"].nunique() >= 2
                and len(prior) >= 100
            ):
                weight = optimize_two_model_weight(
                    prior,
                    separate_column="SeparatePrediction",
                    pooled_column="PooledPrediction",
                )
                source = "prior_aligned_outer_oof"
            else:
                weight = 0.75
                source = "preregistered_cold_start"

            current = aligned.loc[
                aligned["Season"].eq(validation_season)
            ].copy()
            current["Prediction"] = clip_probability(
                weight * current["SeparatePrediction"]
                + (1.0 - weight) * current["PooledPrediction"]
            )
            current["RawPrediction"] = current["Prediction"]
            current["OuterFoldID"] = (
                f"rich_partial_{gender}_{validation_season}"
            )
            current["InnerFoldID"] = None
            current["Architecture"] = "partial_pooling"
            current["Universe"] = "rich"
            current["Model"] = "partial_pooling_ensemble"
            current["CalibrationMethod"] = (
                "component_calibrated_linear_pool"
            )
            current["FeatureCount"] = np.nan
            current["BestIteration"] = np.nan
            PARTIAL_POOLING_PREDICTIONS.append(
                current[
                    [
                        "TargetKey",
                        "Gender",
                        "Season",
                        "Team1ID",
                        "Team2ID",
                        "Team1Win",
                        "Team1Margin",
                        "OuterFoldID",
                        "InnerFoldID",
                        "Architecture",
                        "Universe",
                        "Model",
                        "RawPrediction",
                        "Prediction",
                        "CalibrationMethod",
                        "FeatureCount",
                        "BestIteration",
                    ]
                ]
            )
            PARTIAL_POOLING_WEIGHTS.append(
                {
                    "Gender": gender,
                    "ValidationSeason": validation_season,
                    "SeparateWeight": weight,
                    "PooledWeight": 1.0 - weight,
                    "WeightSource": source,
                    "PriorSeasons": sorted(
                        map(int, prior["Season"].unique())
                    ),
                    "PriorRows": len(prior),
                }
            )

partial_pooling_weights = pd.DataFrame(
    PARTIAL_POOLING_WEIGHTS
)
partial_pooling_weights.to_csv(
    MODEL_REPORTS / f"partial_pooling_weights_{RUN_MODE}.csv",
    index=False,
)

print(
    "Partial-pooling prediction blocks:",
    len(PARTIAL_POOLING_PREDICTIONS),
)
partial_pooling_weights.tail(10)


## 13. Scoreboard and model winners

This section produces the complete metric table, the lowest-score winner, the one-standard-error recommendation, the best separate model, and the best pooled model for each tournament.

In [ ]:
prediction_parts = list(OUTER_PREDICTIONS) + list(
    ENSEMBLE_PREDICTIONS
) + list(PARTIAL_POOLING_PREDICTIONS)
assert prediction_parts, "No development OOF predictions were generated."

development_oof = pd.concat(
    prediction_parts,
    ignore_index=True,
)
development_oof["Prediction"] = clip_probability(
    development_oof["Prediction"]
)
assert development_oof["Season"].max() <= DEVELOPMENT_LAST_SEASON
assert not development_oof["Season"].isin(LOCKED_SEASONS).any()
assert development_oof["Team1Win"].isin([0, 1]).all()
assert development_oof["Prediction"].between(0, 1).all()

duplicate_keys = development_oof.duplicated(
    ["OuterFoldID", "Model", "TargetKey"]
).sum()
assert duplicate_keys == 0, (
    f"Duplicate OOF predictions: {duplicate_keys}"
)

oof_path = (
    PROCESSED
    / "development_oof_predictions_model_comparison.parquet"
)
atomic_write_parquet(oof_path, development_oof)

METRIC_RECORDS: list[dict[str, Any]] = []
SEASON_METRIC_RECORDS: list[pd.DataFrame] = []


def add_metric_group(
    group: pd.DataFrame,
    *,
    architecture: str,
    universe: str,
    model: str,
    evaluation_gender: str,
) -> None:
    metrics, by_season = evaluate_probability_frame(group)
    METRIC_RECORDS.append(
        {
            "Architecture": architecture,
            "Universe": universe,
            "Model": model,
            "ModelDisplay": MODEL_DISPLAY_NAMES.get(model, model),
            "EvaluationGender": evaluation_gender,
            "ComplexityRank": (
                MODEL_SPECS[model].complexity_rank
                if model in MODEL_SPECS
                else 20
                if model == "constrained_ensemble"
                else 21
            ),
            **metrics,
        }
    )
    by_season["Architecture"] = architecture
    by_season["Universe"] = universe
    by_season["Model"] = model
    by_season["EvaluationGender"] = evaluation_gender
    SEASON_METRIC_RECORDS.append(by_season)


for (
    architecture,
    universe,
    model,
), group in development_oof.groupby(
    ["Architecture", "Universe", "Model"],
    observed=True,
):
    add_metric_group(
        group,
        architecture=str(architecture),
        universe=str(universe),
        model=str(model),
        evaluation_gender="ALL",
    )
    for gender, gender_group in group.groupby(
        "Gender", observed=True
    ):
        add_metric_group(
            gender_group,
            architecture=str(architecture),
            universe=str(universe),
            model=str(model),
            evaluation_gender=str(gender),
        )

leaderboard = pd.DataFrame(METRIC_RECORDS).sort_values(
    [
        "EvaluationGender",
        "Universe",
        "MacroSeasonBrier",
        "ComplexityRank",
    ]
).reset_index(drop=True)
season_metrics = pd.concat(
    SEASON_METRIC_RECORDS,
    ignore_index=True,
)

leaderboard.to_csv(
    MODEL_REPORTS / "development_leaderboard.csv",
    index=False,
)
season_metrics.to_csv(
    MODEL_REPORTS
    / "development_metrics_by_season.csv",
    index=False,
)

display_columns = [
    "EvaluationGender",
    "Architecture",
    "ModelDisplay",
    "Rows",
    "Seasons",
    "MacroSeasonBrier",
    "GameWeightedBrier",
    "LogLoss",
    "ROCAUC",
    "AveragePrecision",
    "AUCPR",
    "Accuracy",
    "BalancedAccuracy",
    "Precision",
    "Recall",
    "F1",
    "MCC",
    "CalibrationIntercept",
    "CalibrationSlope",
    "ECE",
]
log_event(
    "development_metrics_created",
    prediction_rows=int(len(development_oof)),
    leaderboard_rows=int(len(leaderboard)),
    models=int(development_oof["Model"].nunique()),
)
leaderboard[display_columns].head(60)


In [ ]:
# Fair comparison: every model and architecture is scored on the same seasons.
comparison_rows = development_oof.loc[
    development_oof["Universe"].eq("rich")
    & development_oof["Season"].isin(MATCHED_VALIDATION_SEASONS)
].copy()

common_records: list[dict[str, Any]] = []
for gender in ("M", "W"):
    gender_rows = comparison_rows.loc[
        comparison_rows["Gender"].eq(gender)
    ]
    for (architecture, model), group in gender_rows.groupby(
        ["Architecture", "Model"],
        observed=True,
    ):
        observed_seasons = sorted(map(int, group["Season"].unique()))
        if observed_seasons != MATCHED_VALIDATION_SEASONS:
            continue
        metrics, _ = evaluate_probability_frame(group)
        common_records.append(
            {
                "Gender": gender,
                "Architecture": str(architecture),
                "Model": str(model),
                "ModelDisplay": MODEL_DISPLAY_NAMES.get(
                    str(model), str(model)
                ),
                "CommonSeasons": json.dumps(observed_seasons),
                "ComplexityRank": (
                    MODEL_SPECS[str(model)].complexity_rank
                    if str(model) in MODEL_SPECS
                    else 20
                    if str(model) == "constrained_ensemble"
                    else 21
                ),
                **metrics,
            }
        )

common_window_leaderboard = pd.DataFrame(common_records).sort_values(
    ["Gender", "MacroSeasonBrier", "ComplexityRank"]
).reset_index(drop=True)
assert not common_window_leaderboard.empty

common_window_leaderboard.to_csv(
    MODEL_REPORTS / "matched_season_model_comparison.csv",
    index=False,
)

requested_models = {
    "elastic_logistic",
    "xgb_classifier",
    "lgb_classifier",
    "torch_mlp",
    "tensorflow_mlp",
}
coverage_records = []
for gender in ("M", "W"):
    for architecture in ("separate_gender", "pooled_common"):
        available = set(
            common_window_leaderboard.loc[
                common_window_leaderboard["Gender"].eq(gender)
                & common_window_leaderboard["Architecture"].eq(architecture),
                "Model",
            ]
        )
        coverage_records.append(
            {
                "Gender": gender,
                "Architecture": architecture,
                "RequestedModels": sorted(requested_models),
                "CompletedModels": sorted(
                    requested_models.intersection(available)
                ),
                "MissingModels": sorted(
                    requested_models.difference(available)
                ),
                "Complete": requested_models.issubset(available),
            }
        )
framework_coverage = pd.DataFrame(coverage_records)
framework_coverage.to_csv(
    MODEL_REPORTS / "requested_framework_coverage.csv",
    index=False,
)
assert framework_coverage["Complete"].all(), framework_coverage

display(
    common_window_leaderboard[
        [
            "Gender",
            "Architecture",
            "ModelDisplay",
            "MacroSeasonBrier",
            "GameWeightedBrier",
            "ROCAUC",
            "AveragePrecision",
            "Precision",
            "Recall",
            "F1",
            "CalibrationSlope",
            "ECE",
        ]
    ].head(60)
)
framework_coverage

In [ ]:
def reliability_table(
    frame: pd.DataFrame,
    *,
    bins: int = 10,
) -> pd.DataFrame:
    """Build reliability groups, including one group for constants."""
    probability = clip_probability(
        frame["Prediction"]
    )
    outcome = frame[
        "Team1Win"
    ].to_numpy(dtype=float)

    if len(probability) == 0:
        return pd.DataFrame(
            columns=[
                "Rows",
                "MeanPrediction",
                "ObservedRate",
            ]
        )

    unique_probability = np.unique(
        probability
    )
    if unique_probability.size == 1:
        return pd.DataFrame(
            {
                "Rows": [int(len(frame))],
                "MeanPrediction": [
                    float(
                        unique_probability[0]
                    )
                ],
                "ObservedRate": [
                    float(np.mean(outcome))
                ],
            }
        )

    bin_count = min(
        int(bins),
        max(2, int(unique_probability.size)),
    )
    try:
        labels = pd.qcut(
            pd.Series(probability),
            q=bin_count,
            duplicates="drop",
        )
    except Exception:
        labels = pd.cut(
            pd.Series(probability),
            bins=bin_count,
            duplicates="drop",
        )

    output = pd.DataFrame(
        {
            "Prediction": probability,
            "Outcome": outcome,
            "Bin": labels,
        }
    )
    grouped = (
        output.groupby(
            "Bin",
            observed=True,
        )
        .agg(
            Rows=("Outcome", "size"),
            MeanPrediction=(
                "Prediction",
                "mean",
            ),
            ObservedRate=(
                "Outcome",
                "mean",
            ),
        )
        .reset_index(drop=True)
    )

    if grouped.empty:
        return pd.DataFrame(
            {
                "Rows": [int(len(frame))],
                "MeanPrediction": [
                    float(
                        np.mean(probability)
                    )
                ],
                "ObservedRate": [
                    float(np.mean(outcome))
                ],
            }
        )
    return grouped

PLOT_ARTIFACTS: list[str] = []

for gender in ("M", "W"):
    ranking = common_window_leaderboard.loc[
        common_window_leaderboard["Gender"].eq(gender)
    ].head(14).copy()
    ranking["Label"] = (
        ranking["Architecture"].str.replace("_", " ").str.title()
        + " — "
        + ranking["ModelDisplay"]
    )

    fig = px.bar(
        ranking.sort_values("MacroSeasonBrier", ascending=True),
        x="MacroSeasonBrier",
        y="Label",
        orientation="h",
        error_x="StandardError" if "StandardError" in ranking.columns else None,
        hover_data={
            "GameWeightedBrier": ":.4f",
            "ROCAUC": ":.4f",
            "AveragePrecision": ":.4f",
            "F1": ":.4f",
            "CalibrationSlope": ":.3f",
            "ECE": ":.4f",
        },
        title=f"{gender}: matched-season probability leaderboard",
        labels={"MacroSeasonBrier": "Mean held-out season Brier", "Label": ""},
    )
    fig.update_yaxes(categoryorder="total descending")
    save_plotly(fig, f"leaderboard_{gender}")
    PLOT_ARTIFACTS.append(f"leaderboard_{gender}.html")

    heatmap_data = ranking.pivot_table(
        index="ModelDisplay",
        columns="Architecture",
        values="MacroSeasonBrier",
        aggfunc="mean",
    )
    fig = px.imshow(
        heatmap_data,
        text_auto=".4f",
        aspect="auto",
        title=f"{gender}: architecture-by-model Brier comparison",
        labels={
            "x": "Architecture",
            "y": "Model",
            "color": "Brier",
        },
    )
    save_plotly(fig, f"architecture_model_heatmap_{gender}")
    PLOT_ARTIFACTS.append(
        f"architecture_model_heatmap_{gender}.html"
    )

    top_keys = list(
        ranking.head(6)[["Architecture", "Model"]]
        .itertuples(index=False, name=None)
    )
    season_plot = season_metrics.loc[
        season_metrics["EvaluationGender"].eq(gender)
        & season_metrics["Universe"].eq("rich")
    ].copy()
    season_plot["Key"] = list(
        zip(season_plot["Architecture"], season_plot["Model"])
    )
    season_plot = season_plot.loc[
        season_plot["Key"].isin(top_keys)
        & season_plot["Season"].isin(MATCHED_VALIDATION_SEASONS)
    ]
    season_plot["Series"] = (
        season_plot["Architecture"].str.replace("_", " ").str.title()
        + " — "
        + season_plot["Model"].map(MODEL_DISPLAY_NAMES).fillna(
            season_plot["Model"]
        )
    )
    fig = px.line(
        season_plot,
        x="Season",
        y="Brier",
        color="Series",
        markers=True,
        title=f"{gender}: Brier score by held-out tournament season",
        labels={"Brier": "Brier score", "Series": "Model"},
    )
    save_plotly(fig, f"brier_by_season_{gender}")
    PLOT_ARTIFACTS.append(f"brier_by_season_{gender}.html")

    curve_rows = []
    reliability_rows = []
    for architecture, model in top_keys[:5]:
        subset = comparison_rows.loc[
            comparison_rows["Gender"].eq(gender)
            & comparison_rows["Architecture"].eq(architecture)
            & comparison_rows["Model"].eq(model)
        ]
        if subset.empty or subset["Team1Win"].nunique() < 2:
            continue
        y = subset["Team1Win"].to_numpy(dtype=int)
        p = clip_probability(subset["Prediction"])
        fpr, tpr, _ = roc_curve(y, p)
        precision_values, recall_values, _ = precision_recall_curve(y, p)
        label = (
            architecture.replace("_", " ").title()
            + " — "
            + MODEL_DISPLAY_NAMES.get(model, model)
        )
        curve_rows.append(
            pd.DataFrame(
                {
                    "X": fpr,
                    "Y": tpr,
                    "Curve": "ROC",
                    "Series": label,
                }
            )
        )
        curve_rows.append(
            pd.DataFrame(
                {
                    "X": recall_values,
                    "Y": precision_values,
                    "Curve": "Precision–Recall",
                    "Series": label,
                }
            )
        )
        reliability = reliability_table(subset)
        reliability["Series"] = label
        reliability_rows.append(reliability)

    if curve_rows:
        curves = pd.concat(curve_rows, ignore_index=True)
        fig = px.line(
            curves,
            x="X",
            y="Y",
            color="Series",
            facet_col="Curve",
            facet_col_spacing=0.08,
            title=f"{gender}: ROC and precision–recall curves",
            labels={"X": "False-positive rate / recall", "Y": "True-positive rate / precision"},
        )
        fig.add_shape(
            type="line",
            x0=0,
            y0=0,
            x1=1,
            y1=1,
            line=dict(dash="dash"),
            row=1,
            col=1,
        )
        save_plotly(fig, f"roc_pr_curves_{gender}")
        PLOT_ARTIFACTS.append(f"roc_pr_curves_{gender}.html")

    if reliability_rows:
        reliability_plot = pd.concat(
            reliability_rows, ignore_index=True
        )
        fig = px.line(
            reliability_plot,
            x="MeanPrediction",
            y="ObservedRate",
            color="Series",
            markers=True,
            hover_data={"Rows": True},
            title=f"{gender}: out-of-fold calibration",
            labels={
                "MeanPrediction": "Mean predicted probability",
                "ObservedRate": "Observed win rate",
            },
        )
        fig.add_shape(
            type="line",
            x0=0,
            y0=0,
            x1=1,
            y1=1,
            line=dict(dash="dash"),
        )
        fig.update_xaxes(range=[0, 1])
        fig.update_yaxes(range=[0, 1])
        save_plotly(fig, f"reliability_{gender}")
        PLOT_ARTIFACTS.append(f"reliability_{gender}.html")

log_event(
    "plotly_scoreboards_created",
    artifacts=PLOT_ARTIFACTS,
)
print("Interactive Plotly reports:", len(PLOT_ARTIFACTS))

## 14. Feature ablation

Ablation measures the incremental value of feature groups while holding the folds and reference models constant. It is not ordinary feature importance: the model is retrained after each feature-block change.

In [ ]:
ABLATION_LADDER: list[dict[str, Any]] = [
    {
        "Stage": 0,
        "Experiment": "constant_probability",
        "Blocks": [],
    },
    {
        "Stage": 1,
        "Experiment": "seed_only",
        "Blocks": [
            "selection_committee_prior",
            "prequential_seed_priors",
        ],
    },
    {
        "Stage": 2,
        "Experiment": "compact_basic",
        "Blocks": [
            "selection_committee_prior",
            "prequential_seed_priors",
            "compact_performance",
            "availability_and_context",
        ],
    },
    {
        "Stage": 3,
        "Experiment": "compact_plus_ratings",
        "Blocks": [
            "selection_committee_prior",
            "prequential_seed_priors",
            "compact_performance",
            "availability_and_context",
            "dynamic_ratings",
            "global_strength_ratings",
            "within_season_normalization",
        ],
    },
    {
        "Stage": 4,
        "Experiment": "schedule_context",
        "Blocks": [
            "selection_committee_prior",
            "prequential_seed_priors",
            "compact_performance",
            "availability_and_context",
            "dynamic_ratings",
            "global_strength_ratings",
            "within_season_normalization",
            "schedule_and_accomplishment",
            "conference_context",
        ],
    },
    {
        "Stage": 5,
        "Experiment": "rich_efficiency",
        "Blocks": [
            "selection_committee_prior",
            "prequential_seed_priors",
            "compact_performance",
            "availability_and_context",
            "dynamic_ratings",
            "global_strength_ratings",
            "within_season_normalization",
            "schedule_and_accomplishment",
            "conference_context",
            "detailed_efficiency",
            "opponent_adjusted_efficiency",
        ],
    },
    {
        "Stage": 6,
        "Experiment": "history_context",
        "Blocks": [
            "selection_committee_prior",
            "prequential_seed_priors",
            "compact_performance",
            "availability_and_context",
            "dynamic_ratings",
            "global_strength_ratings",
            "within_season_normalization",
            "schedule_and_accomplishment",
            "conference_context",
            "detailed_efficiency",
            "opponent_adjusted_efficiency",
            "prior_program_history",
            "prior_coach_history",
        ],
    },
    {
        "Stage": 7,
        "Experiment": "men_massey",
        "Blocks": [
            "selection_committee_prior",
            "prequential_seed_priors",
            "compact_performance",
            "availability_and_context",
            "dynamic_ratings",
            "global_strength_ratings",
            "within_season_normalization",
            "schedule_and_accomplishment",
            "conference_context",
            "detailed_efficiency",
            "opponent_adjusted_efficiency",
            "prior_program_history",
            "prior_coach_history",
            "massey_consensus",
        ],
    },
    {
        "Stage": 8,
        "Experiment": "matchup_interactions",
        "Blocks": [
            "selection_committee_prior",
            "prequential_seed_priors",
            "compact_performance",
            "availability_and_context",
            "dynamic_ratings",
            "global_strength_ratings",
            "within_season_normalization",
            "schedule_and_accomplishment",
            "conference_context",
            "detailed_efficiency",
            "opponent_adjusted_efficiency",
            "prior_program_history",
            "prior_coach_history",
            "massey_consensus",
            "matchup_interactions",
        ],
    },
    {
        "Stage": 9,
        "Experiment": "full_candidate_bank",
        "Blocks": sorted(set(FEATURE_TO_BLOCK.values())),
    },
]


def candidates_for_blocks(
    candidates: Sequence[str],
    blocks: Sequence[str],
) -> list[str]:
    block_set = set(blocks)
    return [
        feature
        for feature in candidates
        if FEATURE_TO_BLOCK.get(feature, "other") in block_set
    ]


def run_ablation_fold(
    outer: pd.Series,
    experiment: dict[str, Any],
    *,
    model_name: str,
) -> pd.DataFrame:
    outer_fold_id = str(outer["OuterFoldID"])
    experiment_name = str(experiment["Experiment"])
    cache_name = f"ablation__{experiment_name}__{model_name}"
    directory = outer_model_directory(outer_fold_id, cache_name)
    if checkpoint_is_valid(directory):
        return pd.read_parquet(
            directory / "outer_predictions.parquet"
        )

    ensure_memory_headroom(
        f"ablation {outer_fold_id} {experiment_name} {model_name}",
        model_name=model_name,
    )

    gender = str(outer["Gender"])
    universe = str(outer["Universe"])
    pooled = gender == "Pooled"
    train_seasons = parse_json_int_list(
        outer["TrainingSeasonsJSON"]
    )
    validation_season = int(outer["ValidationSeason"])
    training = rows_for_context(
        development,
        seasons=train_seasons,
        gender=gender,
        universe=universe,
    )
    validation = rows_for_context(
        development,
        seasons=[validation_season],
        gender=gender,
        universe=universe,
    )

    if experiment_name == "constant_probability":
        probability = np.full(len(validation), 0.5)
        output = make_prediction_frame(
            validation,
            outer_fold_id=outer_fold_id,
            inner_fold_id=None,
            architecture=str(outer["Architecture"]),
            universe=universe,
            model_name=cache_name,
            raw_prediction=probability,
            feature_count=0,
            best_iteration=None,
        )
        output["Prediction"] = probability
        output["CalibrationMethod"] = "identity"
    else:
        all_candidates = candidates_for_context(
            gender=gender,
            universe=universe,
            seed_aware=True,
        )
        block_candidates = candidates_for_blocks(
            all_candidates,
            experiment["Blocks"],
        )
        assert block_candidates, (
            f"No candidates for ablation {experiment_name}"
        )
        spec = MODEL_SPECS[model_name]
        params = default_parameters(spec)

        # Build inner OOF manually with the block-restricted bank.
        inner_oof, best_iterations, _ = run_inner_oof(
            outer=outer,
            spec=spec,
            params=params,
            candidates=block_candidates,
            collect_selector_audits=False,
        )
        calibration = select_cross_fitted_calibrator(
            inner_oof,
            raw_kind=spec.raw_kind,
        )
        features, selection = model_feature_list(
            spec,
            training,
            block_candidates,
            pooled=pooled,
            selection_key=(
                f"{outer_fold_id}__ablation__{experiment_name}"
                f"__{model_name}"
            ),
        )
        fixed_iterations = (
            max(10, int(round(float(np.median(best_iterations)))))
            if best_iterations
            else None
        )
        fitted, raw = fit_predictor(
            spec,
            params,
            training,
            validation,
            features,
            fixed_iterations=fixed_iterations,
            random_seed=SEED + validation_season,
        )
        output = make_prediction_frame(
            validation,
            outer_fold_id=outer_fold_id,
            inner_fold_id=None,
            architecture=str(outer["Architecture"]),
            universe=universe,
            model_name=cache_name,
            raw_prediction=raw,
            feature_count=len(features),
            best_iteration=fitted.best_iteration,
        )
        output["Prediction"] = (
            calibration.fitted_calibrator.predict(raw)
        )
        output["CalibrationMethod"] = calibration.method

    metadata = {
        "status": "complete",
        "contracts": EXPECTED_CONTRACTS,
        "run_mode": RUN_MODE,
        "outer_fold_id": outer_fold_id,
        "model": cache_name,
        "outer_feature_count": int(output["FeatureCount"].iloc[0]),
        "outer_metrics": evaluate_probability_frame(output)[0],
        "calibration_method": str(output["CalibrationMethod"].iloc[0]),
    }
    directory.mkdir(parents=True, exist_ok=True)
    atomic_write_parquet(
        directory / "outer_predictions.parquet", output
    )
    # A placeholder inner artifact keeps checkpoint validation uniform.
    atomic_write_parquet(
        directory / "inner_oof.parquet",
        output.iloc[0:0].copy(),
    )
    atomic_write_json(directory / "metadata.json", metadata)
    return output


ABLATION_PREDICTIONS: list[pd.DataFrame] = []
if MODE["run_ablation"]:
    ablation_models = (
        ["elastic_logistic", "xgb_classifier"]
        if RUN_PROFILE == "full"
        else ["elastic_logistic"]
    )
    rich_outer_plan = outer_plan.loc[
        outer_plan["Universe"].eq("rich")
        & outer_plan["Architecture"].eq("separate_gender")
    ].copy()

    for _, outer in rich_outer_plan.iterrows():
        for experiment in ABLATION_LADDER:
            for model_name in ablation_models:
                try:
                    print(
                        "Ablation:",
                        outer["OuterFoldID"],
                        experiment["Experiment"],
                        model_name,
                    )
                    result = run_ablation_fold(
                        outer,
                        experiment,
                        model_name=model_name,
                    )
                    result["AblationExperiment"] = experiment[
                        "Experiment"
                    ]
                    result["AblationStage"] = int(
                        experiment["Stage"]
                    )
                    result["AblationBaseModel"] = model_name
                    ABLATION_PREDICTIONS.append(result)
                except MemoryPressurePause as exc:
                    pause_message = str(exc)
                    print("MEMORY PRESSURE PAUSE:\n", pause_message)
                    log_event(
                        "ablation_paused_for_memory",
                        outer_fold_id=str(outer["OuterFoldID"]),
                        experiment=str(experiment["Experiment"]),
                        model=model_name,
                        message=pause_message,
                    )
                    raise
                except Exception as exc:
                    failure_record(
                        outer=outer,
                        model_name=(
                            f"ablation__{experiment['Experiment']}"
                            f"__{model_name}"
                        ),
                        exc=exc,
                    )
                    print("ABLATION FAILED:", repr(exc))
                finally:
                    release_runtime_memory(
                        aggressive=(
                            psutil.virtual_memory().available / 1024**3
                            < SOFT_WARNING_FLOOR_GB
                        )
                    )

if ABLATION_PREDICTIONS:
    ablation_oof = pd.concat(
        ABLATION_PREDICTIONS, ignore_index=True
    )
    atomic_write_parquet(
        PROCESSED
        / "development_feature_ablation_oof.parquet",
        ablation_oof,
    )
    ablation_records = []
    for (
        experiment,
        stage,
        base_model,
        gender,
        architecture,
    ), group in ablation_oof.groupby(
        [
            "AblationExperiment",
            "AblationStage",
            "AblationBaseModel",
            "Gender",
            "Architecture",
        ],
        observed=True,
    ):
        metrics, _ = evaluate_probability_frame(group)
        ablation_records.append(
            {
                "Experiment": experiment,
                "Stage": int(stage),
                "BaseModel": base_model,
                "Gender": gender,
                "Architecture": architecture,
                **metrics,
            }
        )
    ablation_leaderboard = pd.DataFrame(
        ablation_records
    ).sort_values(
        ["Gender", "Architecture", "BaseModel", "Stage"]
    )
else:
    ablation_oof = pd.DataFrame()
    ablation_leaderboard = pd.DataFrame()

ablation_leaderboard.to_csv(
    MODEL_REPORTS / "feature_block_ablation.csv",
    index=False,
)

if not ablation_leaderboard.empty:
    ablation_leaderboard["Series"] = (
        ablation_leaderboard["Gender"].astype(str)
        + " — "
        + ablation_leaderboard["BaseModel"].map(
            MODEL_DISPLAY_NAMES
        ).fillna(ablation_leaderboard["BaseModel"])
    )
    baseline_map = (
        ablation_leaderboard.loc[
            ablation_leaderboard["Experiment"].eq("seed_only")
        ]
        .set_index(["Gender", "Architecture", "BaseModel"])[
            "MacroSeasonBrier"
        ]
        .to_dict()
    )
    ablation_leaderboard["DeltaVsSeedOnly"] = [
        row.MacroSeasonBrier
        - baseline_map.get(
            (row.Gender, row.Architecture, row.BaseModel),
            np.nan,
        )
        for row in ablation_leaderboard.itertuples()
    ]

    fig = px.line(
        ablation_leaderboard,
        x="Stage",
        y="MacroSeasonBrier",
        color="Series",
        markers=True,
        hover_data=["Experiment", "DeltaVsSeedOnly", "ROCAUC", "AveragePrecision"],
        title="Feature-block ablation: held-out Brier by cumulative feature set",
        labels={
            "MacroSeasonBrier": "Mean held-out season Brier",
            "Stage": "Feature-block stage",
            "Series": "Tournament / reference model",
        },
    )
    save_plotly(fig, "feature_block_ablation")
    PLOT_ARTIFACTS.append("feature_block_ablation.html")

    ablation_delta = ablation_leaderboard[
        [
            "Gender",
            "Architecture",
            "BaseModel",
            "Stage",
            "Experiment",
            "MacroSeasonBrier",
            "DeltaVsSeedOnly",
        ]
    ].copy()
    ablation_delta.to_csv(
        MODEL_REPORTS / "feature_block_ablation_deltas.csv",
        index=False,
    )
    log_event(
        "ablation_complete",
        rows=int(len(ablation_leaderboard)),
        reference_models=sorted(
            ablation_leaderboard["BaseModel"].unique().tolist()
        ),
    )

ablation_leaderboard.head(40)


## 15. Dimensionality, stability, runtime, and failure diagnostics

These reports show how often each feature was selected, whether selections were stable across seasons, how long every model took, and how much memory it used.

In [ ]:
# Reconstruct selection records from disk so resumed runs are included.
selection_files = [
    path
    for path in (CACHE_DIR / "outer_models").glob(
        "*/*/selected_features.csv"
    )
    if checkpoint_is_valid(path.parent)
]
selection_frames = []
for path in selection_files:
    if "ablation__" in str(path.parent.name):
        continue
    try:
        frame = pd.read_csv(path)
        frame["SelectionFile"] = str(path)
        selection_frames.append(frame)
    except Exception:
        continue

selected_feature_audit = (
    pd.concat(selection_frames, ignore_index=True)
    if selection_frames
    else pd.DataFrame()
)

if not selected_feature_audit.empty:
    selected_only = selected_feature_audit.loc[
        selected_feature_audit["Selected"].astype(bool)
    ].copy()
    selected_only["Block"] = selected_only["Feature"].map(
        FEATURE_TO_BLOCK
    ).fillna(selected_only.get("Block", "other"))
    feature_stability = (
        selected_only.groupby(
            ["Model", "Feature", "Block"], observed=True
        )
        .agg(
            SelectionOccurrences=("OuterFoldID", "nunique"),
            MeanStabilityScore=("StabilityScore", "mean"),
            MedianSelectionRank=("SelectionRank", "median"),
            MeanMissingRate=("MissingRate", "mean"),
        )
        .reset_index()
    )
    model_fold_counts = (
        selected_only.groupby("Model", observed=True)[
            "OuterFoldID"
        ].nunique()
    )
    feature_stability["EligibleOuterFolds"] = (
        feature_stability["Model"].map(model_fold_counts)
    )
    feature_stability["SelectionFrequency"] = (
        feature_stability["SelectionOccurrences"]
        / feature_stability["EligibleOuterFolds"]
    )
    block_stability = (
        selected_only.groupby(["Model", "Block"], observed=True)
        .agg(
            SelectedFeatureOccurrences=("Feature", "size"),
            UniqueFeatures=("Feature", "nunique"),
            OuterFolds=("OuterFoldID", "nunique"),
        )
        .reset_index()
    )

    jaccard_records = []
    for model, model_rows in selected_only.groupby(
        "Model", observed=True
    ):
        feature_sets = {
            fold: set(group["Feature"])
            for fold, group in model_rows.groupby(
                "OuterFoldID", observed=True
            )
        }
        fold_order = (
            outer_plan.loc[
                outer_plan["OuterFoldID"].isin(feature_sets)
            ]
            .sort_values(
                ["Gender", "Architecture", "Universe", "ValidationSeason"]
            )
        )
        for (
            gender,
            architecture,
            universe,
        ), context in fold_order.groupby(
            ["Gender", "Architecture", "Universe"],
            observed=True,
        ):
            ordered_folds = context["OuterFoldID"].tolist()
            for previous, current in zip(
                ordered_folds[:-1],
                ordered_folds[1:],
                strict=True,
            ):
                a = feature_sets[previous]
                b = feature_sets[current]
                union = a | b
                jaccard_records.append(
                    {
                        "Model": model,
                        "Gender": gender,
                        "Architecture": architecture,
                        "Universe": universe,
                        "PreviousFold": previous,
                        "CurrentFold": current,
                        "Jaccard": (
                            len(a & b) / len(union)
                            if union
                            else np.nan
                        ),
                        "PreviousFeatures": len(a),
                        "CurrentFeatures": len(b),
                    }
                )
    selection_jaccard = pd.DataFrame(jaccard_records)
else:
    feature_stability = pd.DataFrame()
    block_stability = pd.DataFrame()
    selection_jaccard = pd.DataFrame()

selected_feature_audit.to_csv(
    MODEL_REPORTS / "selected_feature_audit.csv",
    index=False,
)
feature_stability.to_csv(
    MODEL_REPORTS / "feature_selection_stability.csv",
    index=False,
)
block_stability.to_csv(
    MODEL_REPORTS / "selected_feature_blocks.csv",
    index=False,
)
selection_jaccard.to_csv(
    MODEL_REPORTS / "feature_selection_jaccard.csv",
    index=False,
)

feature_count_audit = (
    development_oof.loc[
        development_oof["FeatureCount"].notna()
    ]
    .groupby(
        ["Architecture", "Universe", "Model", "Gender"],
        observed=True,
    )["FeatureCount"]
    .agg(["min", "median", "max"])
    .reset_index()
)
feature_count_audit.to_csv(
    MODEL_REPORTS / "feature_count_audit.csv",
    index=False,
)

# Standardized linear coefficients are aggregated across rolling folds.
# This diagnoses sign instability and one-season coefficient flips that a
# single fitted model would conceal.
coefficient_files = [
    path
    for path in (CACHE_DIR / "outer_models").glob(
        "*/*/linear_coefficients.csv"
    )
    if "ablation__" not in path.parent.name
    and checkpoint_is_valid(path.parent)
]
coefficient_frames = []
for path in coefficient_files:
    try:
        frame = pd.read_csv(path)
        frame["CoefficientFile"] = str(path)
        coefficient_frames.append(frame)
    except Exception:
        continue

linear_coefficients = (
    pd.concat(coefficient_frames, ignore_index=True)
    if coefficient_frames
    else pd.DataFrame()
)
if not linear_coefficients.empty:
    linear_coefficients["CoefficientSign"] = np.sign(
        linear_coefficients["StandardizedCoefficient"]
    )
    coefficient_stability = (
        linear_coefficients.groupby(
            [
                "Gender",
                "Architecture",
                "Universe",
                "Model",
                "Feature",
                "Block",
            ],
            observed=True,
        )
        .agg(
            OuterFolds=("OuterFoldID", "nunique"),
            MeanCoefficient=("StandardizedCoefficient", "mean"),
            MedianCoefficient=("StandardizedCoefficient", "median"),
            MeanAbsoluteCoefficient=(
                "StandardizedCoefficient",
                lambda values: float(np.mean(np.abs(values))),
            ),
            CoefficientStd=("StandardizedCoefficient", "std"),
            PositiveRate=(
                "StandardizedCoefficient",
                lambda values: float(np.mean(np.asarray(values) > 0)),
            ),
            NegativeRate=(
                "StandardizedCoefficient",
                lambda values: float(np.mean(np.asarray(values) < 0)),
            ),
        )
        .reset_index()
    )
    coefficient_stability["SignConsistency"] = np.maximum(
        coefficient_stability["PositiveRate"],
        coefficient_stability["NegativeRate"],
    )
    coefficient_stability = coefficient_stability.sort_values(
        [
            "Gender",
            "Architecture",
            "Universe",
            "Model",
            "MeanAbsoluteCoefficient",
        ],
        ascending=[True, True, True, True, False],
    )
    coefficient_block_stability = (
        coefficient_stability.groupby(
            ["Gender", "Architecture", "Universe", "Model", "Block"],
            observed=True,
        )
        .agg(
            UniqueFeatures=("Feature", "nunique"),
            MedianSignConsistency=("SignConsistency", "median"),
            MeanAbsoluteCoefficient=("MeanAbsoluteCoefficient", "mean"),
        )
        .reset_index()
    )
else:
    coefficient_stability = pd.DataFrame()
    coefficient_block_stability = pd.DataFrame()

linear_coefficients.to_csv(
    MODEL_REPORTS / "linear_coefficients_all.csv",
    index=False,
)
coefficient_stability.to_csv(
    MODEL_REPORTS / "linear_coefficient_stability.csv",
    index=False,
)
coefficient_block_stability.to_csv(
    MODEL_REPORTS / "linear_coefficient_block_stability.csv",
    index=False,
)

resource_log = pd.DataFrame(RESOURCE_LOG)
resource_log.to_csv(
    MODEL_REPORTS / "resource_usage.csv",
    index=False,
)
failure_frame = pd.DataFrame(MODEL_FAILURES)
failure_frame.to_csv(
    MODEL_REPORTS / "all_failures.csv",
    index=False,
)

print("Selection files:", len(selection_files))
print("Linear coefficient files:", len(coefficient_files))
print("Model failures:", len(failure_frame))
display(feature_count_audit.head(30))
if not selection_jaccard.empty:
    display(
        selection_jaccard.groupby("Model", observed=True)["Jaccard"]
        .agg(["count", "mean", "median", "min"])
        .reset_index()
    )
if not feature_stability.empty:
    feature_plot = (
        feature_stability.sort_values(
            ["Model", "SelectionFrequency", "MeanStabilityScore"],
            ascending=[True, False, False],
        )
        .groupby("Model", observed=True)
        .head(12)
        .copy()
    )
    feature_plot["ModelDisplay"] = feature_plot["Model"].map(
        MODEL_DISPLAY_NAMES
    ).fillna(feature_plot["Model"])
    fig = px.bar(
        feature_plot,
        x="SelectionFrequency",
        y="Feature",
        color="Block",
        facet_col="ModelDisplay",
        facet_col_wrap=2,
        orientation="h",
        hover_data=[
            "SelectionOccurrences",
            "EligibleOuterFolds",
            "MeanStabilityScore",
            "MedianSelectionRank",
        ],
        title="Most consistently selected features across chronological folds",
        labels={"SelectionFrequency": "Selection frequency"},
        height=max(650, 300 * math.ceil(feature_plot["ModelDisplay"].nunique() / 2)),
    )
    fig.update_yaxes(matches=None, showticklabels=True)
    save_plotly(fig, "feature_selection_frequency")
    PLOT_ARTIFACTS.append("feature_selection_frequency.html")

if not resource_log.empty:
    resource_plot = resource_log.copy()
    resource_plot["ModelDisplay"] = resource_plot["Model"].map(
        MODEL_DISPLAY_NAMES
    ).fillna(resource_plot["Model"])
    fig = px.scatter(
        resource_plot,
        x="ElapsedSeconds",
        y="EndRSSMB",
        color="ModelDisplay",
        symbol="Stage",
        hover_data=["OuterFoldID", "StartRSSMB"],
        title="Runtime and memory use by model task",
        labels={
            "ElapsedSeconds": "Elapsed seconds",
            "EndRSSMB": "Process memory after task (MB)",
            "ModelDisplay": "Model",
        },
    )
    save_plotly(fig, "runtime_memory")
    PLOT_ARTIFACTS.append("runtime_memory.html")

log_event(
    "stability_and_resource_diagnostics_complete",
    selection_files=int(len(selection_files)),
    coefficient_files=int(len(coefficient_files)),
    failures=int(len(failure_frame)),
)


## 16. Error analysis and paired uncertainty

Errors are evaluated by season, tournament timing, seed gap, probability confidence, and model pair. Uncertainty intervals resample whole seasons rather than treating games as independent.

In [ ]:
development_metadata = development[
    [
        column
        for column in (
            "TargetKey",
            "DayNum",
            "Team1ID",
            "Team2ID",
            "matchup__seed_gap_abs",
            "matchup__seed_diff",
        )
        if column in development.columns
    ]
].drop_duplicates("TargetKey")

oof_with_context = development_oof.merge(
    development_metadata,
    on=[
        column
        for column in ["TargetKey"]
        if column in development_metadata.columns
    ],
    how="left",
    validate="many_to_one",
    suffixes=("", "_source"),
)
oof_with_context["SquaredError"] = (
    oof_with_context["Prediction"]
    - oof_with_context["Team1Win"]
) ** 2
oof_with_context["Confidence"] = np.maximum(
    oof_with_context["Prediction"],
    1.0 - oof_with_context["Prediction"],
)
oof_with_context["WrongAtHalf"] = (
    (oof_with_context["Prediction"] >= 0.5).astype(int)
    != oof_with_context["Team1Win"].astype(int)
)
oof_with_context["ConfidentWrong"] = (
    oof_with_context["WrongAtHalf"]
    & oof_with_context["Confidence"].ge(0.80)
)

if "matchup__seed_gap_abs" in oof_with_context.columns:
    oof_with_context["SeedGapBand"] = pd.cut(
        oof_with_context["matchup__seed_gap_abs"],
        bins=[-np.inf, 0, 2, 5, 8, 12, np.inf],
        labels=["0", "1-2", "3-5", "6-8", "9-12", "13+"],
    )
else:
    oof_with_context["SeedGapBand"] = "unavailable"

if "DayNum" in oof_with_context.columns:
    oof_with_context["TournamentDayBand"] = pd.cut(
        oof_with_context["DayNum"],
        bins=[-np.inf, 135, 139, 146, 152, np.inf],
        labels=[
            "play_in",
            "rounds_1_2",
            "rounds_3_4",
            "semifinal",
            "final_or_other",
        ],
    )
else:
    oof_with_context["TournamentDayBand"] = "unavailable"

error_slice_records = []
for (
    architecture,
    universe,
    model,
    gender,
    seed_band,
), group in oof_with_context.groupby(
    [
        "Architecture",
        "Universe",
        "Model",
        "Gender",
        "SeedGapBand",
    ],
    observed=True,
):
    error_slice_records.append(
        {
            "Architecture": architecture,
            "Universe": universe,
            "Model": model,
            "Gender": gender,
            "SeedGapBand": str(seed_band),
            "Rows": len(group),
            "Brier": float(group["SquaredError"].mean()),
            "ConfidentWrongRate": float(
                group["ConfidentWrong"].mean()
            ),
            "MeanConfidence": float(group["Confidence"].mean()),
        }
    )
error_slices = pd.DataFrame(error_slice_records)
error_slices.to_csv(
    MODEL_REPORTS / f"error_slices_seed_gap_{RUN_MODE}.csv",
    index=False,
)

round_slice_records = []
for (
    architecture,
    universe,
    model,
    gender,
    round_band,
), group in oof_with_context.groupby(
    [
        "Architecture",
        "Universe",
        "Model",
        "Gender",
        "TournamentDayBand",
    ],
    observed=True,
):
    round_slice_records.append(
        {
            "Architecture": architecture,
            "Universe": universe,
            "Model": model,
            "Gender": gender,
            "TournamentDayBand": str(round_band),
            "Rows": len(group),
            "Brier": float(group["SquaredError"].mean()),
            "ConfidentWrongRate": float(
                group["ConfidentWrong"].mean()
            ),
            "MeanConfidence": float(group["Confidence"].mean()),
        }
    )
round_error_slices = pd.DataFrame(round_slice_records)
round_error_slices.to_csv(
    MODEL_REPORTS / f"error_slices_tournament_round_{RUN_MODE}.csv",
    index=False,
)

worst_errors = (
    oof_with_context.sort_values(
        ["SquaredError", "Confidence"],
        ascending=False,
    )
    .head(250)
)
worst_errors.to_csv(
    MODEL_REPORTS / f"largest_oof_errors_{RUN_MODE}.csv",
    index=False,
)

print(
    "Confident wrong predictions:",
    int(oof_with_context["ConfidentWrong"].sum()),
)
worst_errors[
    [
        "Season",
        "Gender",
        "Architecture",
        "Model",
        "Team1ID",
        "Team2ID",
        "Team1Win",
        "Prediction",
        "SquaredError",
    ]
].head(20)


## 17. Winner adjudication and frozen development recipe

The notebook reports the numerical winner and the simpler one-standard-error recommendation separately. The recipe is written only after all required frameworks, architectures, ablations, plots, and explanation tasks pass.

In [ ]:
def stream_complexity(architecture: str, model: str) -> int:
    if model in MODEL_SPECS:
        return int(MODEL_SPECS[model].complexity_rank)
    if model == "constrained_ensemble":
        return 20
    if model == "partial_pooling_ensemble":
        return 21
    return 99


def row_to_winner_dict(row: pd.Series) -> dict[str, Any]:
    return {
        "architecture": str(row["Architecture"]),
        "model": str(row["Model"]),
        "model_display": str(row["ModelDisplay"]),
        "macro_season_brier": float(row["MacroSeasonBrier"]),
        "game_weighted_brier": float(row["GameWeightedBrier"]),
        "roc_auc": float(row["ROCAUC"]),
        "average_precision": float(row["AveragePrecision"]),
        "precision": float(row["Precision"]),
        "recall": float(row["Recall"]),
        "f1": float(row["F1"]),
        "calibration_slope": float(row["CalibrationSlope"]),
        "ece": float(row["ECE"]),
        "complexity_rank": int(row["ComplexityRank"]),
    }


def select_winners_for_gender(
    frame: pd.DataFrame,
    *,
    gender: str,
) -> tuple[dict[str, Any], pd.DataFrame]:
    candidates = frame.loc[
        frame["Gender"].eq(gender)
    ].copy()
    assert not candidates.empty
    candidates["StandardError"] = (
        candidates["SeasonBrierStd"]
        / np.sqrt(candidates["Seasons"].clip(lower=1))
    )
    candidates = candidates.sort_values(
        ["MacroSeasonBrier", "ComplexityRank"]
    ).reset_index(drop=True)

    raw_best = candidates.iloc[0]
    one_se_limit = float(
        raw_best["MacroSeasonBrier"] + raw_best["StandardError"]
    )
    eligible = candidates.loc[
        candidates["MacroSeasonBrier"] <= one_se_limit + 1e-12
    ].sort_values(["ComplexityRank", "MacroSeasonBrier"])
    recommended = eligible.iloc[0]

    separate_candidates = candidates.loc[
        candidates["Architecture"].eq("separate_gender")
    ]
    pooled_candidates = candidates.loc[
        candidates["Architecture"].eq("pooled_common")
    ]
    best_separate = (
        separate_candidates.iloc[0]
        if not separate_candidates.empty
        else None
    )
    best_pooled = (
        pooled_candidates.iloc[0]
        if not pooled_candidates.empty
        else None
    )

    candidates["OneSELimit"] = one_se_limit
    candidates["WithinOneSE"] = (
        candidates["MacroSeasonBrier"] <= one_se_limit + 1e-12
    )
    candidates["RawBest"] = (
        candidates["Architecture"].eq(raw_best["Architecture"])
        & candidates["Model"].eq(raw_best["Model"])
    )
    candidates["Recommended"] = (
        candidates["Architecture"].eq(recommended["Architecture"])
        & candidates["Model"].eq(recommended["Model"])
    )

    selection = {
        "gender": gender,
        "comparison_seasons": MATCHED_VALIDATION_SEASONS,
        "raw_best": row_to_winner_dict(raw_best),
        "recommended": row_to_winner_dict(recommended),
        "best_separate": (
            row_to_winner_dict(best_separate)
            if best_separate is not None
            else None
        ),
        "best_pooled": (
            row_to_winner_dict(best_pooled)
            if best_pooled is not None
            else None
        ),
        # Compatibility fields used by paired comparisons and the final recipe.
        "architecture": str(recommended["Architecture"]),
        "model": str(recommended["Model"]),
        "macro_season_brier": float(
            recommended["MacroSeasonBrier"]
        ),
        "one_se_threshold": one_se_limit,
        "complexity_rank": int(recommended["ComplexityRank"]),
    }
    return selection, candidates


RECIPE_SELECTIONS: dict[str, dict[str, Any]] = {}
candidate_tables = []
for gender in ("M", "W"):
    selection, candidates = select_winners_for_gender(
        common_window_leaderboard,
        gender=gender,
    )
    RECIPE_SELECTIONS[gender] = selection
    candidate_tables.append(candidates)

winner_candidates = pd.concat(
    candidate_tables, ignore_index=True
)
winner_candidates.to_csv(
    MODEL_REPORTS / "winner_candidates.csv",
    index=False,
)

winner_rows = []
for gender, selection in RECIPE_SELECTIONS.items():
    for label, key in (
        ("Lowest-score winner", "raw_best"),
        ("Recommended winner", "recommended"),
        ("Best separate model", "best_separate"),
        ("Best pooled model", "best_pooled"),
    ):
        details = selection.get(key)
        if details is None:
            continue
        winner_rows.append(
            {
                "Gender": gender,
                "Decision": label,
                **details,
            }
        )
winner_table = pd.DataFrame(winner_rows)
winner_table.to_csv(
    MODEL_REPORTS / "winner_summary.csv",
    index=False,
)

WINNER_SUMMARY = {
    "selection_metric": "macro mean held-out season Brier",
    "comparison_seasons": MATCHED_VALIDATION_SEASONS,
    "men": RECIPE_SELECTIONS["M"],
    "women": RECIPE_SELECTIONS["W"],
    "interpretation": {
        "raw_best": "lowest numerical Brier score",
        "recommended": (
            "least complex candidate within one standard error "
            "of the lowest numerical Brier score"
        ),
    },
}
(MODEL_REPORTS / "winner_summary.json").write_text(
    json.dumps(WINNER_SUMMARY, indent=2),
    encoding="utf-8",
)

for gender, selection in RECIPE_SELECTIONS.items():
    raw = selection["raw_best"]
    recommended = selection["recommended"]
    print(
        f"\n{gender} — lowest-score winner: "
        f"{raw['architecture']} / {raw['model_display']} "
        f"(Brier {raw['macro_season_brier']:.4f})"
    )
    print(
        f"{gender} — recommended winner: "
        f"{recommended['architecture']} / "
        f"{recommended['model_display']} "
        f"(Brier {recommended['macro_season_brier']:.4f})"
    )

fig = px.bar(
    winner_table,
    x="macro_season_brier",
    y="Decision",
    color="model_display",
    facet_col="Gender",
    orientation="h",
    hover_data=[
        "architecture",
        "roc_auc",
        "average_precision",
        "precision",
        "recall",
        "f1",
        "calibration_slope",
        "ece",
    ],
    title="Model winner summary",
    labels={
        "macro_season_brier": "Mean held-out season Brier",
        "model_display": "Model",
        "Decision": "",
    },
)
save_plotly(fig, "winner_summary")
PLOT_ARTIFACTS.append("winner_summary.html")

# Confusion matrices for the two recommended streams.
for gender, selection in RECIPE_SELECTIONS.items():
    subset = comparison_rows.loc[
        comparison_rows["Gender"].eq(gender)
        & comparison_rows["Architecture"].eq(
            selection["architecture"]
        )
        & comparison_rows["Model"].eq(selection["model"])
    ].copy()
    y = subset["Team1Win"].to_numpy(dtype=int)
    predicted = (
        clip_probability(subset["Prediction"])
        >= CLASSIFICATION_THRESHOLD
    ).astype(int)
    matrix = confusion_matrix(y, predicted, labels=[0, 1])
    fig = px.imshow(
        matrix,
        text_auto=True,
        x=["Predicted 0", "Predicted 1"],
        y=["Actual 0", "Actual 1"],
        title=(
            f"{gender}: recommended model confusion matrix "
            f"at threshold {CLASSIFICATION_THRESHOLD:.2f}"
        ),
        labels={"color": "Games"},
    )
    fig.update_yaxes(autorange="reversed")
    save_plotly(fig, f"confusion_matrix_recommended_{gender}")
    PLOT_ARTIFACTS.append(
        f"confusion_matrix_recommended_{gender}.html"
    )

MODEL_RECIPE = {
    "recipe_version": 1,
    "status": "pending_final_integrity_gate",
    "created_from_run_profile": RUN_PROFILE,
    "source_contracts": EXPECTED_CONTRACTS,
    "locked_benchmark_evaluated": False,
    "locked_benchmark_seasons": sorted(LOCKED_SEASONS),
    "matched_validation_seasons": MATCHED_VALIDATION_SEASONS,
    "primary_selections": {
        "men": RECIPE_SELECTIONS["M"],
        "women": RECIPE_SELECTIONS["W"],
    },
    "training_procedure": {
        "feature_selector": "block_aware_season_stability",
        "hyperparameter_search": "bounded_nested_optuna",
        "boosting_rounds": "median_inner_best_iteration",
        "neural_epochs": "median_inner_best_epoch",
        "calibration": "cross_fitted_by_inner_season",
        "ensemble": "nonnegative_simplex_with_redundancy_pruning",
        "separate_and_pooled_comparison": True,
        "pytorch_and_tensorflow_compared": True,
    },
    "resource_guards": {
        "linear_feature_cap": MODE["linear_feature_cap"],
        "tree_feature_cap": MODE["tree_feature_cap"],
        "neural_feature_cap": MODE["neural_feature_cap"],
        "margin_feature_cap": MODE["margin_feature_cap"],
        "maximum_threads": MAX_THREADS,
        "stage2_loaded_during_selection": False,
    },
}
MODEL_RECIPE["recipe_sha256"] = object_sha256(MODEL_RECIPE)

log_event(
    "winner_adjudication_complete",
    men=RECIPE_SELECTIONS["M"],
    women=RECIPE_SELECTIONS["W"],
)
display(winner_table)

In [ ]:
# Paired comparisons of the chosen stream against the seed baseline
# and best standalone rich model on exactly matching games.
paired_records = []
for gender, selection in RECIPE_SELECTIONS.items():
    chosen = development_oof.loc[
        development_oof["Gender"].eq(gender)
        & development_oof["Universe"].eq("rich")
        & development_oof["Architecture"].eq(
            selection["architecture"]
        )
        & development_oof["Model"].eq(selection["model"])
        & development_oof["Season"].isin(
            selection["comparison_seasons"]
        )
    ][["TargetKey", "Season", "Team1Win", "Prediction"]].rename(
        columns={"Prediction": "ChosenPrediction"}
    )

    comparison_candidates = winner_candidates.loc[
        winner_candidates["Gender"].eq(gender)
        & winner_candidates["Model"].eq("seed_logistic")
    ].sort_values("MacroSeasonBrier")
    if not comparison_candidates.empty:
        seed_row = comparison_candidates.iloc[0]
        seed = development_oof.loc[
            development_oof["Gender"].eq(gender)
            & development_oof["Universe"].eq("rich")
            & development_oof["Architecture"].eq(
                seed_row["Architecture"]
            )
            & development_oof["Model"].eq("seed_logistic")
            & development_oof["Season"].isin(
                selection["comparison_seasons"]
            )
        ][["TargetKey", "Prediction"]].rename(
            columns={"Prediction": "SeedPrediction"}
        )
        aligned = chosen.merge(
            seed, on="TargetKey", how="inner", validate="one_to_one"
        )
        if not aligned.empty:
            result = season_cluster_bootstrap_difference(
                aligned,
                probability_a="ChosenPrediction",
                probability_b="SeedPrediction",
                repetitions=int(MODE["bootstrap_repetitions"]),
                seed=SEED + (0 if gender == "M" else 1),
            )
            paired_records.append(
                {
                    "Gender": gender,
                    "ModelA": (
                        f"{selection['architecture']}/"
                        f"{selection['model']}"
                    ),
                    "ModelB": (
                        f"{seed_row['Architecture']}/seed_logistic"
                    ),
                    "Rows": len(aligned),
                    "Seasons": aligned["Season"].nunique(),
                    **result,
                }
            )

paired_comparisons = pd.DataFrame(paired_records)
paired_comparisons.to_csv(
    MODEL_REPORTS / "paired_season_bootstrap.csv",
    index=False,
)
paired_comparisons
if not paired_comparisons.empty:
    fig = px.bar(
        paired_comparisons,
        x="MeanBrierDifferenceAminusB",
        y="Gender",
        error_x=(
            paired_comparisons["CIUpper"]
            - paired_comparisons["MeanBrierDifferenceAminusB"]
        ),
        error_x_minus=(
            paired_comparisons["MeanBrierDifferenceAminusB"]
            - paired_comparisons["CILower"]
        ),
        orientation="h",
        hover_data=[
            "ModelA",
            "ModelB",
            "Rows",
            "Seasons",
            "ProbabilityALowerBrier",
        ],
        title="Recommended model versus seed baseline",
        labels={
            "MeanBrierDifferenceAminusB": (
                "Brier difference (recommended minus seed; lower is better)"
            ),
            "Gender": "Tournament",
        },
    )
    fig.add_vline(x=0, line_dash="dash")
    save_plotly(fig, "recommended_vs_seed_bootstrap")
    PLOT_ARTIFACTS.append("recommended_vs_seed_bootstrap.html")


## 18. Model interpretation

The notebook reports standardized logistic coefficients, native XGBoost and LightGBM contribution values, model-agnostic held-out permutation importance, and neural-network permutation importance. Interactive Plotly charts emphasize both feature magnitude and feature-family context.

In [ ]:
EXPLAINABILITY_RECORDS: list[dict[str, Any]] = []
IMPORTANCE_RECORDS: list[pd.DataFrame] = []
PERMUTATION_RECORDS: list[pd.DataFrame] = []

requested_explanation_models = [
    "elastic_logistic",
    "xgb_classifier",
    "lgb_classifier",
    "torch_mlp",
    "tensorflow_mlp",
]

if MODE["run_explainability"]:
    for gender in ("M", "W"):
        gender_leaderboard = common_window_leaderboard.loc[
            common_window_leaderboard["Gender"].eq(gender)
        ]
        for model_name in requested_explanation_models:
            model_candidates = gender_leaderboard.loc[
                gender_leaderboard["Model"].eq(model_name)
            ].sort_values("MacroSeasonBrier")
            if model_candidates.empty:
                EXPLAINABILITY_RECORDS.append(
                    {
                        "Gender": gender,
                        "Model": model_name,
                        "Status": "failed",
                        "Error": "No completed matched-season model stream.",
                    }
                )
                continue

            best_architecture = str(
                model_candidates.iloc[0]["Architecture"]
            )
            training_context = (
                gender
                if best_architecture == "separate_gender"
                else "Pooled"
            )
            context = outer_plan.loc[
                outer_plan["Architecture"].eq(best_architecture)
                & outer_plan["Gender"].eq(training_context)
                & outer_plan["ValidationSeason"].eq(
                    max(MATCHED_VALIDATION_SEASONS)
                )
            ]
            if context.empty:
                EXPLAINABILITY_RECORDS.append(
                    {
                        "Gender": gender,
                        "Model": model_name,
                        "Architecture": best_architecture,
                        "Status": "failed",
                        "Error": "No 2021 outer fold for the selected architecture.",
                    }
                )
                continue

            outer = context.iloc[0]
            outer_fold_id = str(outer["OuterFoldID"])
            directory = outer_model_directory(
                outer_fold_id, model_name
            )
            metadata_path = directory / "metadata.json"
            if not metadata_path.exists():
                EXPLAINABILITY_RECORDS.append(
                    {
                        "Gender": gender,
                        "Model": model_name,
                        "Architecture": best_architecture,
                        "Status": "failed",
                        "Error": f"Missing checkpoint: {metadata_path}",
                    }
                )
                continue

            try:
                metadata = read_json(metadata_path)
                features = list(metadata["outer_selected_features"])
                params = dict(metadata["parameters"])
                fixed_iterations = metadata.get(
                    "fixed_iterations_from_inner"
                )
                training_seasons = parse_json_int_list(
                    outer["TrainingSeasonsJSON"]
                )
                validation_season = int(outer["ValidationSeason"])

                training = rows_for_context(
                    development,
                    seasons=training_seasons,
                    gender=training_context,
                    universe="rich",
                )
                validation_context = rows_for_context(
                    development,
                    seasons=[validation_season],
                    gender=training_context,
                    universe="rich",
                )
                validation = validation_context.loc[
                    validation_context["Gender"].eq(gender)
                ].copy()
                assert not validation.empty

                spec = MODEL_SPECS[model_name]
                fitted, _ = fit_predictor(
                    spec,
                    params,
                    training,
                    validation_context,
                    features,
                    fixed_iterations=(
                        int(fixed_iterations)
                        if fixed_iterations is not None
                        else None
                    ),
                    random_seed=SEED + validation_season,
                )

                sample = validation.sample(
                    n=min(int(MODE["importance_rows"]), len(validation)),
                    random_state=SEED,
                ).copy()
                x_sample = prepare_numeric_matrix(sample, features)
                attribution_method = "selected_feature_order"
                attribution_values = np.linspace(
                    1.0,
                    0.1,
                    num=len(features),
                    dtype=float,
                )

                if model_name == "elastic_logistic":
                    attribution_values = np.abs(
                        np.asarray(fitted.model.coef_[0], dtype=float)
                    )
                    attribution_method = "absolute_standardized_coefficient"

                elif model_name == "xgb_classifier":
                    matrix = xgb.DMatrix(
                        x_sample,
                        feature_names=features,
                    )
                    contributions = np.asarray(
                        fitted.model.predict(
                            matrix,
                            pred_contribs=True,
                            strict_shape=False,
                        ),
                        dtype=np.float64,
                    )
                    attribution_values = np.mean(
                        np.abs(contributions[:, :-1]), axis=0
                    )
                    attribution_method = "xgboost_native_shap"

                elif model_name == "lgb_classifier":
                    contributions = np.asarray(
                        fitted.model.predict(
                            x_sample,
                            pred_contrib=True,
                            num_iteration=fitted.best_iteration,
                        ),
                        dtype=np.float64,
                    )
                    attribution_values = np.mean(
                        np.abs(contributions[:, :-1]), axis=0
                    )
                    attribution_method = "lightgbm_native_shap"

                elif model_name == "torch_mlp":
                    transformed = fitted.preprocessor.transform(
                        x_sample
                    ).astype(np.float32)
                    tensor = torch.tensor(
                        transformed,
                        dtype=torch.float32,
                        requires_grad=True,
                    )
                    fitted.model.eval()
                    probability = torch.sigmoid(
                        fitted.model(tensor).squeeze(1)
                    )
                    gradient = torch.autograd.grad(
                        probability.sum(),
                        tensor,
                    )[0].detach().cpu().numpy()
                    attribution_values = np.mean(
                        np.abs(gradient * transformed), axis=0
                    )
                    attribution_method = "pytorch_gradient_times_input"

                elif model_name == "tensorflow_mlp":
                    transformed = fitted.preprocessor.transform(
                        x_sample
                    ).astype(np.float32)
                    tensor = tf.convert_to_tensor(transformed)
                    with tf.GradientTape() as tape:
                        tape.watch(tensor)
                        probability = tf.reshape(
                            fitted.model(tensor, training=False),
                            (-1,),
                        )
                        objective = tf.reduce_sum(probability)
                    gradient = tape.gradient(
                        objective,
                        tensor,
                    )
                    if gradient is None:
                        raise RuntimeError(
                            "TensorFlow did not return an input gradient."
                        )
                    gradient = gradient.numpy()
                    attribution_values = np.mean(
                        np.abs(gradient * transformed), axis=0
                    )
                    attribution_method = "tensorflow_gradient_times_input"

                importance = pd.DataFrame(
                    {
                        "Gender": gender,
                        "Architecture": best_architecture,
                        "Model": model_name,
                        "ModelDisplay": MODEL_DISPLAY_NAMES.get(
                            model_name, model_name
                        ),
                        "ValidationSeason": validation_season,
                        "Feature": features,
                        "Block": [
                            FEATURE_TO_BLOCK.get(feature, "other")
                            for feature in features
                        ],
                        "AttributionMethod": attribution_method,
                        "AttributionImportance": attribution_values,
                    }
                ).sort_values(
                    "AttributionImportance", ascending=False
                )
                IMPORTANCE_RECORDS.append(importance)

                candidate_features = importance.head(25)[
                    "Feature"
                ].tolist()
                baseline_probability = fitted.predict_raw(validation)
                baseline_brier = float(
                    np.mean(
                        (
                            baseline_probability
                            - validation["Team1Win"].to_numpy(dtype=float)
                        )
                        ** 2
                    )
                )
                rng = np.random.default_rng(
                    SEED + validation_season
                )
                permutation_rows = []
                for feature in candidate_features:
                    repeat_scores = []
                    for _ in range(
                        int(MODE["permutation_repeats"])
                    ):
                        permuted = validation.copy()
                        values = permuted[feature].to_numpy(copy=True)
                        rng.shuffle(values)
                        permuted[feature] = values
                        probability = fitted.predict_raw(permuted)
                        repeat_scores.append(
                            float(
                                np.mean(
                                    (
                                        probability
                                        - permuted[
                                            "Team1Win"
                                        ].to_numpy(dtype=float)
                                    )
                                    ** 2
                                )
                            )
                        )
                    permutation_rows.append(
                        {
                            "Gender": gender,
                            "Architecture": best_architecture,
                            "Model": model_name,
                            "ModelDisplay": MODEL_DISPLAY_NAMES.get(
                                model_name, model_name
                            ),
                            "ValidationSeason": validation_season,
                            "Feature": feature,
                            "Block": FEATURE_TO_BLOCK.get(
                                feature, "other"
                            ),
                            "BaselineBrier": baseline_brier,
                            "PermutedBrierMean": float(
                                np.mean(repeat_scores)
                            ),
                            "PermutedBrierStd": float(
                                np.std(repeat_scores, ddof=0)
                            ),
                            "BrierIncrease": float(
                                np.mean(repeat_scores)
                                - baseline_brier
                            ),
                            "Repeats": int(
                                MODE["permutation_repeats"]
                            ),
                        }
                    )
                permutation_table = pd.DataFrame(
                    permutation_rows
                ).sort_values("BrierIncrease", ascending=False)
                PERMUTATION_RECORDS.append(permutation_table)

                EXPLAINABILITY_RECORDS.append(
                    {
                        "Gender": gender,
                        "Architecture": best_architecture,
                        "Model": model_name,
                        "ValidationSeason": validation_season,
                        "RowsExplained": int(len(sample)),
                        "SelectedFeatures": int(len(features)),
                        "AttributionMethod": attribution_method,
                        "TopAttributionFeature": (
                            str(importance.iloc[0]["Feature"])
                            if not importance.empty
                            else None
                        ),
                        "TopPermutationFeature": (
                            str(permutation_table.iloc[0]["Feature"])
                            if not permutation_table.empty
                            else None
                        ),
                        "Status": "complete",
                    }
                )

                del (
                    training,
                    validation_context,
                    validation,
                    fitted,
                    sample,
                    x_sample,
                    importance,
                    permutation_table,
                )
                if model_name == "tensorflow_mlp" and tf is not None:
                    tf.keras.utils.clear_session(free_memory=True)
                gc.collect()

            except Exception as exc:
                EXPLAINABILITY_RECORDS.append(
                    {
                        "Gender": gender,
                        "Architecture": best_architecture,
                        "Model": model_name,
                        "Status": "failed",
                        "Error": repr(exc),
                    }
                )
                log_event(
                    "explainability_failed",
                    gender=gender,
                    architecture=best_architecture,
                    model=model_name,
                    error=repr(exc),
                )
                print(
                    "EXPLAINABILITY FAILED:",
                    gender,
                    model_name,
                    repr(exc),
                )

explainability_summary = pd.DataFrame(EXPLAINABILITY_RECORDS)
all_feature_importance = (
    pd.concat(IMPORTANCE_RECORDS, ignore_index=True)
    if IMPORTANCE_RECORDS
    else pd.DataFrame()
)
all_permutation_importance = (
    pd.concat(PERMUTATION_RECORDS, ignore_index=True)
    if PERMUTATION_RECORDS
    else pd.DataFrame()
)

explainability_summary.to_csv(
    MODEL_REPORTS / "explainability_summary.csv",
    index=False,
)
all_feature_importance.to_csv(
    MODEL_REPORTS / "model_feature_attributions.csv",
    index=False,
)
all_permutation_importance.to_csv(
    MODEL_REPORTS / "held_out_permutation_importance.csv",
    index=False,
)

if not all_feature_importance.empty:
    for gender in ("M", "W"):
        top = (
            all_feature_importance.loc[
                all_feature_importance["Gender"].eq(gender)
            ]
            .sort_values(
                ["Model", "AttributionImportance"],
                ascending=[True, False],
            )
            .groupby("Model", observed=True)
            .head(12)
            .copy()
        )
        fig = px.bar(
            top,
            x="AttributionImportance",
            y="Feature",
            color="Block",
            facet_col="ModelDisplay",
            facet_col_wrap=2,
            orientation="h",
            hover_data=[
                "Architecture",
                "AttributionMethod",
                "ValidationSeason",
            ],
            title=f"{gender}: model-specific feature attributions",
            height=max(
                700,
                320 * math.ceil(top["ModelDisplay"].nunique() / 2),
            ),
        )
        fig.update_yaxes(matches=None, showticklabels=True)
        save_plotly(fig, f"feature_attribution_{gender}")
        PLOT_ARTIFACTS.append(
            f"feature_attribution_{gender}.html"
        )

if not all_permutation_importance.empty:
    for gender in ("M", "W"):
        top = (
            all_permutation_importance.loc[
                all_permutation_importance["Gender"].eq(gender)
            ]
            .sort_values(
                ["Model", "BrierIncrease"],
                ascending=[True, False],
            )
            .groupby("Model", observed=True)
            .head(10)
            .copy()
        )
        fig = px.bar(
            top,
            x="BrierIncrease",
            y="Feature",
            color="Block",
            facet_col="ModelDisplay",
            facet_col_wrap=2,
            orientation="h",
            error_x="PermutedBrierStd",
            hover_data=[
                "Architecture",
                "BaselineBrier",
                "PermutedBrierMean",
                "Repeats",
            ],
            title=f"{gender}: held-out permutation importance",
            height=max(
                700,
                320 * math.ceil(top["ModelDisplay"].nunique() / 2),
            ),
        )
        fig.update_yaxes(matches=None, showticklabels=True)
        save_plotly(fig, f"permutation_importance_{gender}")
        PLOT_ARTIFACTS.append(
            f"permutation_importance_{gender}.html"
        )

log_event(
    "explainability_complete",
    completed=int(
        explainability_summary["Status"].eq("complete").sum()
        if not explainability_summary.empty
        else 0
    ),
    failed=int(
        explainability_summary["Status"].eq("failed").sum()
        if not explainability_summary.empty
        else 0
    ),
)
explainability_summary

## 19. Final integrity verification and recipe freeze

This section is self-contained and idempotent. It verifies the full matched experiment,
confirms that the locked benchmark and Stage 2 data were not used, audits every requested
framework and architecture, validates dimensionality, ensemble, metric, ablation, report,
and interpretation completeness, and writes the single authoritative development recipe.

Rerunning this cell does not retrain or retune a model.

In [ ]:
"""Self-contained final integrity verification and recipe freeze.

This cell does not train or tune models. It writes the authoritative
recipe only after every blocking check passes and is safe to rerun.
"""

FINAL_CHECKS: list[dict[str, Any]] = []

FINALIZATION_VERSION = 2
FINALIZATION_TIMESTAMP_UTC = datetime.now(
    timezone.utc
).isoformat()
FINALIZATION_LOG_PATH = (
    MODEL_REPORTS / "finalization.log"
)


def atomic_write_text(
    path: Path,
    text: str,
) -> None:
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )
    temporary = path.with_suffix(
        path.suffix + ".tmp"
    )
    temporary.write_text(
        text,
        encoding="utf-8",
    )
    os.replace(temporary, path)


def log_finalization(
    message: str,
    **details: Any,
) -> None:
    record = {
        "timestamp_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "message": message,
        **details,
    }
    with FINALIZATION_LOG_PATH.open(
        "a",
        encoding="utf-8",
    ) as handle:
        handle.write(
            json.dumps(
                record,
                default=str,
            )
            + "\n"
        )
    log_event(
        "finalization",
        **record,
    )


log_finalization(
    "finalization_started",
    version=FINALIZATION_VERSION,
    development_oof_rows=int(
        len(development_oof)
    ),
)


def add_final_check(
    name: str,
    passed: bool,
    details: str,
    *,
    blocking: bool = True,
) -> None:
    FINAL_CHECKS.append(
        {
            "Check": name,
            "Passed": bool(passed),
            "Blocking": bool(blocking),
            "Details": details,
        }
    )


expected_core_tasks = int(
    sum(len(json.loads(value)) for value in outer_plan["ModelsJSON"])
)
completed_core_tasks = int(run_progress["Status"].eq("complete").sum())
failed_core_tasks = int(run_progress["Status"].eq("failed").sum())

add_final_check(
    "development rows only",
    development["DatasetRole"].eq("development").all(),
    f"rows={len(development):,}",
)
add_final_check(
    "locked seasons absent from all model-selection rows",
    not development["Season"].isin(LOCKED_SEASONS).any()
    and not development_oof["Season"].isin(LOCKED_SEASONS).any(),
    f"locked={sorted(LOCKED_SEASONS)}",
)
add_final_check(
    "Stage 2 matrix was not loaded",
    "stage2_matchups" not in globals(),
    f"Stage 2 remains on disk: {STAGE2_STORE_PATH}",
)
add_final_check(
    "matched outer seasons preserved",
    set(map(int, comparison_rows["Season"].unique()))
    == set(MATCHED_VALIDATION_SEASONS),
    str(sorted(map(int, comparison_rows["Season"].unique()))),
)
add_final_check(
    "all planned core model tasks completed",
    completed_core_tasks == expected_core_tasks
    and failed_core_tasks == 0,
    (
        f"expected={expected_core_tasks}; "
        f"completed={completed_core_tasks}; failed={failed_core_tasks}"
    ),
)
add_final_check(
    "no registered model, ensemble, or ablation failures",
    len(MODEL_FAILURES) == 0,
    f"registered_failures={len(MODEL_FAILURES)}",
)
add_final_check(
    "all requested frameworks completed in separate and pooled architectures",
    framework_coverage["Complete"].all(),
    framework_coverage[
        ["Gender", "Architecture", "Complete", "MissingModels"]
    ].to_json(orient="records"),
)
required_metric_columns = {
    "MacroSeasonBrier",
    "GameWeightedBrier",
    "LogLoss",
    "ROCAUC",
    "AveragePrecision",
    "AUCPR",
    "Accuracy",
    "BalancedAccuracy",
    "Precision",
    "Recall",
    "F1",
    "MCC",
    "CalibrationIntercept",
    "CalibrationSlope",
    "ECE",
}
metric_columns_present = required_metric_columns.issubset(
    common_window_leaderboard.columns
)
metric_integrity_issues = pd.DataFrame()
if metric_columns_present:
    metric_long = common_window_leaderboard[
        [
            "Gender",
            "Architecture",
            "Model",
            *sorted(required_metric_columns),
        ]
    ].melt(
        id_vars=[
            "Gender",
            "Architecture",
            "Model",
        ],
        var_name="Metric",
        value_name="Value",
    )
    numeric_value = pd.to_numeric(
        metric_long["Value"],
        errors="coerce",
    )
    metric_long["Finite"] = np.isfinite(
        numeric_value
    )
    metric_integrity_issues = metric_long.loc[
        ~metric_long["Finite"]
    ].reset_index(drop=True)

metric_integrity_issues.to_csv(
    MODEL_REPORTS / "metric_integrity_issues.csv",
    index=False,
)
add_final_check(
    "complete finite probability and classification metric suite",
    (
        metric_columns_present
        and metric_integrity_issues.empty
    ),
    (
        f"metrics={sorted(required_metric_columns)}; "
        f"nonfinite_values={len(metric_integrity_issues)}"
    ),
)
add_final_check(
    "OOF probabilities are finite and bounded",
    np.isfinite(
        development_oof["Prediction"].to_numpy(dtype=float)
    ).all()
    and development_oof["Prediction"].between(0, 1).all(),
    f"rows={len(development_oof):,}",
)
add_final_check(
    "OOF prediction keys are unique within model streams",
    development_oof.duplicated(
        ["OuterFoldID", "Architecture", "Model", "TargetKey"]
    ).sum()
    == 0,
    "OuterFoldID + Architecture + Model + TargetKey",
)

model_caps = {
    "elastic_logistic": MODE["linear_feature_cap"],
    "xgb_classifier": MODE["tree_feature_cap"],
    "lgb_classifier": MODE["tree_feature_cap"],
    "torch_mlp": MODE["neural_feature_cap"],
    "tensorflow_mlp": MODE["neural_feature_cap"],
    "ridge_margin": MODE["margin_feature_cap"],
    "xgb_margin": MODE["margin_feature_cap"],
    "lgb_margin": MODE["margin_feature_cap"],
}
cap_failures = []
for model_name, cap in model_caps.items():
    observed = development_oof.loc[
        development_oof["Model"].eq(model_name)
        & development_oof["FeatureCount"].notna(),
        "FeatureCount",
    ]
    if not observed.empty and float(observed.max()) > float(cap):
        cap_failures.append(
            {
                "model": model_name,
                "observed": float(observed.max()),
                "cap": int(cap),
            }
        )
add_final_check(
    "model-family feature caps respected",
    not cap_failures,
    json.dumps(cap_failures if cap_failures else model_caps),
)

if not ensemble_weights.empty:
    weight_sums = ensemble_weights.groupby(
        "OuterFoldID", observed=True
    )["Weight"].sum()
    add_final_check(
        "ensemble weights are nonnegative",
        ensemble_weights["Weight"].ge(-1e-12).all(),
        f"minimum={ensemble_weights['Weight'].min():.8f}",
    )
    add_final_check(
        "ensemble weights sum to one",
        np.allclose(weight_sums.to_numpy(), 1.0, atol=1e-7),
        (
            f"min={weight_sums.min():.8f}; "
            f"max={weight_sums.max():.8f}"
        ),
    )
else:
    add_final_check(
        "ensemble output available",
        False,
        "No ensemble weights were produced.",
    )

add_final_check(
    "feature ablation completed",
    (
        not MODE["run_ablation"]
        or (
            not ablation_leaderboard.empty
            and set(ablation_leaderboard["BaseModel"])
            >= {"elastic_logistic", "xgb_classifier"}
            and set(ablation_leaderboard["Gender"]) >= {"M", "W"}
        )
    ),
    (
        "disabled"
        if not MODE["run_ablation"]
        else f"rows={len(ablation_leaderboard):,}"
    ),
)

expected_explanation_pairs = {
    (gender, model)
    for gender in ("M", "W")
    for model in requested_explanation_models
}
completed_explanation_pairs = (
    set(
        map(
            tuple,
            explainability_summary.loc[
                explainability_summary["Status"].eq("complete"),
                ["Gender", "Model"],
            ].itertuples(index=False, name=None),
        )
    )
    if not explainability_summary.empty
    else set()
)
add_final_check(
    "interpretation completed for every requested framework and tournament",
    (
        not MODE["run_explainability"]
        or expected_explanation_pairs.issubset(
            completed_explanation_pairs
        )
    ),
    (
        f"expected={len(expected_explanation_pairs)}; "
        f"completed={len(completed_explanation_pairs)}"
    ),
)

missing_plot_files = [
    name
    for name in sorted(set(PLOT_ARTIFACTS))
    if not (FIGURE_DIR / name).exists()
]
add_final_check(
    "interactive Plotly reports created",
    not missing_plot_files and len(set(PLOT_ARTIFACTS)) >= 10,
    (
        f"reports={len(set(PLOT_ARTIFACTS))}; "
        f"missing={missing_plot_files}"
    ),
)
add_final_check(
    "source feature store integrity preserved",
    READINESS_02["blocking_feature_check_failures"] == 0
    and READINESS_02["symmetry_failures"] == 0,
    (
        f"feature_checks={READINESS_02['blocking_feature_checks']}; "
        f"symmetry_checks={READINESS_02['symmetry_checks']}"
    ),
)
add_final_check(
    "winner summary contains both tournaments",
    {"M", "W"}.issubset(RECIPE_SELECTIONS),
    json.dumps(sorted(RECIPE_SELECTIONS)),
)

final_checks = pd.DataFrame(FINAL_CHECKS)
blocking_failures = final_checks.loc[
    final_checks["Blocking"] & ~final_checks["Passed"]
]
final_checks.to_csv(
    MODEL_REPORTS / "final_checks.csv",
    index=False,
)

if blocking_failures.empty:
    MODEL_RECIPE["status"] = "frozen_development_recipe"
    MODEL_RECIPE["recipe_sha256"] = object_sha256(
        {
            key: value
            for key, value in MODEL_RECIPE.items()
            if key != "recipe_sha256"
        }
    )
    recipe_path = CONFIG_DIR / "model_recipe.yaml"
    recipe_text = yaml.safe_dump(
        MODEL_RECIPE,
        sort_keys=False,
    )
    atomic_write_text(
        recipe_path,
        recipe_text,
    )
    atomic_write_text(
        MODEL_REPORTS / "model_recipe.yaml",
        recipe_text,
    )
else:
    recipe_path = MODEL_REPORTS / "model_recipe_blocked.yaml"
    blocked_recipe = dict(MODEL_RECIPE)
    blocked_recipe["status"] = "blocked_by_integrity_gate"
    blocked_recipe["blocking_failures"] = (
        blocking_failures["Check"].tolist()
    )
    atomic_write_text(
        recipe_path,
        yaml.safe_dump(
            blocked_recipe,
            sort_keys=False,
        ),
    )

# Remove obsolete recipe filenames so the repository has one authoritative recipe.
for obsolete_name in (
    "model_recipe_v1.yaml",
    "model_recipe_v2.yaml",
    "model_recipe_smoke.yaml",
):
    obsolete_path = CONFIG_DIR / obsolete_name
    if obsolete_path.exists():
        obsolete_path.unlink()

artifact_paths = {
    "development_oof": oof_path,
    "leaderboard": MODEL_REPORTS / "development_leaderboard.csv",
    "matched_comparison": (
        MODEL_REPORTS / "matched_season_model_comparison.csv"
    ),
    "winner_summary": MODEL_REPORTS / "winner_summary.json",
    "winner_candidates": MODEL_REPORTS / "winner_candidates.csv",
    "metrics_by_season": (
        MODEL_REPORTS / "development_metrics_by_season.csv"
    ),
    "ablation": MODEL_REPORTS / "feature_block_ablation.csv",
    "explainability": MODEL_REPORTS / "explainability_summary.csv",
    "permutation_importance": (
        MODEL_REPORTS / "held_out_permutation_importance.csv"
    ),
    "final_checks": MODEL_REPORTS / "final_checks.csv",
    "recipe": recipe_path,
    "run_log": LOG_DIR / "run.log",
    "events": EVENTS_PATH,
}

artifact_manifest = pd.DataFrame(
    [
        {
            "Artifact": name,
            "Path": str(path),
            "Exists": path.exists(),
            "SizeMB": (
                round(path.stat().st_size / 1024**2, 3)
                if path.exists()
                else np.nan
            ),
            "SHA256": (
                sha256_file(path) if path.exists() else None
            ),
            "SplitContractSHA256": SPLITS["contract_sha256"],
            "FeatureContractSHA256": FEATURE_CONFIG[
                "feature_contract_sha256"
            ],
            "ModelContractSHA256": MODEL_CONFIG[
                "model_contract_sha256"
            ],
        }
        for name, path in artifact_paths.items()
    ]
)
artifact_manifest.to_csv(
    MODEL_REPORTS / "artifact_manifest.csv",
    index=False,
)

readiness = {
    "notebook": "03_model_comparison_and_diagnostics.ipynb",
    "status": (
        "complete" if blocking_failures.empty else "blocking_failures"
    ),
    "run_profile": RUN_PROFILE,
    "notebook_implementation_version": NOTEBOOK_IMPLEMENTATION_VERSION,
    "finalization_version": FINALIZATION_VERSION,
    "contracts": EXPECTED_CONTRACTS,
    "matched_validation_seasons": MATCHED_VALIDATION_SEASONS,
    "development_rows": int(len(development)),
    "development_oof_rows": int(len(development_oof)),
    "planned_core_tasks": expected_core_tasks,
    "completed_core_tasks": completed_core_tasks,
    "failed_core_tasks": failed_core_tasks,
    "registered_failures": int(len(MODEL_FAILURES)),
    "requested_framework_coverage_complete": bool(
        framework_coverage["Complete"].all()
    ),
    "ablation_rows": int(len(ablation_leaderboard)),
    "completed_explanation_pairs": int(
        len(completed_explanation_pairs)
    ),
    "expected_explanation_pairs": int(
        len(expected_explanation_pairs)
    ),
    "plotly_reports": sorted(set(PLOT_ARTIFACTS)),
    "blocking_final_check_failures": int(
        len(blocking_failures)
    ),
    "winner_summary": WINNER_SUMMARY,
    "recipe_status": MODEL_RECIPE["status"],
    "recipe_sha256": MODEL_RECIPE["recipe_sha256"],
    "maximum_observed_feature_count": int(
        development_oof.loc[
            development_oof["FeatureCount"].notna(),
            "FeatureCount",
        ].max()
    ),
    "peak_logged_rss_mb": (
        float(resource_log["EndRSSMB"].max())
        if not resource_log.empty
        else float(rss_mb())
    ),
    "stage2_loaded": False,
    "locked_benchmark_evaluated": False,
}
atomic_write_json(
    MODEL_REPORTS / "03_readiness_summary.json",
    readiness,
)

men_rec = RECIPE_SELECTIONS["M"]["recommended"]
women_rec = RECIPE_SELECTIONS["W"]["recommended"]
development_report = f"""# Model Development and Comparison Report

## Validation design

- Matched held-out seasons: {MATCHED_VALIDATION_SEASONS}
- Entire tournament seasons are held out.
- Training seasons always precede validation seasons.
- Inner folds select preprocessing, features, parameters, training length, and calibration.
- The 2022–2025 benchmark and 2026 Stage 2 data were not opened.

## Model families

Logistic regression, XGBoost, LightGBM, PyTorch, and TensorFlow were compared in separate and pooled architectures. Point-margin models and constrained ensembles were also evaluated.

## Recommended development selections

### Men's tournament

- Architecture: {men_rec['architecture']}
- Model: {men_rec['model_display']}
- Mean held-out season Brier: {men_rec['macro_season_brier']:.6f}
- ROC AUC: {men_rec['roc_auc']:.6f}
- Average precision: {men_rec['average_precision']:.6f}
- F1 at 0.50: {men_rec['f1']:.6f}

### Women's tournament

- Architecture: {women_rec['architecture']}
- Model: {women_rec['model_display']}
- Mean held-out season Brier: {women_rec['macro_season_brier']:.6f}
- ROC AUC: {women_rec['roc_auc']:.6f}
- Average precision: {women_rec['average_precision']:.6f}
- F1 at 0.50: {women_rec['f1']:.6f}

## Interpretation of “winner”

The numerical winner has the lowest mean season-level Brier score. The recommended winner applies the one-standard-error rule and may choose a simpler model when its performance is statistically indistinguishable from the lowest score.

## Feature ablation

Ablation retrains reference models after cumulative feature blocks are added. It measures whether ratings, efficiency, schedule, history, rankings, and matchup interactions improve held-out seasons rather than merely fitting the training sample.

## Next boundary

The replacement notebook `04` may evaluate the frozen recipe on 2022–2025 once. No model or feature decision may be changed after those outcomes are viewed without creating a new explicitly documented experiment.
"""
(MODEL_REPORTS / "MODEL_DEVELOPMENT_REPORT.md").write_text(
    development_report,
    encoding="utf-8",
)

finalization_summary = {
    "status": (
        "complete"
        if blocking_failures.empty
        else "blocked"
    ),
    "timestamp_utc": (
        FINALIZATION_TIMESTAMP_UTC
    ),
    "finalization_version": (
        FINALIZATION_VERSION
    ),
    "core_tasks": {
        "expected": expected_core_tasks,
        "completed": completed_core_tasks,
        "failed": failed_core_tasks,
    },
    "registered_failures": int(
        len(MODEL_FAILURES)
    ),
    "metric_integrity_issues": int(
        len(metric_integrity_issues)
    ),
    "explanation_pairs": {
        "expected": int(
            len(expected_explanation_pairs)
        ),
        "completed": int(
            len(completed_explanation_pairs)
        ),
    },
    "blocking_failures": (
        blocking_failures[
            "Check"
        ].tolist()
    ),
    "recipe_path": str(recipe_path),
    "recipe_status": (
        MODEL_RECIPE["status"]
        if blocking_failures.empty
        else "blocked_by_integrity_gate"
    ),
}
atomic_write_json(
    MODEL_REPORTS
    / "03_finalization_summary.json",
    finalization_summary,
)

log_event(
    "notebook_completed",
    status=readiness["status"],
    blocking_failures=int(len(blocking_failures)),
    recipe_status=MODEL_RECIPE["status"],
    men_recommended=men_rec,
    women_recommended=women_rec,
)

log_finalization(
    "finalization_checks_completed",
    blocking_failures=int(
        len(blocking_failures)
    ),
    failed_checks=blocking_failures[
        "Check"
    ].tolist(),
)
display(final_checks)
if not blocking_failures.empty:
    display(blocking_failures)
    raise AssertionError(
        "Notebook 03 has a genuine blocking integrity failure. "
        "Review final_checks.csv and metric_integrity_issues.csv. "
        "Completed checkpoints remain reusable."
    )

print(
    "\nNOTEBOOK 03 COMPLETE — the matched model comparison, nested validation, "
    "calibration, ablation, interpretation, and portfolio reports all passed."
)
print(
    "The development recipe is frozen at configs/model_recipe.yaml. "
    "The 2022–2025 benchmark and 2026 Stage 2 matrix remain unopened."
)
print("\nRecommended men's model:", men_rec["architecture"], "/", men_rec["model_display"])
print("Recommended women's model:", women_rec["architecture"], "/", women_rec["model_display"])

# Stop after the completion gate

Proceed only after the final cell prints `NOTEBOOK 03 COMPLETE` and writes:

```text
configs/model_recipe.yaml
reports/modeling/03_model_comparison/03_readiness_summary.json
reports/modeling/03_model_comparison/03_finalization_summary.json
reports/modeling/03_model_comparison/final_checks.csv
reports/modeling/03_model_comparison/winner_summary.json
```

Notebook `04` is the first stage permitted to evaluate the locked 2022–2025
benchmark and score the 2026 Stage 2 matrix.